<a href="https://colab.research.google.com/github/eow2003/Intership/blob/main/Zoho_Automatic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

  Chỉ cần chạy duy nhất block dưới

In [ ]:
# ============================================================
# NOTEBOOK CLEAN VERSION - 3 CELLS
# Google Colab + Google Drive
# ============================================================

# ============================================================
# CELL 1: SETUP GOOGLE DRIVE + FOLDERS + PACKAGES
# ============================================================

!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-vie poppler-utils
!pip -q install pymupdf pandas openpyxl rapidfuzz unidecode pytesseract pillow tqdm requests

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

BASE_DIR = Path("/content/drive/MyDrive/Zoho_Procurement")

TBMT_DIR = BASE_DIR / "TBMT"
KHLCNT_DIR = BASE_DIR / "KHLCNT"
OUTPUT_DIR = BASE_DIR / "output_excel"
INPUT_MATCH_DIR = BASE_DIR / "input_match"
LOG_DIR = BASE_DIR / "logs"
CONFIG_DIR = BASE_DIR / "config"
TEXT_CACHE_DIR = LOG_DIR / "text_cache"
CHECKPOINT_DIR = LOG_DIR / "checkpoints"

for folder in [TBMT_DIR, KHLCNT_DIR, OUTPUT_DIR, INPUT_MATCH_DIR, LOG_DIR, CONFIG_DIR, TEXT_CACHE_DIR, CHECKPOINT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR, BASE_DIR.exists())
print("TBMT_DIR:", TBMT_DIR)
print("KHLCNT_DIR:", KHLCNT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("INPUT_MATCH_DIR:", INPUT_MATCH_DIR)
print("LOG_DIR:", LOG_DIR)
print("CONFIG_DIR:", CONFIG_DIR)
print("TEXT_CACHE_DIR:", TEXT_CACHE_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

print("\nTBMT PDFs:")
for f in sorted(TBMT_DIR.glob("*.pdf")):
    print("-", f.name)

print("\nKHLCNT PDFs:")
for f in sorted(KHLCNT_DIR.glob("*.pdf")):
    print("-", f.name)


# ============================================================
# CELL 2: PARSE PDF FROM DRIVE -> FIXED EXCEL OUTPUT
# - Giữ parser cũ
# - Không sinh nhiều file timestamp
# - Skip PDF đã xử lý theo tên file
# - Append dữ liệu mới vào output cũ
# - Dedupe lần cuối theo Account Name / Số TBMT / Số KHLCNT
# - Xuất 1 file tổng + 3 file import cố định cho Zoho
# ============================================================

import re
import unicodedata
import hashlib
from pathlib import Path
from datetime import datetime

import fitz
import pandas as pd
from rapidfuzz import fuzz
from tqdm import tqdm
from PIL import Image
import pytesseract


# ============================================================
# OUTPUT / REGISTRY SETTINGS
# ============================================================

PROCESSED_PDF_REGISTRY = CONFIG_DIR / "processed_pdf_registry.xlsx"

MASTER_OUTPUT_PATH = OUTPUT_DIR / "Zoho_PDF_Import_Output_LATEST.xlsx"
ACCOUNTS_IMPORT_PATH = OUTPUT_DIR / "Accounts_for_import.xlsx"
KHLCNT_IMPORT_PATH = OUTPUT_DIR / "KHLCNT_for_import.xlsx"
TBMT_IMPORT_PATH = OUTPUT_DIR / "TBMT_for_import.xlsx"
KHLCNT_ITEMS_MATCH_PATH = OUTPUT_DIR / "KHLCNT_items_for_match.xlsx"
MATCH_CANDIDATES_HISTORY_PATH = LOG_DIR / "Match_Candidates_History.xlsx"
PARSE_CHECKPOINT_PATH = CHECKPOINT_DIR / "parse_checkpoint_latest.xlsx"
DIRECT_SCAN_MEMORY_DIR = OUTPUT_DIR / "direct_scan_memory"
DIRECT_SCAN_MEMORY_DIR.mkdir(parents=True, exist_ok=True)

DIRECT_SCAN_MEMORY_PATH = DIRECT_SCAN_MEMORY_DIR / "direct_khlcnt_scan_memory.xlsx"

print("DIRECT_SCAN_MEMORY_PATH:", DIRECT_SCAN_MEMORY_PATH)
FORCE_REPROCESS_PDFS = False  # True nếu muốn đọc lại toàn bộ PDF, bỏ qua processed_pdf_registry

print("MASTER_OUTPUT_PATH:", MASTER_OUTPUT_PATH)
print("ACCOUNTS_IMPORT_PATH:", ACCOUNTS_IMPORT_PATH)
print("KHLCNT_IMPORT_PATH:", KHLCNT_IMPORT_PATH)
print("TBMT_IMPORT_PATH:", TBMT_IMPORT_PATH)
print("KHLCNT_ITEMS_MATCH_PATH:", KHLCNT_ITEMS_MATCH_PATH)
print("MATCH_CANDIDATES_HISTORY_PATH:", MATCH_CANDIDATES_HISTORY_PATH)


# ============================================================
# BASIC HELPERS
# ============================================================

def clean_text(value):
    if value is None:
        return ""
    value = str(value)
    value = value.replace("\xa0", " ")
    value = value.replace("\ufeff", "")
    value = re.sub(r"[ \t]+", " ", value)
    value = re.sub(r"\n+", " ", value)
    value = value.strip(" ,")
    if value in ["-", ""]:
        return ""
    return value


def normalize_key(value):
    value = clean_text(value).lower()
    value = unicodedata.normalize("NFD", value)
    value = "".join(ch for ch in value if unicodedata.category(ch) != "Mn")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return value.strip()


def money_to_number(value):
    value = clean_text(value)
    if not value:
        return ""
    digits = re.sub(r"[^\d]", "", value)
    return int(digits) if digits else ""


def parse_vn_datetime(value):
    value = clean_text(value)
    if not value:
        return ""

    m = re.search(r"(\d{1,2})\s*giờ\s*(\d{1,2})\s*ngày\s*(\d{2}/\d{2}/\d{4})", value)
    if m:
        hour = int(m.group(1))
        minute = int(m.group(2))
        date = m.group(3)
        return f"{date} {hour:02d}:{minute:02d}"

    m = re.search(r"(\d{2}/\d{2}/\d{4})\s+(\d{1,2}:\d{2})", value)
    if m:
        return f"{m.group(1)} {m.group(2)}"

    return value


def date_only(value):
    value = clean_text(value)
    m = re.search(r"(\d{2}/\d{2}/\d{4})", value)
    return m.group(1) if m else ""


def get_field(block, label, labels):
    other_labels = [re.escape(x) for x in labels if x != label]
    next_label_pattern = "|".join(other_labels)
    pattern = re.escape(label) + r"\s*(.*?)(?=\s*(?:" + next_label_pattern + r")|$)"
    m = re.search(pattern, block, flags=re.S)
    return clean_text(m.group(1)) if m else ""


def file_sha256(path, chunk_size=1024 * 1024):
    """Hash file để tránh PDF đổi tên nhưng nội dung giống nhau bị xử lý lại."""
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def safe_replace_excel(tmp_path, final_path):
    """Ghi file tạm xong mới replace file chính, tránh Colab ngắt làm hư output."""
    tmp_path = Path(tmp_path)
    final_path = Path(final_path)
    if not tmp_path.exists():
        raise FileNotFoundError(f"Không thấy file tạm để replace: {tmp_path}")
    tmp_path.replace(final_path)


def save_checkpoint_workbook(path, sheets_dict):
    """Checkpoint nhẹ sau mỗi file PDF để nếu Colab dừng vẫn còn dữ liệu tạm."""
    path = Path(path)
    tmp_path = path.with_suffix(".tmp.xlsx")
    with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
        for sheet_name, df in sheets_dict.items():
            if df is None:
                df = pd.DataFrame()
            df.to_excel(writer, index=False, sheet_name=sheet_name[:31])
    safe_replace_excel(tmp_path, path)


def extract_text_from_pdf(pdf_path, use_ocr_if_empty=True, force_reextract=False):
    """
    Đọc PDF có text cache theo file hash.
    - Nếu PDF đổi tên nhưng binary giống nhau: đọc lại text cache, không OCR lại.
    - Nếu Colab ngắt giữa chừng: lần sau các file đã cache sẽ chạy nhanh hơn.
    """
    pdf_path = Path(pdf_path)
    pdf_hash = file_sha256(pdf_path)
    cache_path = TEXT_CACHE_DIR / f"{pdf_hash}.txt"

    if cache_path.exists() and not force_reextract:
        print(f"Using cached text: {pdf_path.name}")
        return cache_path.read_text(encoding="utf-8", errors="ignore")

    doc = fitz.open(pdf_path)
    pages_text = []

    for page_idx, page in enumerate(tqdm(doc, desc=f"Reading {pdf_path.name}")):
        text = page.get_text("text").strip()

        if text:
            pages_text.append(text)
            continue

        if use_ocr_if_empty:
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            img_path = f"/content/_ocr_{pdf_path.stem}_{page_idx}.png"
            pix.save(img_path)
            img = Image.open(img_path)
            ocr_text = pytesseract.image_to_string(img, lang="vie+eng")
            pages_text.append(ocr_text)

    full_text = "\n".join(pages_text)

    tmp_cache = cache_path.with_suffix(".tmp.txt")
    tmp_cache.write_text(full_text, encoding="utf-8")
    tmp_cache.replace(cache_path)

    return full_text


def classify_product(ten_goi, du_an):
    text = normalize_key(ten_goi + " " + du_an)

    if any(k in text for k in ["thuoc", "vac xin", "vaccine"]):
        return "Hàng hóa", "Thuốc"
    if any(k in text for k in ["hoa chat", "sinh pham", "invitro", "ivd", "xet nghiem"]):
        return "Hàng hóa", "Hóa chất IVD - vật tư"
    if any(k in text for k in ["vat tu", "tieu hao", "y dung cu"]):
        return "Hàng hóa", "Vật tư y tế"
    if any(k in text for k in ["thiet bi", "may", "giuong", "tu dau giuong"]):
        return "Hàng hóa", "Thiết bị"
    if any(k in text for k in ["thue", "dich vu", "phan mem", "sua chua", "kiem dinh"]):
        return "Phi tư vấn", "Dịch vụ"

    return "Hàng hóa", ""

# ============================================================
# TEXT-SAFE EXCEL HELPERS
# Giữ mã và version dạng text: 00, 01, 02 không bị Excel đổi thành 0,1,2
# ============================================================

from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment


TEXT_SAFE_COLUMNS = [
    "Số TBMT",
    "Mã gốc TBMT",
    "Mã Phiên Bản",
    "Phiên Bản Trước dự kiến",
    "Số KHLCNT",
    "Số KHLCNT scan từ PDF",
    "Mã gốc KHLCNT",
    "Phiên bản KHLCNT",
    "Phiên Bản Trước_KHLC dự kiến",
    "Mã TBMT match mới nhất",
    "Mã TBMT mới nhất",
    "TBMT mới nhất",
    "Số KHLCNT ứng viên",
    "KHLCNT Item Key match",
    "KHLCNT Item Key",
]


def is_empty_text(v):
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except Exception:
        pass
    return str(v).strip() == "" or str(v).strip().lower() in ["nan", "none"]


def fix_version_value(v):
    """
    0    -> 00
    1    -> 01
    1.0  -> 01
    '02' -> 02
    """
    if is_empty_text(v):
        return ""

    s = str(v).strip()

    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]

    if re.fullmatch(r"\d+", s):
        return s.zfill(2)

    return s


def fix_text_value(v):
    if is_empty_text(v):
        return ""

    s = str(v).strip()

    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]

    return s


def fix_code_text_columns(df):
    """
    Fix các cột mã và version sau khi read_excel hoặc trước khi export.
    """
    if df is None or df.empty:
        return df

    df = df.copy()

    version_cols = [
        "Mã Phiên Bản",
        "Phiên bản KHLCNT",
    ]

    for col in version_cols:
        if col in df.columns:
            df[col] = df[col].apply(fix_version_value).astype(str)

    code_cols = [
        "Số TBMT",
        "Mã gốc TBMT",
        "Phiên Bản Trước dự kiến",
        "Số KHLCNT",
        "Mã gốc KHLCNT",
        "Phiên Bản Trước_KHLC dự kiến",
        "Mã TBMT match mới nhất",
        "Mã TBMT mới nhất",
        "TBMT mới nhất",
        "Số KHLCNT ứng viên",
    ]

    for col in code_cols:
        if col in df.columns:
            df[col] = df[col].apply(fix_text_value).astype(str)

    return df


def write_excel_sheet_text_safe(writer, df, sheet_name, text_columns=None):
    """
    Ghi Excel và format các cột mã/version là Text.
    """
    text_columns = text_columns or []

    df = fix_code_text_columns(df)
    df.to_excel(writer, index=False, sheet_name=sheet_name)

    ws = writer.sheets[sheet_name]
    ws.freeze_panes = "A2"

    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center")

    for idx, col_name in enumerate(df.columns, start=1):
        col_letter = get_column_letter(idx)

        if col_name in text_columns:
            for cell in ws[col_letter]:
                cell.number_format = "@"

        try:
            values = df[col_name].head(100).fillna("").astype(str).tolist()
            max_len = max([len(str(col_name))] + [len(x) for x in values])
            ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 45)
        except Exception:
            pass
# ============================================================
# PDF REGISTRY HELPERS
# ============================================================

def load_processed_registry():
    if PROCESSED_PDF_REGISTRY.exists():
        try:
            df = pd.read_excel(PROCESSED_PDF_REGISTRY)
        except Exception:
            df = pd.DataFrame()
    else:
        df = pd.DataFrame()

    needed_cols = ["source_type", "file_name", "file_path", "size_bytes", "modified_time", "file_hash", "processed_at"]

    for col in needed_cols:
        if col not in df.columns:
            df[col] = ""

    return df[needed_cols]


def get_pdf_info(pdf_path: Path, source_type: str):
    stat = pdf_path.stat()
    return {
        "source_type": source_type,
        "file_name": pdf_path.name,
        "file_path": str(pdf_path),
        "size_bytes": stat.st_size,
        "modified_time": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "file_hash": file_sha256(pdf_path),
    }


def filter_new_pdfs(pdf_paths, source_type, registry_df):
    """
    Skip PDF đã xử lý theo 2 lớp:
    1) source_type + file_name: tránh chạy lại file cũ.
    2) source_type + file_hash: tránh PDF đổi tên nhưng nội dung binary giống nhau.
    """
    if FORCE_REPROCESS_PDFS:
        print("FORCE_REPROCESS_PDFS=True: sẽ xử lý lại toàn bộ PDF trong thư mục.")
        return list(pdf_paths), []

    registry_df = registry_df.copy()

    old_name_keys = set(zip(
        registry_df["source_type"].astype(str),
        registry_df["file_name"].astype(str)
    ))

    old_hash_keys = set()
    if "file_hash" in registry_df.columns:
        old_hash_keys = set(zip(
            registry_df["source_type"].astype(str),
            registry_df["file_hash"].astype(str)
        ))

    new_files = []
    skipped_files = []

    for p in pdf_paths:
        name_key = (source_type, p.name)
        try:
            hash_key = (source_type, file_sha256(p))
        except Exception:
            hash_key = (source_type, "")

        if name_key in old_name_keys or (hash_key[1] and hash_key in old_hash_keys):
            skipped_files.append(p)
        else:
            new_files.append(p)

    return new_files, skipped_files


def append_processed_registry(processed_files):
    if not processed_files:
        print("Không có PDF mới để ghi registry.")
        return

    old_df = load_processed_registry()

    new_df = pd.DataFrame(processed_files)
    new_df["processed_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    combined = pd.concat([old_df, new_df], ignore_index=True, sort=False)
    # Dedupe registry theo tên file và hash để tránh xử lý lại file đổi tên
    if "file_hash" in combined.columns:
        combined = combined.drop_duplicates(subset=["source_type", "file_hash"], keep="last")
    combined = combined.drop_duplicates(subset=["source_type", "file_name"], keep="last")

    tmp_registry = PROCESSED_PDF_REGISTRY.with_suffix(".tmp.xlsx")
    combined.to_excel(tmp_registry, index=False)
    safe_replace_excel(tmp_registry, PROCESSED_PDF_REGISTRY)
    print("Updated processed PDF registry:", PROCESSED_PDF_REGISTRY)


def read_existing_sheet(path, sheet_name, columns=None):
    if not path.exists():
        return pd.DataFrame(columns=columns or [])

    try:
        df = pd.read_excel(path, sheet_name=sheet_name)
        if columns:
            for c in columns:
                if c not in df.columns:
                    df[c] = ""
            df = df[columns + [c for c in df.columns if c not in columns]]
        return df
    except Exception:
        return pd.DataFrame(columns=columns or [])


# ============================================================
# TBMT PARSER
# ============================================================

TBMT_LABELS = [
    "Bên mời thầu:", "Địa chỉ bên mời thầu:", "Dự án:", "Gói thầu:",
    "Giá gói thầu (VND):", "Nguồn vốn:", "Phương thức lựa chọn nhà thầu:",
    "Hình thức lựa chọn nhà thầu:", "Thời gian phát hành hồ sơ:",
    "Bảo đảm dự thầu (VND):", "Hình thức bảo đảm dự thầu:", "Địa điểm phát hành:",
    "Thời gian đóng thầu:", "Thời gian mở thầu:", "Thời gian thực hiện hợp đồng:",
]

ZOHO_TBMT_COLUMNS = [
    "STT nguồn", "File nguồn", "Số TBMT", "Mã gốc TBMT", "Mã Phiên Bản",
    "Phiên Bản Mới Nhất", "Phiên Bản Trước dự kiến", "Số KHLCNT","Số KHLCNT scan từ PDF",
    "Chủ đầu tư", "Đơn vị mời thầu", "Địa chỉ bên mời thầu", "Dự án",
    "Tên gói thầu", "Mã gói thầu", "Lĩnh vực", "Nhóm sản phẩm",
    "Giá gói thầu", "Tổng giá trị", "Nguồn vốn", "Phương thức lựa chọn nhà thầu",
    "Hình thức lựa chọn nhà thầu", "Trong nước/Quốc tế", "Hình thức nộp thầu",
    "Online/Offline", "Thời điểm đăng tải", "Ngày đăng", "Thời gian phát hành hồ sơ",
    "Bảo đảm dự thầu", "Hình thức bảo đảm dự thầu", "Địa điểm phát hành e-HSMT",
    "Địa điểm nhận e-HSDT", "Thời điểm đóng thầu", "Ngày đóng", "Thời điểm mở thầu",
    "Thời gian thực hiện hợp đồng", "Trạng thái match KHLCNT", "Điểm match KHLCNT",
    "Nguồn match KHLCNT", "Ghi chú match KHLCNT", "Match key đơn vị", "Match key giá", "Match key tên gói",
]


def split_tbmt_records(text):
    pattern = re.compile(r"(?m)^\s*(\d+)\.\s+Bên mời thầu:")
    matches = list(pattern.finditer(text))
    records = []

    for i, m in enumerate(matches):
        stt = int(m.group(1))
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        records.append((stt, text[start:end]))

    return records


def extract_tbmt_code(goi_thau):
    text = clean_text(goi_thau)
    m = re.search(r"(?:Số thông báo|Mã TBMT)\s*:\s*(IB\d+)\s*-\s*(\d{2})", text)
    if not m:
        return "", "", ""
    root = m.group(1)
    version = m.group(2)
    return f"{root}-{version}", root, version
def normalize_khlcnt_code(v):
    s = clean_text(v).upper().replace(" ", "")
    m = re.search(r"(PL\d+)\s*-?\s*(\d{2})", s)
    if m:
        return f"{m.group(1)}-{m.group(2)}"
    return s
def extract_khlcnt_code_from_tbmt(block):
    """
    Tìm Số KHLCNT trực tiếp trong block TBMT.
    Ví dụ:
    - Số KHLCNT: PL2600034802-00
    - KHLCNT: PL2600034802-00
    - Kế hoạch lựa chọn nhà thầu: PL2600034802-00
    - Số kế hoạch lựa chọn nhà thầu: PL2600034802-00
    """
    text = clean_text(block)
    if not text:
        return ""

    patterns = [
        r"Số\s+KHLCNT\s*[:：]?\s*(PL\d+\s*-\s*\d{2})",
        r"KHLCNT\s*[:：]?\s*(PL\d+\s*-\s*\d{2})",
        r"Kế\s*hoạch\s*lựa\s*chọn\s*nhà\s*thầu\s*[:：]?\s*(PL\d+\s*-\s*\d{2})",
        r"Số\s+kế\s*hoạch\s*lựa\s*chọn\s*nhà\s*thầu\s*[:：]?\s*(PL\d+\s*-\s*\d{2})",
        r"(PL\d+\s*-\s*\d{2})",
    ]

    for pat in patterns:
        m = re.search(pat, text, flags=re.I)
        if m:
            return normalize_khlcnt_code(m.group(1)) if "normalize_khlcnt_code" in globals() else m.group(1).replace(" ", "").upper()

    return ""

def extract_posted_datetime(goi_thau):
    text = clean_text(goi_thau)
    m = re.search(r"Thời điểm đăng tải:\s*(\d{2}/\d{2}/\d{4}\s+\d{1,2}:\d{2})", text)
    return m.group(1) if m else ""


def clean_goi_thau_name(goi_thau):
    text = clean_text(goi_thau)
    text = re.sub(r"\((?:Số thông báo|Mã TBMT)\s*:.*?\)\s*$", "", text, flags=re.S).strip()
    return clean_text(text)


def parse_tbmt_record(stt, block, filename):
    ben_moi_thau_raw = get_field(block, "Bên mời thầu:", TBMT_LABELS)
    dia_chi = get_field(block, "Địa chỉ bên mời thầu:", TBMT_LABELS)
    du_an = get_field(block, "Dự án:", TBMT_LABELS)
    goi_thau_raw = get_field(block, "Gói thầu:", TBMT_LABELS)

    gia_goi = money_to_number(get_field(block, "Giá gói thầu (VND):", TBMT_LABELS))
    nguon_von = get_field(block, "Nguồn vốn:", TBMT_LABELS)
    phuong_thuc = get_field(block, "Phương thức lựa chọn nhà thầu:", TBMT_LABELS)
    hinh_thuc = get_field(block, "Hình thức lựa chọn nhà thầu:", TBMT_LABELS)
    phat_hanh = parse_vn_datetime(get_field(block, "Thời gian phát hành hồ sơ:", TBMT_LABELS))
    bao_dam = money_to_number(get_field(block, "Bảo đảm dự thầu (VND):", TBMT_LABELS))
    hinh_thuc_bao_dam = get_field(block, "Hình thức bảo đảm dự thầu:", TBMT_LABELS)
    dia_diem_phat_hanh = get_field(block, "Địa điểm phát hành:", TBMT_LABELS)
    dong_thau = parse_vn_datetime(get_field(block, "Thời gian đóng thầu:", TBMT_LABELS))
    mo_thau = parse_vn_datetime(get_field(block, "Thời gian mở thầu:", TBMT_LABELS))
    thuc_hien_hd = get_field(block, "Thời gian thực hiện hợp đồng:", TBMT_LABELS)

    ben_moi_thau = ben_moi_thau_raw
    if "Chủ đầu tư" in ben_moi_thau_raw:
        parts = ben_moi_thau_raw.split(":")
        ben_moi_thau = clean_text(parts[-1])

    chu_dau_tu = ben_moi_thau

    ma_tbmt, ma_tbmt_goc, phien_ban_tbmt = extract_tbmt_code(goi_thau_raw)
    so_khlcnt_scan_pdf = extract_khlcnt_code_from_tbmt(block)
    thoi_diem_dang_tai = extract_posted_datetime(goi_thau_raw)
    ten_goi = clean_goi_thau_name(goi_thau_raw)

    linh_vuc, nhom_sp = classify_product(ten_goi, du_an)

    trong_nuoc_quoc_te = ""
    if "trong nước" in hinh_thuc.lower():
        trong_nuoc_quoc_te = "Trong nước"
    elif "quốc tế" in hinh_thuc.lower():
        trong_nuoc_quoc_te = "Quốc tế"

    online_offline = "Online" if "muasamcong" in dia_diem_phat_hanh.lower() else ""

    return {
        "STT nguồn": stt, "File nguồn": filename, "Số TBMT": ma_tbmt,
        "Mã gốc TBMT": ma_tbmt_goc, "Mã Phiên Bản": phien_ban_tbmt,
        "Phiên Bản Mới Nhất": False, "Phiên Bản Trước dự kiến": "", "Số KHLCNT": "",
        "Số KHLCNT scan từ PDF": so_khlcnt_scan_pdf,
        "Chủ đầu tư": chu_dau_tu, "Đơn vị mời thầu": ben_moi_thau,
        "Địa chỉ bên mời thầu": dia_chi, "Dự án": du_an, "Tên gói thầu": ten_goi,
        "Mã gói thầu": "", "Lĩnh vực": linh_vuc, "Nhóm sản phẩm": nhom_sp,
        "Giá gói thầu": gia_goi, "Tổng giá trị": gia_goi, "Nguồn vốn": nguon_von,
        "Phương thức lựa chọn nhà thầu": phuong_thuc, "Hình thức lựa chọn nhà thầu": hinh_thuc,
        "Trong nước/Quốc tế": trong_nuoc_quoc_te, "Hình thức nộp thầu": online_offline,
        "Online/Offline": online_offline, "Thời điểm đăng tải": thoi_diem_dang_tai,
        "Ngày đăng": date_only(thoi_diem_dang_tai), "Thời gian phát hành hồ sơ": phat_hanh,
        "Bảo đảm dự thầu": bao_dam, "Hình thức bảo đảm dự thầu": hinh_thuc_bao_dam,
        "Địa điểm phát hành e-HSMT": dia_diem_phat_hanh,
        "Địa điểm nhận e-HSDT": dia_diem_phat_hanh, "Thời điểm đóng thầu": dong_thau,
        "Ngày đóng": date_only(dong_thau), "Thời điểm mở thầu": mo_thau,
        "Thời gian thực hiện hợp đồng": thuc_hien_hd,
        "Trạng thái match KHLCNT": "Scan được Số KHLCNT từ PDF" if so_khlcnt_scan_pdf else "Chưa match",
        "Điểm match KHLCNT": "",
        "Nguồn match KHLCNT": "Scan trực tiếp từ TBMT PDF" if so_khlcnt_scan_pdf else "",
        "Ghi chú match KHLCNT": f"Scan được Số KHLCNT từ TBMT PDF: {so_khlcnt_scan_pdf}" if so_khlcnt_scan_pdf else "",
        "Match key đơn vị": normalize_key(chu_dau_tu),
        "Match key giá": str(gia_goi) if gia_goi != "" else "",
        "Match key tên gói": normalize_key(ten_goi),
    }


# ============================================================
# KHLCNT PARSER
# ============================================================

KHLCNT_LABELS = [
    "Số KHLCNT:", "Tên dự án:", "Tên dự toán mua sắm:", "Tên chủ đầu tư:",
    "Tổng mức đầu tư:", "Nội dung phê duyệt:", "Tên gói thầu:", "Nguồn vốn:",
    "Giá gói thầu:", "Hình thức lựa chọn:", "Hình thức hợp đồng:",
    "Thời gian lựa chọn:", "Thời gian thực hiện:",
]

ZOHO_KHLCNT_COLUMNS = [
    "STT nguồn", "File nguồn", "Số KHLCNT", "Mã gốc KHLCNT", "Phiên bản KHLCNT",
    "KHLCNT mới nhất", "Phiên Bản Trước_KHLC dự kiến", "Trạng thái KHLCNT",
    "Chủ đầu tư-KHLCNT", "Tên dự án", "Tên dự toán mua sắm", "Tổng mức đầu tư",
    "Nội dung phê duyệt", "Ngày phê duyệt", "Nguồn vốn", "Phân loại KHLCNT",
    "Lĩnh vực tổng quát", "Tên gói thầu trong KHMT", "Giá gói thầu trong KHMT",
    "Online/Offline", "Sơ tuyển", "Phương thức lựa chọn", "Hình thức lựa chọn",
    "Hình thức hợp đồng", "Thời gian lựa chọn", "Thời gian thực hiện theo KHMT",
    "Đã match TBMT?", "Mã TBMT match mới nhất", "Ghi chú gói KHMT", "TBMT mới nhất",
    "Mã TBMT mới nhất", "Match key đơn vị", "Match key giá", "Match key tên gói",
]

# Bảng phụ dùng để match chi tiết từng gói thầu con trong KHLCNT.
# KHLCNT_Zoho_Import vẫn là bảng cha để import Zoho: 1 Số KHLCNT = 1 record.
# KHLCNT_Items_Match chỉ dùng trong Colab: 1 gói thầu con = 1 dòng.
ZOHO_KHLCNT_ITEM_COLUMNS = [
    "KHLCNT Item Key", "STT nguồn", "File nguồn", "Số KHLCNT", "Mã gốc KHLCNT", "Phiên bản KHLCNT",
    "KHLCNT mới nhất", "Chủ đầu tư-KHLCNT", "Tên dự án", "Tên dự toán mua sắm", "Tổng mức đầu tư",
    "Nội dung phê duyệt", "Ngày phê duyệt", "Nguồn vốn", "Phân loại KHLCNT", "Lĩnh vực tổng quát",
    "Tên gói thầu trong KHMT", "Giá gói thầu trong KHMT", "Online/Offline", "Sơ tuyển",
    "Phương thức lựa chọn", "Hình thức lựa chọn", "Hình thức hợp đồng", "Thời gian lựa chọn",
    "Thời gian thực hiện theo KHMT", "Match item order", "Match key đơn vị", "Match key giá", "Match key tên gói",
]


def split_khlcnt_records(text):
    pattern = re.compile(r"(?m)^\s*(\d+)\.\s+Kế hoạch lựa chọn nhà thầu")
    matches = list(pattern.finditer(text))
    records = []

    for i, m in enumerate(matches):
        stt = int(m.group(1))
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        records.append((stt, text[start:end]))

    return records


def extract_khlcnt_code(value):
    value = clean_text(value)
    m = re.search(r"(PL\d+)\s*-\s*(\d{2})", value)
    if not m:
        return value, "", ""
    root = m.group(1)
    version = m.group(2)
    return f"{root}-{version}", root, version


def extract_approval_date(value):
    value = clean_text(value)
    m = re.search(r"(\d{2}/\d{2}/\d{4})", value)
    return m.group(1) if m else ""


def parse_hinh_thuc_lua_chon(value):
    value = clean_text(value)
    parts = [clean_text(p) for p in value.split(",")]

    hinh_thuc = parts[0] if len(parts) > 0 else ""
    online_offline = ""
    so_tuyen = ""
    phuong_thuc = ""

    for p in parts[1:]:
        low = p.lower()
        if "qua mạng" in low or "không qua mạng" in low:
            online_offline = p
        elif "sơ tuyển" in low:
            so_tuyen = p
        elif "giai đoạn" in low or "túi hồ sơ" in low:
            phuong_thuc = p

    return hinh_thuc, online_offline, so_tuyen, phuong_thuc


def parse_khlcnt_record(stt, block, filename):
    so_khlcnt_raw = get_field(block, "Số KHLCNT:", KHLCNT_LABELS)
    so_khlcnt, ma_goc, phien_ban = extract_khlcnt_code(so_khlcnt_raw)

    ten_du_an = get_field(block, "Tên dự án:", KHLCNT_LABELS)
    ten_du_toan = get_field(block, "Tên dự toán mua sắm:", KHLCNT_LABELS)
    chu_dau_tu = get_field(block, "Tên chủ đầu tư:", KHLCNT_LABELS)

    if not ten_du_an:
        ten_du_an = ten_du_toan

    tong_muc = money_to_number(get_field(block, "Tổng mức đầu tư:", KHLCNT_LABELS))
    noi_dung_phe_duyet = get_field(block, "Nội dung phê duyệt:", KHLCNT_LABELS)
    ngay_phe_duyet = extract_approval_date(noi_dung_phe_duyet)
    ten_goi = get_field(block, "Tên gói thầu:", KHLCNT_LABELS)
    nguon_von = get_field(block, "Nguồn vốn:", KHLCNT_LABELS)
    gia_goi = money_to_number(get_field(block, "Giá gói thầu:", KHLCNT_LABELS))
    hinh_thuc_raw = get_field(block, "Hình thức lựa chọn:", KHLCNT_LABELS)
    hinh_thuc_hd = get_field(block, "Hình thức hợp đồng:", KHLCNT_LABELS)
    thoi_gian_lua_chon = get_field(block, "Thời gian lựa chọn:", KHLCNT_LABELS)
    thoi_gian_thuc_hien = get_field(block, "Thời gian thực hiện:", KHLCNT_LABELS)

    hinh_thuc, online_offline, so_tuyen, phuong_thuc = parse_hinh_thuc_lua_chon(hinh_thuc_raw)
    linh_vuc, nhom_sp = classify_product(ten_goi, ten_du_an)

    return {
        "STT nguồn": stt, "File nguồn": filename, "Số KHLCNT": so_khlcnt,
        "Mã gốc KHLCNT": ma_goc, "Phiên bản KHLCNT": phien_ban,
        "KHLCNT mới nhất": False, "Phiên Bản Trước_KHLC dự kiến": "",
        "Trạng thái KHLCNT": "Đang có hiệu lực", "Chủ đầu tư-KHLCNT": chu_dau_tu,
        "Tên dự án": ten_du_an, "Tên dự toán mua sắm": ten_du_toan,
        "Tổng mức đầu tư": tong_muc, "Nội dung phê duyệt": noi_dung_phe_duyet,
        "Ngày phê duyệt": ngay_phe_duyet, "Nguồn vốn": nguon_von,
        "Phân loại KHLCNT": "Mua sắm", "Lĩnh vực tổng quát": linh_vuc,
        "Tên gói thầu trong KHMT": ten_goi, "Giá gói thầu trong KHMT": gia_goi,
        "Online/Offline": online_offline, "Sơ tuyển": so_tuyen,
        "Phương thức lựa chọn": phuong_thuc, "Hình thức lựa chọn": hinh_thuc,
        "Hình thức hợp đồng": hinh_thuc_hd, "Thời gian lựa chọn": thoi_gian_lua_chon,
        "Thời gian thực hiện theo KHMT": thoi_gian_thuc_hien,
        "Đã match TBMT?": False, "Mã TBMT match mới nhất": "", "Ghi chú gói KHMT": "",
        "TBMT mới nhất": "", "Mã TBMT mới nhất": "",
        "Match key đơn vị": normalize_key(chu_dau_tu),
        "Match key giá": str(gia_goi) if gia_goi != "" else "",
        "Match key tên gói": normalize_key(ten_goi),
    }



# ============================================================
# KHLCNT ITEMS HELPERS
# - Dùng để giữ nhiều gói thầu con trong cùng một Số KHLCNT khi match.
# - Không thay đổi bảng KHLCNT_Zoho_Import dùng cho Zoho.
# ============================================================

def safe_slug(value, max_len=80):
    s = normalize_key(value)
    s = re.sub(r"\s+", "-", s).strip("-")
    return s[:max_len] if s else "blank"


def make_khlcnt_item_key(so_khlcnt, ten_goi, gia_goi, order_no):
    so_khlcnt = fix_text_value(so_khlcnt)
    gia_txt = fix_text_value(gia_goi)
    ten_slug = safe_slug(ten_goi, max_len=90)
    return f"{so_khlcnt}__{order_no:03d}__{ten_slug}__{gia_txt}"


def split_khlcnt_item_blocks(block):
    """
    Tách các gói thầu con trong một KHLCNT block.
    Nếu PDF chỉ có 1 gói, hàm trả về 1 block.
    """
    starts = [m.start() for m in re.finditer(r"Tên\s+gói\s+thầu\s*:", block, flags=re.I)]

    if not starts:
        return [block]

    item_blocks = []
    for i, start in enumerate(starts):
        end = starts[i + 1] if i + 1 < len(starts) else len(block)
        item_blocks.append(block[start:end])

    return item_blocks


def parse_khlcnt_items(stt, block, filename):
    """
    Parse item-level rows để match TBMT với đúng gói thầu con.
    Parent vẫn dùng parse_khlcnt_record() để import Zoho.
    """
    parent = parse_khlcnt_record(stt, block, filename)
    item_blocks = split_khlcnt_item_blocks(block)
    rows = []

    for order_no, item_block in enumerate(item_blocks, start=1):
        ten_goi = get_field(item_block, "Tên gói thầu:", KHLCNT_LABELS)
        if not ten_goi:
            ten_goi = parent.get("Tên gói thầu trong KHMT", "")

        nguon_von = get_field(item_block, "Nguồn vốn:", KHLCNT_LABELS) or parent.get("Nguồn vốn", "")
        gia_goi = money_to_number(get_field(item_block, "Giá gói thầu:", KHLCNT_LABELS))
        if gia_goi == "":
            gia_goi = parent.get("Giá gói thầu trong KHMT", "")

        hinh_thuc_raw = get_field(item_block, "Hình thức lựa chọn:", KHLCNT_LABELS)
        if not hinh_thuc_raw:
            # Rebuild từ parent để tránh mất thông tin nếu item block không đủ field.
            hinh_thuc_raw = ", ".join([
                fix_text_value(parent.get("Hình thức lựa chọn", "")),
                fix_text_value(parent.get("Online/Offline", "")),
                fix_text_value(parent.get("Sơ tuyển", "")),
                fix_text_value(parent.get("Phương thức lựa chọn", "")),
            ]).strip(" ,")

        hinh_thuc, online_offline, so_tuyen, phuong_thuc = parse_hinh_thuc_lua_chon(hinh_thuc_raw)

        hinh_thuc_hd = get_field(item_block, "Hình thức hợp đồng:", KHLCNT_LABELS) or parent.get("Hình thức hợp đồng", "")
        thoi_gian_lua_chon = get_field(item_block, "Thời gian lựa chọn:", KHLCNT_LABELS) or parent.get("Thời gian lựa chọn", "")
        thoi_gian_thuc_hien = get_field(item_block, "Thời gian thực hiện:", KHLCNT_LABELS) or parent.get("Thời gian thực hiện theo KHMT", "")

        linh_vuc, nhom_sp = classify_product(ten_goi, parent.get("Tên dự án", ""))

        so_khlcnt = parent.get("Số KHLCNT", "")
        item_key = make_khlcnt_item_key(so_khlcnt, ten_goi, gia_goi, order_no)

        rows.append({
            "KHLCNT Item Key": item_key,
            "STT nguồn": stt,
            "File nguồn": filename,
            "Số KHLCNT": so_khlcnt,
            "Mã gốc KHLCNT": parent.get("Mã gốc KHLCNT", ""),
            "Phiên bản KHLCNT": parent.get("Phiên bản KHLCNT", ""),
            "KHLCNT mới nhất": parent.get("KHLCNT mới nhất", False),
            "Chủ đầu tư-KHLCNT": parent.get("Chủ đầu tư-KHLCNT", ""),
            "Tên dự án": parent.get("Tên dự án", ""),
            "Tên dự toán mua sắm": parent.get("Tên dự toán mua sắm", ""),
            "Tổng mức đầu tư": parent.get("Tổng mức đầu tư", ""),
            "Nội dung phê duyệt": parent.get("Nội dung phê duyệt", ""),
            "Ngày phê duyệt": parent.get("Ngày phê duyệt", ""),
            "Nguồn vốn": nguon_von,
            "Phân loại KHLCNT": parent.get("Phân loại KHLCNT", "Mua sắm"),
            "Lĩnh vực tổng quát": linh_vuc,
            "Tên gói thầu trong KHMT": ten_goi,
            "Giá gói thầu trong KHMT": gia_goi,
            "Online/Offline": online_offline,
            "Sơ tuyển": so_tuyen,
            "Phương thức lựa chọn": phuong_thuc,
            "Hình thức lựa chọn": hinh_thuc,
            "Hình thức hợp đồng": hinh_thuc_hd,
            "Thời gian lựa chọn": thoi_gian_lua_chon,
            "Thời gian thực hiện theo KHMT": thoi_gian_thuc_hien,
            "Match item order": order_no,
            "Match key đơn vị": normalize_key(parent.get("Chủ đầu tư-KHLCNT", "")),
            "Match key giá": str(gia_goi) if gia_goi != "" else "",
            "Match key tên gói": normalize_key(ten_goi),
        })

    return rows


def mark_khlcnt_items_latest(df_items):
    if df_items.empty or "Mã gốc KHLCNT" not in df_items.columns or "Phiên bản KHLCNT" not in df_items.columns:
        return df_items

    df_items = df_items.copy()
    df_items["_kh_version_num"] = pd.to_numeric(
        df_items["Phiên bản KHLCNT"].astype(str).str.replace(" ", "", regex=False),
        errors="coerce"
    ).fillna(-1).astype(int)

    max_version = df_items.groupby("Mã gốc KHLCNT")["_kh_version_num"].transform("max")
    df_items["KHLCNT mới nhất"] = df_items["_kh_version_num"] == max_version
    df_items = df_items.drop(columns=["_kh_version_num"])
    return df_items

# ============================================================
# VERSION + DEDUPE + MATCH HELPERS
# ============================================================

def mark_latest_versions(df, root_col, version_col, full_code_col, latest_col, prev_col):
    if df.empty or root_col not in df.columns or version_col not in df.columns:
        return df

    df = df.copy()

    df["_version_num"] = pd.to_numeric(
        df[version_col].astype(str).str.replace(" ", "", regex=False),
        errors="coerce"
    ).fillna(-1).astype(int)

    max_version = df.groupby(root_col)["_version_num"].transform("max")
    df[latest_col] = df["_version_num"] == max_version

    df = df.sort_values([root_col, "_version_num"], ascending=[True, True])
    df[prev_col] = df.groupby(root_col)[full_code_col].shift(1).fillna("")

    df = df.drop(columns=["_version_num"])
    return df


def clean_key_series(s):
    return s.astype(str).str.strip().replace({"nan": "", "None": "", "NaN": ""})


def dedupe_by_key(df, key_col, keep="last"):
    if df.empty or key_col not in df.columns:
        return df

    tmp = df.copy()
    tmp["_dedupe_key"] = clean_key_series(tmp[key_col])
    tmp = (
        tmp[tmp["_dedupe_key"] != ""]
        .drop_duplicates("_dedupe_key", keep=keep)
        .drop(columns=["_dedupe_key"])
        .reset_index(drop=True)
    )
    return tmp
DIRECT_SCAN_MEMORY_COLUMNS = [
    "Số TBMT",
    "Tên gói thầu",
    "Dự án",
    "Chủ đầu tư",
    "Giá gói thầu",
    "Số KHLCNT scan từ PDF",
    "Số KHLCNT chuẩn hóa",
    "Trạng thái direct scan",
    "Ghi chú",
    "First seen at",
    "Last checked at",
    "Source",
]


def load_direct_scan_memory():
    if DIRECT_SCAN_MEMORY_PATH.exists():
        try:
            df = pd.read_excel(DIRECT_SCAN_MEMORY_PATH, dtype=str)
        except Exception:
            df = pd.DataFrame()
    else:
        df = pd.DataFrame()

    for col in DIRECT_SCAN_MEMORY_COLUMNS:
        if col not in df.columns:
            df[col] = ""

    return df[DIRECT_SCAN_MEMORY_COLUMNS]


def update_direct_scan_memory_and_apply(df_tbmt, df_khlcnt):
    """
    Cell 2 xử lý riêng direct scan:
    - Nếu scan được Số KHLCNT từ TBMT PDF và KHLCNT tổng đã có:
        update thẳng Số KHLCNT vào df_tbmt.
    - Nếu scan được nhưng KHLCNT tổng chưa có:
        lưu vào direct_scan_memory, không ghi lookup để tránh lỗi import Zoho.
    - Những dòng có direct scan sẽ không cần fuzzy so sánh tên gói/dự án/giá nữa.
    """
    if df_tbmt is None or df_tbmt.empty:
        return df_tbmt

    df_tbmt = df_tbmt.copy()

    if "Số KHLCNT scan từ PDF" not in df_tbmt.columns:
        df_tbmt["Số KHLCNT scan từ PDF"] = ""

    if "Số KHLCNT" not in df_tbmt.columns:
        df_tbmt["Số KHLCNT"] = ""

    for col in ["Trạng thái match KHLCNT", "Điểm match KHLCNT", "Nguồn match KHLCNT", "Ghi chú match KHLCNT"]:
        if col not in df_tbmt.columns:
            df_tbmt[col] = ""

    kh_codes = set()
    if df_khlcnt is not None and not df_khlcnt.empty and "Số KHLCNT" in df_khlcnt.columns:
        kh_codes = set(
            df_khlcnt["Số KHLCNT"]
            .fillna("")
            .astype(str)
            .apply(normalize_khlcnt_code)
            .str.strip()
        )

    df_mem_old = load_direct_scan_memory()

    old_first_seen = {}
    if not df_mem_old.empty:
        for _, r in df_mem_old.iterrows():
            old_first_seen[clean_text(r.get("Số TBMT", ""))] = clean_text(r.get("First seen at", ""))

    memory_rows = []

    # 1) Lấy direct scan từ df_tbmt hiện tại
    for idx, row in df_tbmt.iterrows():
        so_tbmt = clean_text(row.get("Số TBMT", ""))
        kh_scan = normalize_khlcnt_code(row.get("Số KHLCNT scan từ PDF", ""))

        if not so_tbmt or not kh_scan:
            continue

        df_tbmt.at[idx, "Số KHLCNT scan từ PDF"] = kh_scan

        if kh_scan in kh_codes:
            df_tbmt.at[idx, "Số KHLCNT"] = kh_scan
            df_tbmt.at[idx, "Trạng thái match KHLCNT"] = "Direct KHLCNT valid"
            df_tbmt.at[idx, "Điểm match KHLCNT"] = 100
            df_tbmt.at[idx, "Nguồn match KHLCNT"] = "Scan trực tiếp từ TBMT PDF"
            df_tbmt.at[idx, "Ghi chú match KHLCNT"] = f"Số KHLCNT scan từ TBMT đã có trong KHLCNT tổng: {kh_scan}"

            status = "Đã có trong KHLCNT tổng - đã apply Cell 2"
            note = "Cell 2 đã update Số KHLCNT vào TBMT vì KHLCNT đã có trong tổng"
        else:
            # Không ghi lookup nếu chưa có KHLCNT tổng
            df_tbmt.at[idx, "Số KHLCNT"] = ""
            df_tbmt.at[idx, "Trạng thái match KHLCNT"] = "Direct KHLCNT chờ KHLCNT tổng"
            df_tbmt.at[idx, "Điểm match KHLCNT"] = ""
            df_tbmt.at[idx, "Nguồn match KHLCNT"] = "Scan trực tiếp từ TBMT PDF"
            df_tbmt.at[idx, "Ghi chú match KHLCNT"] = f"Scan được {kh_scan} nhưng chưa có trong KHLCNT tổng; lưu direct memory"

            status = "Chưa có trong KHLCNT tổng"
            note = "Scan được mã KHLCNT từ TBMT PDF; lưu memory, lần sau kiểm tra lại"

        memory_rows.append({
            "Số TBMT": so_tbmt,
            "Tên gói thầu": clean_text(row.get("Tên gói thầu", "")),
            "Dự án": clean_text(row.get("Dự án", "")),
            "Chủ đầu tư": clean_text(row.get("Chủ đầu tư", "")),
            "Giá gói thầu": clean_text(row.get("Giá gói thầu", "")),
            "Số KHLCNT scan từ PDF": kh_scan,
            "Số KHLCNT chuẩn hóa": kh_scan,
            "Trạng thái direct scan": status,
            "Ghi chú": note,
            "First seen at": old_first_seen.get(so_tbmt, datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
            "Last checked at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Source": "Direct scan TBMT PDF",
        })

    # 2) Giữ lại memory cũ nếu TBMT đó chưa xuất hiện lại trong df_tbmt
    current_tbmt_set = set([r["Số TBMT"] for r in memory_rows])

    if not df_mem_old.empty:
        for _, r in df_mem_old.iterrows():
            so_tbmt = clean_text(r.get("Số TBMT", ""))
            if so_tbmt and so_tbmt not in current_tbmt_set:
                old_row = {col: clean_text(r.get(col, "")) for col in DIRECT_SCAN_MEMORY_COLUMNS}
                old_row["Last checked at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                memory_rows.append(old_row)

    df_mem = pd.DataFrame(memory_rows)

    if not df_mem.empty:
        df_mem["Số TBMT"] = df_mem["Số TBMT"].fillna("").astype(str).str.strip()
        df_mem["Số KHLCNT chuẩn hóa"] = df_mem["Số KHLCNT chuẩn hóa"].fillna("").astype(str).apply(normalize_khlcnt_code)
        df_mem = df_mem.drop_duplicates(subset=["Số TBMT"], keep="last").reset_index(drop=True)
    else:
        df_mem = pd.DataFrame(columns=DIRECT_SCAN_MEMORY_COLUMNS)

    with pd.ExcelWriter(DIRECT_SCAN_MEMORY_PATH, engine="openpyxl") as writer:
        write_excel_sheet_text_safe(writer, df_mem, "Direct_Scan_Memory", TEXT_SAFE_COLUMNS + ["Số KHLCNT scan từ PDF", "Số KHLCNT chuẩn hóa"])

    print("Direct scan memory:", DIRECT_SCAN_MEMORY_PATH)
    print("Direct scan memory rows:", len(df_mem))
    print("Direct scan rows in TBMT:", int(df_tbmt["Số KHLCNT scan từ PDF"].fillna("").astype(str).str.strip().ne("").sum()))

    return df_tbmt

def build_match_candidates(df_kh, df_tbmt, min_score=78):
    if df_kh.empty or df_tbmt.empty:
        return pd.DataFrame()

    candidates = []

    for _, kh in df_kh.iterrows():
        kh_unit = kh.get("Match key đơn vị", "")
        kh_price = str(kh.get("Match key giá", ""))
        kh_name = kh.get("Match key tên gói", "")

        for _, tb in df_tbmt.iterrows():
            tb_unit = tb.get("Match key đơn vị", "")
            tb_price = str(tb.get("Match key giá", ""))
            tb_name = tb.get("Match key tên gói", "")

            unit_score = fuzz.token_set_ratio(kh_unit, tb_unit) if kh_unit and tb_unit else 0
            name_score = fuzz.token_set_ratio(kh_name, tb_name) if kh_name and tb_name else 0
            price_score = 100 if kh_price and tb_price and kh_price == tb_price else 0

            total_score = round(unit_score * 0.35 + name_score * 0.45 + price_score * 0.20, 2)

            if total_score >= min_score or (price_score == 100 and unit_score >= 70):
                candidates.append({
                    "Số KHLCNT": kh.get("Số KHLCNT", ""),
                    "Số TBMT": tb.get("Số TBMT", ""),
                    "Chủ đầu tư KHLCNT": kh.get("Chủ đầu tư-KHLCNT", ""),
                    "Đơn vị mời thầu TBMT": tb.get("Đơn vị mời thầu", ""),
                    "Tên gói KHLCNT": kh.get("Tên gói thầu trong KHMT", ""),
                    "Tên gói TBMT": tb.get("Tên gói thầu", ""),
                    "Giá KHLCNT": kh.get("Giá gói thầu trong KHMT", ""),
                    "Giá TBMT": tb.get("Giá gói thầu", ""),
                    "Ngày phê duyệt KHLCNT": kh.get("Ngày phê duyệt", ""),
                    "Ngày đăng TBMT": tb.get("Ngày đăng", ""),
                    "Unit score": unit_score,
                    "Name score": name_score,
                    "Price score": price_score,
                    "Total score": total_score,
                    "Gợi ý": "Có thể match" if total_score >= 80 else "Cần kiểm tra lại",
                })

    df = pd.DataFrame(candidates)
    if not df.empty:
        df = df.sort_values(["Total score", "Price score", "Name score"], ascending=False)

    return df


# ============================================================
# RUN PARSING - ONLY NEW PDFs
# ============================================================

registry_df = load_processed_registry()

all_tbmt_pdfs = sorted(TBMT_DIR.glob("*.pdf"))
all_khlcnt_pdfs = sorted(KHLCNT_DIR.glob("*.pdf"))

tbmt_pdf_files, skipped_tbmt_pdf_files = filter_new_pdfs(all_tbmt_pdfs, "TBMT", registry_df)
khlcnt_pdf_files, skipped_khlcnt_pdf_files = filter_new_pdfs(all_khlcnt_pdfs, "KHLCNT", registry_df)

print("All TBMT PDFs:", len(all_tbmt_pdfs))
print("New TBMT PDFs:", len(tbmt_pdf_files))
print("Skipped TBMT PDFs:", len(skipped_tbmt_pdf_files))

print("All KHLCNT PDFs:", len(all_khlcnt_pdfs))
print("New KHLCNT PDFs:", len(khlcnt_pdf_files))
print("Skipped KHLCNT PDFs:", len(skipped_khlcnt_pdf_files))

all_tbmt_rows = []
all_khlcnt_rows = []
all_khlcnt_item_rows = []
debug_rows = []

for pdf_path in tbmt_pdf_files:
    filename = pdf_path.name
    text = extract_text_from_pdf(pdf_path)
    records = split_tbmt_records(text)

    debug_rows.append({"File": filename, "Type": "TBMT", "Text length": len(text), "Records found": len(records)})

    for stt, block in records:
        all_tbmt_rows.append(parse_tbmt_record(stt, block, filename))

    save_checkpoint_workbook(PARSE_CHECKPOINT_PATH, {
        "TBMT_checkpoint": pd.DataFrame(all_tbmt_rows),
        "KHLCNT_checkpoint": pd.DataFrame(all_khlcnt_rows),
        "KHLCNT_Items_checkpoint": pd.DataFrame(all_khlcnt_item_rows),
        "Debug_checkpoint": pd.DataFrame(debug_rows),
    })

for pdf_path in khlcnt_pdf_files:
    filename = pdf_path.name
    text = extract_text_from_pdf(pdf_path)
    records = split_khlcnt_records(text)

    debug_rows.append({"File": filename, "Type": "KHLCNT", "Text length": len(text), "Records found": len(records)})

    for stt, block in records:
        all_khlcnt_rows.append(parse_khlcnt_record(stt, block, filename))
        all_khlcnt_item_rows.extend(parse_khlcnt_items(stt, block, filename))

    save_checkpoint_workbook(PARSE_CHECKPOINT_PATH, {
        "TBMT_checkpoint": pd.DataFrame(all_tbmt_rows),
        "KHLCNT_checkpoint": pd.DataFrame(all_khlcnt_rows),
        "KHLCNT_Items_checkpoint": pd.DataFrame(all_khlcnt_item_rows),
        "Debug_checkpoint": pd.DataFrame(debug_rows),
    })

df_tbmt_new = pd.DataFrame(all_tbmt_rows, columns=ZOHO_TBMT_COLUMNS)
df_khlcnt_new = pd.DataFrame(all_khlcnt_rows, columns=ZOHO_KHLCNT_COLUMNS)
df_khlcnt_items_new = pd.DataFrame(all_khlcnt_item_rows, columns=ZOHO_KHLCNT_ITEM_COLUMNS)
df_debug_new = pd.DataFrame(debug_rows)


# ============================================================
# LOAD EXISTING OUTPUT + APPEND NEW ROWS
# ============================================================

df_tbmt_old = read_existing_sheet(MASTER_OUTPUT_PATH, "TBMT_Zoho_Import", ZOHO_TBMT_COLUMNS)
df_khlcnt_old = read_existing_sheet(MASTER_OUTPUT_PATH, "KHLCNT_Zoho_Import", ZOHO_KHLCNT_COLUMNS)
df_khlcnt_items_old = read_existing_sheet(MASTER_OUTPUT_PATH, "KHLCNT_Items_Match", ZOHO_KHLCNT_ITEM_COLUMNS)
df_debug_old = read_existing_sheet(MASTER_OUTPUT_PATH, "Debug")

df_tbmt = pd.concat([df_tbmt_old, df_tbmt_new], ignore_index=True, sort=False)
df_khlcnt = pd.concat([df_khlcnt_old, df_khlcnt_new], ignore_index=True, sort=False)
df_khlcnt_items = pd.concat([df_khlcnt_items_old, df_khlcnt_items_new], ignore_index=True, sort=False)
df_debug = pd.concat([df_debug_old, df_debug_new], ignore_index=True, sort=False)
# Fix mã/version sau khi đọc output cũ + dữ liệu mới
df_tbmt = fix_code_text_columns(df_tbmt)
df_khlcnt = fix_code_text_columns(df_khlcnt)
df_khlcnt_items = fix_code_text_columns(df_khlcnt_items)
for col in ZOHO_TBMT_COLUMNS:
    if col not in df_tbmt.columns:
        df_tbmt[col] = ""

for col in ZOHO_KHLCNT_COLUMNS:
    if col not in df_khlcnt.columns:
        df_khlcnt[col] = ""

for col in ZOHO_KHLCNT_ITEM_COLUMNS:
    if col not in df_khlcnt_items.columns:
        df_khlcnt_items[col] = ""

df_tbmt = df_tbmt[ZOHO_TBMT_COLUMNS + [c for c in df_tbmt.columns if c not in ZOHO_TBMT_COLUMNS]]
df_khlcnt = df_khlcnt[ZOHO_KHLCNT_COLUMNS + [c for c in df_khlcnt.columns if c not in ZOHO_KHLCNT_COLUMNS]]
df_khlcnt_items = df_khlcnt_items[ZOHO_KHLCNT_ITEM_COLUMNS + [c for c in df_khlcnt_items.columns if c not in ZOHO_KHLCNT_ITEM_COLUMNS]]

# Version logic after combine
df_tbmt = mark_latest_versions(df_tbmt, "Mã gốc TBMT", "Mã Phiên Bản", "Số TBMT", "Phiên Bản Mới Nhất", "Phiên Bản Trước dự kiến")
df_khlcnt = mark_latest_versions(df_khlcnt, "Mã gốc KHLCNT", "Phiên bản KHLCNT", "Số KHLCNT", "KHLCNT mới nhất", "Phiên Bản Trước_KHLC dự kiến")
df_khlcnt_items = mark_khlcnt_items_latest(df_khlcnt_items)

# Final dedupe requested by Số TBMT / Số KHLCNT
# TBMT: ưu tiên giữ dòng đã có Số KHLCNT / đã match, tránh file mới trùng làm mất match cũ

if not df_tbmt.empty and "Số TBMT" in df_tbmt.columns:
    df_tbmt["_dedupe_key"] = clean_key_series(df_tbmt["Số TBMT"])

    df_tbmt["_has_khlcnt_for_dedupe"] = (
        df_tbmt["Số KHLCNT"]
        .astype(str)
        .str.strip()
        .replace({"nan": "", "None": "", "NaN": ""})
        .ne("")
    )

    df_tbmt["_has_match_status_for_dedupe"] = (
        df_tbmt["Trạng thái match KHLCNT"]
        .astype(str)
        .str.strip()
        .replace({"nan": "", "None": "", "NaN": ""})
        .ne("")
    )

    df_tbmt = (
        df_tbmt[df_tbmt["_dedupe_key"] != ""]
        .sort_values(
            ["_dedupe_key", "_has_khlcnt_for_dedupe", "_has_match_status_for_dedupe"],
            ascending=[True, False, False]
        )
        .drop_duplicates("_dedupe_key", keep="first")
        .drop(columns=["_dedupe_key", "_has_khlcnt_for_dedupe", "_has_match_status_for_dedupe"])
        .reset_index(drop=True)
    )

# KHLCNT parent: 1 Số KHLCNT = 1 dòng để import Zoho.
df_khlcnt = dedupe_by_key(df_khlcnt, "Số KHLCNT", keep="last")

# KHLCNT items: 1 gói thầu con = 1 dòng để match TBMT.
# Không dedupe theo Số KHLCNT, vì một KHLCNT có thể có nhiều gói.
if not df_khlcnt_items.empty and "KHLCNT Item Key" in df_khlcnt_items.columns:
    df_khlcnt_items = dedupe_by_key(df_khlcnt_items, "KHLCNT Item Key", keep="last")

# Fix lại sau version logic + dedupe để giữ 00/01/02
df_tbmt = fix_code_text_columns(df_tbmt)
df_khlcnt = fix_code_text_columns(df_khlcnt)
df_khlcnt_items = fix_code_text_columns(df_khlcnt_items)
# Accounts from combined deduped data
account_names = set()

if not df_tbmt.empty:
    account_names.update(x for x in df_tbmt["Chủ đầu tư"].dropna().astype(str) if x.strip())
    account_names.update(x for x in df_tbmt["Đơn vị mời thầu"].dropna().astype(str) if x.strip())

if not df_khlcnt.empty:
    account_names.update(x for x in df_khlcnt["Chủ đầu tư-KHLCNT"].dropna().astype(str) if x.strip())

df_accounts = pd.DataFrame({
    "Account Name": sorted(account_names),
    "Nguồn tạo": "PDF TBMT/KHLCNT",
    "Ghi chú": "Import/upsert trước để lookup Chủ đầu tư/Đơn vị mời thầu không lỗi"
})
df_accounts = dedupe_by_key(df_accounts, "Account Name", keep="last")
# Xử lý direct KHLCNT scan từ TBMT PDF.
# Nếu mã PL đã có trong KHLCNT tổng thì update Số KHLCNT luôn.
# Nếu chưa có thì lưu direct_scan_memory và KHÔNG cần đưa đi fuzzy match.
df_tbmt = update_direct_scan_memory_and_apply(df_tbmt, df_khlcnt)
df_match = pd.DataFrame()  # Match chính thức chạy ở Cell 3 bằng KHLCNT_Items_Match

print("After combine + dedupe:")
print("Accounts:", len(df_accounts))
print("TBMT:", len(df_tbmt))
print("KHLCNT parent:", len(df_khlcnt))
print("KHLCNT items for match:", len(df_khlcnt_items))
print("Match candidates preview:", len(df_match))


# ============================================================
# EXPORT FIXED OUTPUT FILES - TEXT SAFE
# ============================================================

df_accounts = fix_code_text_columns(df_accounts)
df_tbmt = fix_code_text_columns(df_tbmt)
df_khlcnt = fix_code_text_columns(df_khlcnt)
df_khlcnt_items = fix_code_text_columns(df_khlcnt_items)
df_match = fix_code_text_columns(df_match)
df_debug = fix_code_text_columns(df_debug)

TMP_MASTER_OUTPUT_PATH = MASTER_OUTPUT_PATH.with_name(MASTER_OUTPUT_PATH.stem + ".tmp.xlsx")
with pd.ExcelWriter(TMP_MASTER_OUTPUT_PATH, engine="openpyxl") as writer:
    write_excel_sheet_text_safe(writer, df_accounts, "Accounts_Import", TEXT_SAFE_COLUMNS)
    write_excel_sheet_text_safe(writer, df_tbmt, "TBMT_Zoho_Import", TEXT_SAFE_COLUMNS)
    write_excel_sheet_text_safe(writer, df_khlcnt, "KHLCNT_Zoho_Import", TEXT_SAFE_COLUMNS)
    write_excel_sheet_text_safe(writer, df_khlcnt_items, "KHLCNT_Items_Match", TEXT_SAFE_COLUMNS)
    write_excel_sheet_text_safe(writer, df_match, "Match_Candidates", TEXT_SAFE_COLUMNS)
    write_excel_sheet_text_safe(writer, df_debug, "Debug", TEXT_SAFE_COLUMNS)

    huong_dan = pd.DataFrame({
        "Bước": ["1", "2", "3", "4", "5", "6"],
        "Nội dung": [
            "Import/upsert Accounts_for_import.xlsx trước.",
            "Import/upsert KHLCNT_for_import.xlsx sau Accounts.",
            "Chạy Cell 3 để tạo TBMT_matched_for_import.xlsx nếu cần match Số KHLCNT.",
            "Import/upsert TBMT_matched_for_import.xlsx hoặc TBMT_for_import.xlsx sau KHLCNT.",
            "Output cố định, mỗi lần chạy sẽ update file cũ, không sinh nhiều file timestamp.",
            "PDF đã xử lý sẽ được ghi vào config/processed_pdf_registry.xlsx để lần sau không đọc trùng.",
        ],
    })

    write_excel_sheet_text_safe(writer, huong_dan, "Huong_dan", [])

safe_replace_excel(TMP_MASTER_OUTPUT_PATH, MASTER_OUTPUT_PATH)

# Tách riêng từng file để Zoho import không đọc nhầm sheet đầu
with pd.ExcelWriter(ACCOUNTS_IMPORT_PATH, engine="openpyxl") as writer:
    write_excel_sheet_text_safe(writer, df_accounts, "Accounts_Import", TEXT_SAFE_COLUMNS)

with pd.ExcelWriter(KHLCNT_IMPORT_PATH, engine="openpyxl") as writer:
    write_excel_sheet_text_safe(writer, df_khlcnt, "KHLCNT_Zoho_Import", TEXT_SAFE_COLUMNS)

# File phụ chỉ dùng cho match/debug, không cần import Zoho nếu đang dùng 2 module.
with pd.ExcelWriter(KHLCNT_ITEMS_MATCH_PATH, engine="openpyxl") as writer:
    write_excel_sheet_text_safe(writer, df_khlcnt_items, "KHLCNT_Items_Match", TEXT_SAFE_COLUMNS)

with pd.ExcelWriter(TBMT_IMPORT_PATH, engine="openpyxl") as writer:
    write_excel_sheet_text_safe(writer, df_tbmt, "TBMT_Zoho_Import", TEXT_SAFE_COLUMNS)

# Chỉ mark processed sau khi export thành công
processed_infos = []
for p in tbmt_pdf_files:
    processed_infos.append(get_pdf_info(p, "TBMT"))
for p in khlcnt_pdf_files:
    processed_infos.append(get_pdf_info(p, "KHLCNT"))
append_processed_registry(processed_infos)

print("\nDONE CELL 2")
print("Master output:", MASTER_OUTPUT_PATH)
print("Accounts import:", ACCOUNTS_IMPORT_PATH)
print("KHLCNT import:", KHLCNT_IMPORT_PATH)
print("KHLCNT items for match:", KHLCNT_ITEMS_MATCH_PATH)
print("TBMT import:", TBMT_IMPORT_PATH)


# ============================================================
# CELL 3: MATCH TBMT -> KHLCNT WITHOUT MANUAL MAPPING
# Source:
# - Cell 2 fixed output Excel: output_excel/Zoho_PDF_Import_Output_LATEST.xlsx
# - Optional Zoho export Excel: input_match/
# Output:
# - output_excel/TBMT_matched_for_import.xlsx
# - logs/Match_Candidates.xlsx
# ============================================================

import re
import pandas as pd
from pathlib import Path
from datetime import datetime

try:
    from rapidfuzz import fuzz
except Exception:
    !pip -q install rapidfuzz
    from rapidfuzz import fuzz

try:
    from unidecode import unidecode
except Exception:
    !pip -q install unidecode
    from unidecode import unidecode


# ============================================================
# 0. PATHS
# ============================================================

CELL2_OUTPUT_PATH = OUTPUT_DIR / "Zoho_PDF_Import_Output_LATEST.xlsx"

ZOHO_TBMT_EXPORT_PATH = None
ZOHO_KHLCNT_EXPORT_PATH = None

TBMT_SHEET = "TBMT_Zoho_Import"
KHLCNT_SHEET = "KHLCNT_Zoho_Import"
KHLCNT_ITEM_SHEET = "KHLCNT_Items_Match"

AUTO_MATCH_SCORE = 80
REVIEW_MATCH_SCORE = 72
OVERWRITE_EXISTING_KHLCNT = False

MATCHED_TBMT_OUTPUT_PATH = OUTPUT_DIR / "TBMT_matched_for_import.xlsx"
MATCH_LOG_PATH = LOG_DIR / "Match_Candidates.xlsx"


def latest_excel_by_keyword(folder: Path, keywords, exclude_keywords=None):
    exclude_keywords = exclude_keywords or []
    files = []

    for p in folder.glob("*.xlsx"):
        name = p.name.lower()

        if p.name.startswith("~$"):
            continue

        if any(k.lower() in name for k in exclude_keywords):
            continue

        if any(k.lower() in name for k in keywords):
            files.append(p)

    files = sorted(files, key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None


if not CELL2_OUTPUT_PATH.exists():
    raise FileNotFoundError(f"Không thấy output Cell 2: {CELL2_OUTPUT_PATH}")

if ZOHO_TBMT_EXPORT_PATH:
    zoho_tbmt_file = Path(ZOHO_TBMT_EXPORT_PATH)
else:
    zoho_tbmt_file = latest_excel_by_keyword(
        INPUT_MATCH_DIR,
        keywords=["tbmt", "bao_dau_thau", "bao dau thau"],
        exclude_keywords=["matched", "suggest", "candidate", "manual"]
    )

if ZOHO_KHLCNT_EXPORT_PATH:
    zoho_khlcnt_file = Path(ZOHO_KHLCNT_EXPORT_PATH)
else:
    zoho_khlcnt_file = latest_excel_by_keyword(
        INPUT_MATCH_DIR,
        keywords=["khlcnt", "ke_hoach", "ke hoach", "khmt"],
        exclude_keywords=["matched", "suggest", "candidate", "manual"]
    )

print("Cell 2 output:", CELL2_OUTPUT_PATH)

if zoho_tbmt_file and zoho_tbmt_file.exists():
    print("Zoho TBMT export:", zoho_tbmt_file)
else:
    zoho_tbmt_file = None
    print("Không có Zoho TBMT export trong input_match/.")

if zoho_khlcnt_file and zoho_khlcnt_file.exists():
    print("Zoho KHLCNT export:", zoho_khlcnt_file)
else:
    zoho_khlcnt_file = None
    print("Không có Zoho KHLCNT export trong input_match/.")


def read_sheet_or_first(path, preferred_sheet):
    try:
        return pd.read_excel(path, sheet_name=preferred_sheet)
    except Exception:
        return pd.read_excel(path, sheet_name=0)


df_tbmt_cell2 = read_sheet_or_first(CELL2_OUTPUT_PATH, TBMT_SHEET)

# Ưu tiên sheet KHLCNT_Items_Match để match đúng gói thầu con.
# Nếu master cũ chưa có sheet này, fallback về KHLCNT_Zoho_Import.
try:
    df_khlcnt_cell2 = pd.read_excel(CELL2_OUTPUT_PATH, sheet_name=KHLCNT_ITEM_SHEET)
    print("Using KHLCNT items for matching:", KHLCNT_ITEM_SHEET)
except Exception:
    df_khlcnt_cell2 = read_sheet_or_first(CELL2_OUTPUT_PATH, KHLCNT_SHEET)
    print("Không thấy KHLCNT_Items_Match, fallback dùng KHLCNT_Zoho_Import để match.")

df_tbmt_cell2["_Nguồn dữ liệu"] = "Cell2_PDF_Output"
df_khlcnt_cell2["_Nguồn dữ liệu"] = "Cell2_KHLCNT_Items_Match"

tbmt_frames = [df_tbmt_cell2]
khlcnt_frames = [df_khlcnt_cell2]

if zoho_tbmt_file:
    df_tbmt_zoho = read_sheet_or_first(zoho_tbmt_file, TBMT_SHEET)
    df_tbmt_zoho["_Nguồn dữ liệu"] = "Zoho_TBMT_Export"
    tbmt_frames.append(df_tbmt_zoho)

if zoho_khlcnt_file:
    df_khlcnt_zoho = read_sheet_or_first(zoho_khlcnt_file, KHLCNT_SHEET)
    df_khlcnt_zoho["_Nguồn dữ liệu"] = "Zoho_KHLCNT_Export"
    khlcnt_frames.append(df_khlcnt_zoho)

df_tbmt = pd.concat(tbmt_frames, ignore_index=True, sort=False)
df_khlcnt = pd.concat(khlcnt_frames, ignore_index=True, sort=False)
# Fix mã/version sau khi đọc Excel vào Cell 3
df_tbmt = fix_code_text_columns(df_tbmt)
df_khlcnt = fix_code_text_columns(df_khlcnt)
print("TBMT total before dedupe:", len(df_tbmt))
print("KHLCNT total before cleanup:", len(df_khlcnt))


def find_col_soft(df, names, required=False):
    lower = {str(c).strip().lower(): c for c in df.columns}

    for n in names:
        key = n.strip().lower()
        if key in lower:
            return lower[key]

    if required:
        raise KeyError(f"Không tìm thấy cột nào trong danh sách: {names}")

    return None


C_TBMT_SO = find_col_soft(df_tbmt, ["Số TBMT", "So TBMT", "Name"], required=True)
C_TBMT_KHLCNT = find_col_soft(df_tbmt, ["Số KHLCNT", "So KHLCNT", "Số KHLCNT.Name", "So KHLCNT.Name"], required=False)
C_TBMT_GOI = find_col_soft(df_tbmt, ["Tên gói thầu", "Ten goi thau"], required=True)
C_TBMT_DU_AN = find_col_soft(df_tbmt, ["Dự án", "Du an"], required=False)
C_TBMT_CDT = find_col_soft(df_tbmt, ["Chủ đầu tư", "Chu dau tu"], required=False)
C_TBMT_DVMT = find_col_soft(df_tbmt, ["Đơn vị mời thầu", "Don vi moi thau"], required=False)
C_TBMT_GIA = find_col_soft(df_tbmt, ["Giá gói thầu", "Gia goi thau", "Tổng giá trị", "Tong gia tri"], required=False)

C_KH_SO = find_col_soft(df_khlcnt, ["Số KHLCNT", "So KHLCNT", "Name"], required=True)
C_KH_GOI = find_col_soft(df_khlcnt, ["Tên gói thầu trong KHMT", "Ten goi thau trong KHMT", "Tên gói thầu", "Ten goi thau"], required=False)
C_KH_DU_AN = find_col_soft(df_khlcnt, ["Tên dự án", "Ten du an", "Dự án", "Du an"], required=False)
C_KH_CDT = find_col_soft(df_khlcnt, ["Chủ đầu tư-KHLCNT", "Chu dau tu-KHLCNT", "Chủ đầu tư", "Chu dau tu"], required=False)
C_KH_GIA = find_col_soft(df_khlcnt, ["Giá gói thầu trong KHMT", "Gia goi thau trong KHMT", "Giá gói thầu", "Gia goi thau", "Tổng mức đầu tư", "Tong muc dau tu"], required=False)
C_TBMT_VERSION = find_col_soft(df_tbmt, ["Mã Phiên Bản", "Ma Phien Ban", "Phiên bản TBMT", "Phien ban TBMT"], required=False)
C_KH_VERSION = find_col_soft(df_khlcnt, ["Phiên bản KHLCNT", "Phien ban KHLCNT"], required=False)
C_KH_LATEST = find_col_soft(df_khlcnt, ["KHLCNT mới nhất", "KHLCNT moi nhat"], required=False)
C_KH_ITEM_KEY = find_col_soft(df_khlcnt, ["KHLCNT Item Key", "KHL CNT Item Key"], required=False)
C_KH_ITEM_ORDER = find_col_soft(df_khlcnt, ["Match item order", "STT nguồn", "STT nguon"], required=False)

if C_TBMT_KHLCNT is None:
    C_TBMT_KHLCNT = "Số KHLCNT"
    df_tbmt[C_TBMT_KHLCNT] = ""

for col in ["Trạng thái match KHLCNT", "Điểm match KHLCNT", "Nguồn match KHLCNT", "Ghi chú match KHLCNT", "Số KHLCNT ứng viên", "Top 3 KHLCNT candidates", "KHLCNT Item Key match", "Tên gói KHLCNT match", "Giá KHLCNT match", "Phiên bản KHLCNT match", "Version score"]:
    if col not in df_tbmt.columns:
        df_tbmt[col] = ""

print("\nDetected columns:")
print("TBMT Số:", C_TBMT_SO)
print("TBMT Số KHLCNT:", C_TBMT_KHLCNT)
print("TBMT Tên gói:", C_TBMT_GOI)
print("TBMT Dự án:", C_TBMT_DU_AN)
print("TBMT Chủ đầu tư:", C_TBMT_CDT)
print("TBMT Đơn vị mời thầu:", C_TBMT_DVMT)
print("TBMT Giá:", C_TBMT_GIA)
print("KHLCNT Số:", C_KH_SO)
print("KHLCNT Tên gói:", C_KH_GOI)
print("KHLCNT Dự án:", C_KH_DU_AN)
print("KHLCNT Chủ đầu tư:", C_KH_CDT)
print("KHLCNT Giá:", C_KH_GIA)
print("TBMT version:", C_TBMT_VERSION)
print("KHLCNT version:", C_KH_VERSION)
print("KHLCNT latest flag:", C_KH_LATEST)
print("KHLCNT item key:", C_KH_ITEM_KEY)

# Reuse helpers: is_blank, clean_text, clean_number-like function for matching

def is_blank(v):
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except Exception:
        pass
    return str(v).strip() == ""


def clean_value(v):
    if is_blank(v):
        return ""
    return str(v).strip()


def get_value(row, col):
    if col is None:
        return ""
    return row.get(col, "")


def clean_number(v):
    if is_blank(v):
        return None

    if isinstance(v, (int, float)) and not isinstance(v, bool):
        try:
            if pd.isna(v):
                return None
        except Exception:
            pass
        return float(v)

    s = str(v).strip()
    s = re.sub(r"[^\d,.\-]", "", s)

    if s in ["", "-", ".", ","]:
        return None

    try:
        if "." in s and "," in s:
            if s.rfind(",") > s.rfind("."):
                s = s.replace(".", "").replace(",", ".")
            else:
                s = s.replace(",", "")
        elif "," in s:
            parts = s.split(",")
            if len(parts) == 2 and len(parts[1]) in [1, 2]:
                s = parts[0].replace(".", "") + "." + parts[1]
            else:
                s = s.replace(",", "")
        elif "." in s:
            parts = s.split(".")
            if len(parts) > 2:
                s = "".join(parts)
            elif len(parts) == 2 and len(parts[1]) == 3:
                s = s.replace(".", "")

        return float(s)
    except Exception:
        return None


def norm_text(v):
    s = clean_value(v).lower()
    s = unidecode(s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    noise_words = ["goi thau", "du an", "mua sam", "cung cap", "hang hoa", "thiet bi", "vat tu", "dich vu", "nam 2024", "nam 2025", "nam 2026"]
    for w in noise_words:
        s = s.replace(w, " ")

    s = re.sub(r"\s+", " ", s).strip()
    return s


def price_score(a, b):
    a_num = clean_number(a)
    b_num = clean_number(b)

    if a_num is None or b_num is None or a_num <= 0 or b_num <= 0:
        return None

    diff_ratio = abs(a_num - b_num) / max(a_num, b_num)

    if diff_ratio <= 0.01:
        return 100
    if diff_ratio <= 0.03:
        return 95
    if diff_ratio <= 0.05:
        return 88
    if diff_ratio <= 0.10:
        return 75
    if diff_ratio <= 0.20:
        return 55

    return 0


def to_version_num(v):
    if is_blank(v):
        return None
    s = str(v).strip()
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    m = re.search(r"\d+", s)
    if not m:
        return None
    try:
        return int(m.group(0))
    except Exception:
        return None


def bool_value(v):
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    return s in ["true", "1", "yes", "y", "đúng", "dung"]


def calc_version_score(tbmt_row, kh_row):
    tv = to_version_num(get_value(tbmt_row, C_TBMT_VERSION))
    kv = to_version_num(get_value(kh_row, C_KH_VERSION))

    if tv is None or kv is None:
        return ""

    diff = abs(tv - kv)
    if diff == 0:
        return 100
    if diff == 1:
        return 85
    if diff == 2:
        return 70
    if diff == 3:
        return 55
    return 40


def calc_order_score(tbmt_row, kh_row):
    """
    Tín hiệu phụ: nếu STT nguồn / thứ tự item gần nhau thì cộng nhẹ.
    Không auto match chỉ vì cùng thứ tự.
    """
    try:
        tb_order = int(float(str(get_value(tbmt_row, "STT nguồn")).strip()))
        kh_order = int(float(str(get_value(kh_row, C_KH_ITEM_ORDER)).strip()))
    except Exception:
        return ""

    diff = abs(tb_order - kh_order)
    if diff == 0:
        return 100
    if diff == 1:
        return 90
    if diff == 2:
        return 80
    if diff <= 5:
        return 65
    return 40


def calc_score(tbmt_row, kh_row):
    tbmt_goi = norm_text(get_value(tbmt_row, C_TBMT_GOI))
    kh_goi = norm_text(get_value(kh_row, C_KH_GOI))

    tbmt_du_an = norm_text(get_value(tbmt_row, C_TBMT_DU_AN))
    kh_du_an = norm_text(get_value(kh_row, C_KH_DU_AN))

    tbmt_cdt = norm_text(get_value(tbmt_row, C_TBMT_CDT))
    tbmt_dvmt = norm_text(get_value(tbmt_row, C_TBMT_DVMT))
    kh_cdt = norm_text(get_value(kh_row, C_KH_CDT))

    goi_score = fuzz.token_set_ratio(tbmt_goi, kh_goi) if tbmt_goi and kh_goi else 0
    du_an_score = fuzz.token_set_ratio(tbmt_du_an, kh_du_an) if tbmt_du_an and kh_du_an else 0

    cdt_score_1 = fuzz.token_set_ratio(tbmt_cdt, kh_cdt) if tbmt_cdt and kh_cdt else 0
    cdt_score_2 = fuzz.token_set_ratio(tbmt_dvmt, kh_cdt) if tbmt_dvmt and kh_cdt else 0
    cdt_score = max(cdt_score_1, cdt_score_2)

    ps = price_score(get_value(tbmt_row, C_TBMT_GIA), get_value(kh_row, C_KH_GIA))
    vs = calc_version_score(tbmt_row, kh_row)
    os = calc_order_score(tbmt_row, kh_row)

    # Base score: ưu tiên tên gói + dự án + chủ đầu tư + giá.
    if ps is None:
        base_total = 0.50 * goi_score + 0.30 * du_an_score + 0.20 * cdt_score
    else:
        base_total = 0.45 * goi_score + 0.25 * du_an_score + 0.20 * cdt_score + 0.10 * ps

    # Version score giúp TBMT -00 ưu tiên KHLCNT -00, TBMT -01 ưu tiên KHLCNT -01...
    # Order score chỉ là tín hiệu phụ, không đủ để tự động match nếu các điểm chính yếu thấp.
    total = base_total
    if vs != "":
        total = 0.90 * total + 0.10 * vs
    if os != "":
        total = 0.97 * total + 0.03 * os

    return {
        "total": round(total, 2),
        "base_total": round(base_total, 2),
        "goi": round(goi_score, 2),
        "du_an": round(du_an_score, 2),
        "cdt": round(cdt_score, 2),
        "gia": "" if ps is None else round(ps, 2),
        "version": "" if vs == "" else round(vs, 2),
        "order": "" if os == "" else round(os, 2),
    }


def match_one(tbmt_row, kh_candidates):
    scored = []

    for _, kh_row in kh_candidates.iterrows():
        so_kh = clean_value(get_value(kh_row, C_KH_SO))
        if not so_kh:
            continue

        sc = calc_score(tbmt_row, kh_row)
        kh_version_num = to_version_num(get_value(kh_row, C_KH_VERSION))
        is_latest = bool_value(get_value(kh_row, C_KH_LATEST)) if C_KH_LATEST else False

        candidate = {
            "Số KHLCNT": so_kh,
            "KHLCNT Item Key": clean_value(get_value(kh_row, C_KH_ITEM_KEY)),
            "Điểm": sc["total"],
            "Base": sc["base_total"],
            "Gói": sc["goi"],
            "Dự án": sc["du_an"],
            "Chủ đầu tư": sc["cdt"],
            "Giá": sc["gia"],
            "Version": sc["version"],
            "Order": sc["order"],
            "Phiên bản KHLCNT": clean_value(get_value(kh_row, C_KH_VERSION)),
            "KHLCNT mới nhất": is_latest,
            "Tên gói KHLCNT": clean_value(get_value(kh_row, C_KH_GOI)),
            "Giá KHLCNT": clean_value(get_value(kh_row, C_KH_GIA)),
            "Dự án KHLCNT": clean_value(get_value(kh_row, C_KH_DU_AN)),
            "Chủ đầu tư KHLCNT": clean_value(get_value(kh_row, C_KH_CDT)),
            "Nguồn KHLCNT": clean_value(get_value(kh_row, "_Nguồn dữ liệu")),
        }

        # Sort key: total trước, rồi version gần hơn, rồi điểm gói, giá, dự án, rồi version KHLCNT cao hơn.
        sort_key = (
            sc["total"],
            -999 if sc["version"] == "" else sc["version"],
            sc["goi"],
            -999 if sc["gia"] == "" else sc["gia"],
            sc["du_an"],
            kh_version_num if kh_version_num is not None else -1,
            1 if is_latest else 0,
        )
        scored.append({"kh_row": kh_row, "score": sc, "candidate": candidate, "sort_key": sort_key})

    if not scored:
        return None, []

    scored = sorted(scored, key=lambda x: x["sort_key"], reverse=True)
    best = scored[0]
    top = [x["candidate"] for x in scored[:3]]
    return best, top


def is_safe_auto(sc, top3=None):
    price_ok = (sc["gia"] == "" or sc["gia"] >= 75)
    version_ok = (sc.get("version", "") == "" or sc.get("version", 0) >= 70)

    # Nếu top 1 và top 2 quá sát nhưng khác Số KHLCNT, để review để tránh gán nhầm.
    unique_best_ok = True
    if top3 and len(top3) >= 2:
        try:
            gap = float(top3[0].get("Điểm", 0)) - float(top3[1].get("Điểm", 0))
            if top3[0].get("Số KHLCNT") != top3[1].get("Số KHLCNT") and gap < 1.5:
                unique_best_ok = False
        except Exception:
            pass

    return (
        sc["total"] >= AUTO_MATCH_SCORE
        and sc["goi"] >= 90
        and sc["du_an"] >= 85
        and sc["cdt"] >= 70
        and price_ok
        and version_ok
        and unique_best_ok
    )


# ============================================================
# CLEAN / DEDUPE DATA
# ============================================================

df_tbmt["_tbmt_key"] = df_tbmt[C_TBMT_SO].astype(str).str.strip()
df_tbmt["_has_khlcnt"] = df_tbmt[C_TBMT_KHLCNT].astype(str).str.strip().ne("") & df_tbmt[C_TBMT_KHLCNT].astype(str).str.lower().ne("nan")

df_tbmt["_source_priority"] = df_tbmt["_Nguồn dữ liệu"].map({"Cell2_PDF_Output": 0, "Zoho_TBMT_Export": 1}).fillna(9)

df_tbmt = (
    df_tbmt
    .sort_values(["_tbmt_key", "_has_khlcnt", "_source_priority"], ascending=[True, False, True])
    .drop_duplicates("_tbmt_key", keep="first")
    .reset_index(drop=True)
)

df_khlcnt["_khlcnt_key"] = df_khlcnt[C_KH_SO].astype(str).str.strip()
df_khlcnt = df_khlcnt[df_khlcnt["_khlcnt_key"] != ""].copy()

kh_candidates = df_khlcnt.copy()

print("\nAfter cleanup:")
print("TBMT rows:", len(df_tbmt))
print("KHLCNT candidate rows:", len(kh_candidates))
print("TBMT đã có Số KHLCNT trước match:", int(df_tbmt["_has_khlcnt"].sum()))


# ============================================================
# RUN MATCH
# ============================================================

match_logs = []
kept_count = 0
auto_count = 0
review_count = 0
no_match_count = 0

for idx, tbmt_row in df_tbmt.iterrows():
    so_tbmt = clean_value(get_value(tbmt_row, C_TBMT_SO))
    existing_kh = clean_value(get_value(tbmt_row, C_TBMT_KHLCNT))

    direct_kh_scan = ""
    if "Số KHLCNT scan từ PDF" in df_tbmt.columns:
        direct_kh_scan = clean_value(tbmt_row.get("Số KHLCNT scan từ PDF", ""))

    # Nếu TBMT scan được Số KHLCNT từ PDF thì KHÔNG fuzzy nữa.
    # Dù KHLCNT tổng chưa có, Cell 2 direct memory hoặc Cell 4 memory sẽ xử lý sau.
    if direct_kh_scan:
        kept_count += 1

        df_tbmt.at[idx, "Trạng thái match KHLCNT"] = "Direct scan - bỏ qua fuzzy match"
        df_tbmt.at[idx, "Số KHLCNT ứng viên"] = direct_kh_scan
        df_tbmt.at[idx, "Nguồn match KHLCNT"] = "Scan trực tiếp từ TBMT PDF"
        df_tbmt.at[idx, "Ghi chú match KHLCNT"] = (
            f"TBMT scan được Số KHLCNT từ PDF: {direct_kh_scan}. "
            "Không chạy so sánh tên gói/dự án/chủ đầu tư/giá."
        )

        match_logs.append({
            "Số TBMT": so_tbmt,
            "Trạng thái": "Direct scan - bỏ qua fuzzy match",
            "Số KHLCNT chọn": existing_kh,
            "Số KHLCNT ứng viên": direct_kh_scan,
            "Điểm": "",
            "Gói score": "",
            "Dự án score": "",
            "Chủ đầu tư score": "",
            "Giá score": "",
            "Tên gói TBMT": clean_value(get_value(tbmt_row, C_TBMT_GOI)),
            "Dự án TBMT": clean_value(get_value(tbmt_row, C_TBMT_DU_AN)),
            "Chủ đầu tư TBMT": clean_value(get_value(tbmt_row, C_TBMT_CDT)),
            "Đơn vị mời thầu TBMT": clean_value(get_value(tbmt_row, C_TBMT_DVMT)),
            "Top 3 candidates": "",
            "Ghi chú": "Đã scan được Số KHLCNT từ PDF nên không fuzzy match",
        })
        continue

    if existing_kh and existing_kh.lower() != "nan" and not OVERWRITE_EXISTING_KHLCNT:
        kept_count += 1

        df_tbmt.at[idx, "Trạng thái match KHLCNT"] = "Giữ Số KHLCNT sẵn có"
        df_tbmt.at[idx, "Số KHLCNT ứng viên"] = existing_kh
        df_tbmt.at[idx, "Ghi chú match KHLCNT"] = "Đã có Số KHLCNT nên không match lại"

        match_logs.append({
            "Số TBMT": so_tbmt,
            "Trạng thái": "Giữ Số KHLCNT sẵn có",
            "Số KHLCNT chọn": existing_kh,
            "Số KHLCNT ứng viên": existing_kh,
            "Điểm": "",
            "Gói score": "",
            "Dự án score": "",
            "Chủ đầu tư score": "",
            "Giá score": "",
            "Tên gói TBMT": clean_value(get_value(tbmt_row, C_TBMT_GOI)),
            "Dự án TBMT": clean_value(get_value(tbmt_row, C_TBMT_DU_AN)),
            "Chủ đầu tư TBMT": clean_value(get_value(tbmt_row, C_TBMT_CDT)),
            "Đơn vị mời thầu TBMT": clean_value(get_value(tbmt_row, C_TBMT_DVMT)),
            "Top 3 candidates": "",
            "KHLCNT Item Key match": "",
            "Tên gói KHLCNT match": "",
            "Giá KHLCNT match": "",
            "Phiên bản KHLCNT match": "",
            "Version score": "",
            "Ghi chú": "Đã có Số KHLCNT nên không match lại",
        })
        continue
    if direct_kh_scan:
      continue
    best, top3 = match_one(tbmt_row, kh_candidates)

    if best is None:
        no_match_count += 1
        status = "Không match"
        chosen = ""
        best_so_kh = ""
        sc = {"total": "", "goi": "", "du_an": "", "cdt": "", "gia": ""}
        top3 = []

    else:
        sc = best["score"]
        cand = best["candidate"]
        best_so_kh = cand["Số KHLCNT"]

        if is_safe_auto(sc, top3):
            auto_count += 1
            status = "Auto matched - safe"
            chosen = best_so_kh
            df_tbmt.at[idx, C_TBMT_KHLCNT] = best_so_kh

        elif sc["total"] >= REVIEW_MATCH_SCORE:
            review_count += 1
            status = "Cần kiểm tra"
            chosen = ""

        else:
            no_match_count += 1
            status = "Không match"
            chosen = ""

    df_tbmt.at[idx, "Trạng thái match KHLCNT"] = status

    if sc["total"] != "":
        df_tbmt.at[idx, "Điểm match KHLCNT"] = int(round(sc["total"]))

    df_tbmt.at[idx, "Nguồn match KHLCNT"] = "Python fuzzy safe: Cell2 output + Zoho export"
    df_tbmt.at[idx, "Số KHLCNT ứng viên"] = best_so_kh
    df_tbmt.at[idx, "Top 3 KHLCNT candidates"] = str(top3)

    if best is not None:
        df_tbmt.at[idx, "KHLCNT Item Key match"] = cand.get("KHLCNT Item Key", "")
        df_tbmt.at[idx, "Tên gói KHLCNT match"] = cand.get("Tên gói KHLCNT", "")
        df_tbmt.at[idx, "Giá KHLCNT match"] = cand.get("Giá KHLCNT", "")
        df_tbmt.at[idx, "Phiên bản KHLCNT match"] = cand.get("Phiên bản KHLCNT", "")
        df_tbmt.at[idx, "Version score"] = sc.get("version", "")

    df_tbmt.at[idx, "Ghi chú match KHLCNT"] = f"{status}; ứng viên {best_so_kh}; item={df_tbmt.at[idx, 'KHLCNT Item Key match']}; goi={sc['goi']}, du_an={sc['du_an']}, cdt={sc['cdt']}, gia={sc['gia']}, version={sc.get('version', '')}"

    match_logs.append({
        "Số TBMT": so_tbmt,
        "Trạng thái": status,
        "Số KHLCNT chọn": chosen,
        "Số KHLCNT ứng viên": best_so_kh,
        "Điểm": sc["total"],
        "Gói score": sc["goi"],
        "Dự án score": sc["du_an"],
        "Chủ đầu tư score": sc["cdt"],
        "Giá score": sc["gia"],
        "Version score": sc.get("version", ""),
        "Order score": sc.get("order", ""),
        "KHLCNT Item Key match": cand.get("KHLCNT Item Key", "") if best is not None else "",
        "Tên gói KHLCNT match": cand.get("Tên gói KHLCNT", "") if best is not None else "",
        "Giá KHLCNT match": cand.get("Giá KHLCNT", "") if best is not None else "",
        "Phiên bản KHLCNT match": cand.get("Phiên bản KHLCNT", "") if best is not None else "",
        "Tên gói TBMT": clean_value(get_value(tbmt_row, C_TBMT_GOI)),
        "Dự án TBMT": clean_value(get_value(tbmt_row, C_TBMT_DU_AN)),
        "Chủ đầu tư TBMT": clean_value(get_value(tbmt_row, C_TBMT_CDT)),
        "Đơn vị mời thầu TBMT": clean_value(get_value(tbmt_row, C_TBMT_DVMT)),
        "Top 3 candidates": str(top3),
    })


df_match_candidates = pd.DataFrame(match_logs)

# Giữ lịch sử match candidates để lần sau có KHLCNT mới vẫn xem lại được các lần match trước
MATCH_RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
if not df_match_candidates.empty:
    df_match_candidates.insert(0, "Match run id", MATCH_RUN_ID)

print("\nSummary:")
print("Kept existing:", kept_count)
print("Auto matched safe:", auto_count)
print("Need review:", review_count)
print("No match:", no_match_count)
print("TBMT có Số KHLCNT sau match:", df_tbmt[C_TBMT_KHLCNT].astype(str).str.strip().ne("").sum())
# ============================================================
# UPDATE MASTER OUTPUT AFTER AUTO MATCH
# Cập nhật Auto matched ngược lại vào Zoho_PDF_Import_Output_LATEST.xlsx
# ============================================================

def update_master_latest_after_auto_match(
    master_output_path,
    df_tbmt_matched,
    tbmt_key_col,
    tbmt_sheet_name="TBMT_Zoho_Import",
):
    master_output_path = Path(master_output_path)

    if not master_output_path.exists():
        print("Không thấy master output để cập nhật:", master_output_path)
        return 0

    if df_tbmt_matched is None or df_tbmt_matched.empty:
        print("df_tbmt_matched rỗng, không cập nhật master.")
        return 0

    # Chỉ cập nhật dòng Auto matched có Số KHLCNT
    auto_mask = (
        df_tbmt_matched["Trạng thái match KHLCNT"]
        .astype(str)
        .str.contains("Auto matched", case=False, na=False)
        &
        df_tbmt_matched[C_TBMT_KHLCNT]
        .astype(str)
        .str.strip()
        .replace({"nan": "", "None": "", "NaN": ""})
        .ne("")
    )

    df_auto = df_tbmt_matched.loc[auto_mask].copy()
    df_auto = fix_code_text_columns(df_auto)

    if df_auto.empty:
        print("Không có dòng Auto matched nào để cập nhật vào master.")
        return 0

    update_cols = [
        C_TBMT_KHLCNT,
        "Trạng thái match KHLCNT",
        "Điểm match KHLCNT",
        "Nguồn match KHLCNT",
        "Ghi chú match KHLCNT",
        "Số KHLCNT ứng viên",
        "Top 3 KHLCNT candidates",
        "KHLCNT Item Key match",
        "Tên gói KHLCNT match",
        "Giá KHLCNT match",
        "Phiên bản KHLCNT match",
        "Version score",
    ]

    update_cols = [c for c in update_cols if c in df_auto.columns]

    sheets = pd.read_excel(master_output_path, sheet_name=None, dtype=str)

    if tbmt_sheet_name not in sheets:
        raise KeyError(f"Không tìm thấy sheet {tbmt_sheet_name} trong {master_output_path}")

    df_master_tbmt = sheets[tbmt_sheet_name].copy()
    df_master_tbmt = fix_code_text_columns(df_master_tbmt)

    df_master_tbmt["_tbmt_key_update"] = (
        df_master_tbmt[tbmt_key_col]
        .astype(str)
        .str.strip()
    )

    df_auto["_tbmt_key_update"] = (
        df_auto[tbmt_key_col]
        .astype(str)
        .str.strip()
    )

    df_auto_update = (
        df_auto[["_tbmt_key_update"] + update_cols]
        .drop_duplicates("_tbmt_key_update", keep="last")
        .copy()
    )

    df_master_tbmt = df_master_tbmt.merge(
        df_auto_update,
        on="_tbmt_key_update",
        how="left",
        suffixes=("", "__AUTO")
    )

    updated_count = 0

    for col in update_cols:
        auto_col = col + "__AUTO"

        if auto_col not in df_master_tbmt.columns:
            continue

        if col not in df_master_tbmt.columns:
            df_master_tbmt[col] = ""

        valid_mask = (
            df_master_tbmt[auto_col]
            .astype(str)
            .str.strip()
            .replace({"nan": "", "None": "", "NaN": ""})
            .ne("")
        )

        updated_count = max(updated_count, int(valid_mask.sum()))
        df_master_tbmt.loc[valid_mask, col] = df_master_tbmt.loc[valid_mask, auto_col]

        df_master_tbmt = df_master_tbmt.drop(columns=[auto_col])

    df_master_tbmt = df_master_tbmt.drop(columns=["_tbmt_key_update"], errors="ignore")
    df_master_tbmt = fix_code_text_columns(df_master_tbmt)

    sheets[tbmt_sheet_name] = df_master_tbmt

    tmp_master_path = master_output_path.with_name(master_output_path.stem + ".tmp.xlsx")
    with pd.ExcelWriter(tmp_master_path, engine="openpyxl") as writer:
        for sheet_name, df_sheet in sheets.items():
            df_sheet = fix_code_text_columns(df_sheet)
            write_excel_sheet_text_safe(writer, df_sheet, sheet_name, TEXT_SAFE_COLUMNS)
    safe_replace_excel(tmp_master_path, master_output_path)

    print(f"Updated master LATEST after auto match: {updated_count} TBMT rows")
    print("Master file updated:", master_output_path)

    return updated_count


updated_master_count = update_master_latest_after_auto_match(
    master_output_path=MASTER_OUTPUT_PATH,
    df_tbmt_matched=df_tbmt,
    tbmt_key_col=C_TBMT_SO,
    tbmt_sheet_name="TBMT_Zoho_Import",
)
# ============================================================
# UPDATE TBMT_FOR_IMPORT AFTER MATCH
# Để người nhập liệu import đúng file đã có Số KHLCNT Auto matched
# ============================================================

df_tbmt_for_import = (
    df_tbmt
    .drop(
        columns=[
            "_tbmt_key",
            "_has_khlcnt",
            "_source_priority",
            "_Nguồn dữ liệu",
        ],
        errors="ignore"
    )
    .copy()
)

df_tbmt_for_import = fix_code_text_columns(df_tbmt_for_import)

with pd.ExcelWriter(TBMT_IMPORT_PATH, engine="openpyxl") as writer:
    write_excel_sheet_text_safe(
        writer,
        df_tbmt_for_import,
        "TBMT_Zoho_Import",
        TEXT_SAFE_COLUMNS
    )

print("Updated TBMT_for_import after match:", TBMT_IMPORT_PATH)
# ============================================================
# EXPORT MATCH OUTPUT - TEXT SAFE
# ============================================================

df_tbmt_export = (
    df_tbmt
    .drop(columns=["_tbmt_key", "_has_khlcnt", "_source_priority", "_Nguồn dữ liệu"], errors="ignore")
    .copy()
)

df_khlcnt_export = (
    df_khlcnt
    .drop(columns=["_khlcnt_key", "_Nguồn dữ liệu"], errors="ignore")
    .copy()
)
df_tbmt_export = fix_code_text_columns(df_tbmt_export)
df_match_candidates = fix_code_text_columns(df_match_candidates)
df_khlcnt_export = fix_code_text_columns(df_khlcnt_export)

with pd.ExcelWriter(MATCHED_TBMT_OUTPUT_PATH, engine="openpyxl") as writer:
    write_excel_sheet_text_safe(
        writer,
        df_tbmt_export,
        "TBMT_Matched_Import",
        TEXT_SAFE_COLUMNS
    )

    write_excel_sheet_text_safe(
        writer,
        df_match_candidates,
        "Match_Candidates",
        TEXT_SAFE_COLUMNS
    )

    write_excel_sheet_text_safe(
        writer,
        df_khlcnt_export,
        "KHLCNT_Items_Used",
        TEXT_SAFE_COLUMNS
    )

with pd.ExcelWriter(MATCH_LOG_PATH, engine="openpyxl") as writer:
    write_excel_sheet_text_safe(
        writer,
        df_match_candidates,
        "Match_Candidates",
        TEXT_SAFE_COLUMNS
    )

# Append lịch sử match candidates, không xoá các lần chạy trước
try:
    if MATCH_CANDIDATES_HISTORY_PATH.exists():
        df_match_history_old = pd.read_excel(MATCH_CANDIDATES_HISTORY_PATH, dtype=str)
    else:
        df_match_history_old = pd.DataFrame()

    df_match_history = pd.concat([df_match_history_old, df_match_candidates], ignore_index=True, sort=False)
    tmp_history = MATCH_CANDIDATES_HISTORY_PATH.with_suffix(".tmp.xlsx")
    with pd.ExcelWriter(tmp_history, engine="openpyxl") as writer:
        write_excel_sheet_text_safe(writer, df_match_history, "Match_Candidates_History", TEXT_SAFE_COLUMNS)
    safe_replace_excel(tmp_history, MATCH_CANDIDATES_HISTORY_PATH)
    print("Updated match candidates history:", MATCH_CANDIDATES_HISTORY_PATH)
except Exception as e:
    print("Không append được Match_Candidates_History, nhưng file match mới vẫn đã xuất:", e)

print("\nDONE CELL 3")
print("Matched TBMT output:", MATCHED_TBMT_OUTPUT_PATH)
print("Match log:", MATCH_LOG_PATH)

display(df_tbmt[[C_TBMT_SO, C_TBMT_KHLCNT, "Trạng thái match KHLCNT", "Điểm match KHLCNT", "Số KHLCNT ứng viên", "Ghi chú match KHLCNT"]].head(100))


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/Zoho_Procurement True
TBMT_DIR: /content/drive/MyDrive/Zoho_Procurement/TBMT
KHLCNT_DIR: /content/drive/MyDrive/Zoho_Procurement/KHLCNT
OUTPUT_DIR: /content/drive/MyDrive/Zoho_Procurement/output_excel
INPUT_MATCH_DIR: /content/drive/MyDrive/Zoho_Procurement/input_match
LOG_DIR: /content/drive/MyDrive/Zoho_Procurement/logs
CONFIG_DIR: /content/drive/MyDrive/Zoho_Procurement/config
TEXT_CACHE_DIR: /content/drive/MyDrive/Zoho_Procurement/logs/text_cache
CHECKPOINT_DIR: /content/drive/MyDrive/Zoho_Procurement/logs/checkpoints

TBMT PDFs:
- TBMT_10_3_2026.pdf
- TBMT_10_4_2026.pdf
- TBMT_10_5_2026.pdf
- TBMT_10_6_2026.pdf
- TBMT_11_3_2026.pdf
- TBMT_11_4_2026.pdf
- TBMT_11_5_2026.pdf
- TBMT_11_6_2026.pdf
- TBMT_12_3_2026.pdf
- TBMT_12_4_2026.pdf
- 

/tmp/ipykernel_1848/450300090.py:1252: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_tbmt = pd.concat([df_tbmt_old, df_tbmt_new], ignore_index=True, sort=False)
/tmp/ipykernel_1848/450300090.py:1253: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_khlcnt = pd.concat([df_khlcnt_old, df_khlcnt_new], ignore_index=True, sort=False)
/tmp/ipykernel_1848/450300090.py:1254: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this wi

Direct scan memory: /content/drive/MyDrive/Zoho_Procurement/output_excel/direct_scan_memory/direct_khlcnt_scan_memory.xlsx
Direct scan memory rows: 6123
Direct scan rows in TBMT: 6123
After combine + dedupe:
Accounts: 1851
TBMT: 6123
KHLCNT parent: 9164
KHLCNT items for match: 10873
Match candidates preview: 0


/tmp/ipykernel_1848/450300090.py:394: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  values = df[col_name].head(100).fillna("").astype(str).tolist()
/tmp/ipykernel_1848/450300090.py:394: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  values = df[col_name].head(100).fillna("").astype(str).tolist()
/tmp/ipykernel_1848/450300090.py:394: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('

Không có PDF mới để ghi registry.

DONE CELL 2
Master output: /content/drive/MyDrive/Zoho_Procurement/output_excel/Zoho_PDF_Import_Output_LATEST.xlsx
Accounts import: /content/drive/MyDrive/Zoho_Procurement/output_excel/Accounts_for_import.xlsx
KHLCNT import: /content/drive/MyDrive/Zoho_Procurement/output_excel/KHLCNT_for_import.xlsx
KHLCNT items for match: /content/drive/MyDrive/Zoho_Procurement/output_excel/KHLCNT_items_for_match.xlsx
TBMT import: /content/drive/MyDrive/Zoho_Procurement/output_excel/TBMT_for_import.xlsx
Cell 2 output: /content/drive/MyDrive/Zoho_Procurement/output_excel/Zoho_PDF_Import_Output_LATEST.xlsx
Không có Zoho TBMT export trong input_match/.
Không có Zoho KHLCNT export trong input_match/.
Using KHLCNT items for matching: KHLCNT_Items_Match
TBMT total before dedupe: 6123
KHLCNT total before cleanup: 10873

Detected columns:
TBMT Số: Số TBMT
TBMT Số KHLCNT: Số KHLCNT
TBMT Tên gói: Tên gói thầu
TBMT Dự án: Dự án
TBMT Chủ đầu tư: Chủ đầu tư
TBMT Đơn vị mời thầu: 

,Số TBMT,Số KHLCNT,Trạng thái match KHLCNT,Điểm match KHLCNT,Số KHLCNT ứng viên,Ghi chú match KHLCNT
0,IB2500489278-01,,Direct scan - bỏ qua fuzzy match,NaN,NAN,TBMT scan được Số KHLCNT từ PDF: NAN. Không ch...
1,IB2500589421-00,,Direct scan - bỏ qua fuzzy match,NaN,NAN,TBMT scan được Số KHLCNT từ PDF: NAN. Không ch...
2,IB2500589421-01,,Direct scan - bỏ qua fuzzy match,NaN,NAN,TBMT scan được Số KHLCNT từ PDF: NAN. Không ch...
3,IB2500629894-00,,Direct scan - bỏ qua fuzzy match,NaN,NAN,TBMT scan được Số KHLCNT từ PDF: NAN. Không ch...
4,IB2600009965-00,,Direct scan - bỏ qua fuzzy match,NaN,NAN,TBMT scan được Số KHLCNT từ PDF: NAN. Không ch...
...,...,...,...,...,...,...
95,IB2600070017-00,,Direct scan - bỏ qua fuzzy match,NaN,NAN,TBMT scan được Số KHLCNT từ PDF: NAN. Không ch...
96,IB2600070048-00,,Direct scan - bỏ qua fuzzy match,NaN,NAN,TBMT scan được Số KHLCNT từ PDF: NAN. Không ch...
97,IB2600070058-00,,Direct scan - bỏ qua fuzzy match,NaN,NAN,TBMT scan được Số KHLCNT từ PDF: NAN. Không ch...
98,IB2600070185-00,,Direct scan - bỏ qua fuzzy match,NaN,NAN,TBMT scan được Số KHLCNT từ PDF: NAN. Không ch...


In [ ]:
# ============================================================
# CELL 2 - MANUAL KHLCNT INPUT -> MODEL TRAINING -> AUTO UPDATE
# Google Colab + Google Drive
# ============================================================
# Purpose:
# 1) Read outputs from the existing main pipeline without modifying that code.
# 2) Create a simple manual input file containing TBMT codes that still need KHLCNT.
# 3) User copies a full KHLCNT code from the web into the file.
# 4) The entered pair is treated as label=1 for model training when the KHLCNT exists in master.
# 5) Pending entered codes are stored in memory and rechecked every run.
# 6) Confirmed mappings update BOTH:
#       - output_excel/Zoho_PDF_Import_Output_LATEST.xlsx
#       - output_excel/TBMT_for_import.xlsx
# 7) XGBoost trains from confirmed manual labels and may auto-apply high-confidence predictions.
# ============================================================

import os
import re
import json
import math
import subprocess
import sys
from pathlib import Path
from datetime import datetime

# ============================================================
# PACKAGE HELPERS
# ============================================================

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pip_name])

ensure_package("pandas")
ensure_package("openpyxl")
ensure_package("rapidfuzz")
ensure_package("unidecode")
ensure_package("joblib")
ensure_package("sklearn", "scikit-learn")
ensure_package("xgboost")

import pandas as pd
import numpy as np
import joblib
from rapidfuzz import fuzz
from unidecode import unidecode
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity

# sentence-transformers is optional. If it cannot be installed/downloaded,
# the code still runs with text/fuzzy/ngram features.
USE_SENTENCE_EMBEDDING = True
try:
    ensure_package("sentence_transformers", "sentence-transformers")
    from sentence_transformers import SentenceTransformer
except Exception as e:
    USE_SENTENCE_EMBEDDING = False
    SentenceTransformer = None
    print("Sentence embedding unavailable. Fallback to fuzzy/ngram features only.")
    print("Reason:", repr(e))

# ============================================================
# 0. PATHS - DO NOT MOVE OLD PIPELINE FILES
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

BASE_DIR = Path("/content/drive/MyDrive/Zoho_Procurement")
OUTPUT_DIR = BASE_DIR / "output_excel"
LOG_DIR = BASE_DIR / "logs"

# Files created by the old/main code. Keep these locations unchanged.
MASTER_OUTPUT_PATH = OUTPUT_DIR / "Zoho_PDF_Import_Output_LATEST.xlsx"
TBMT_IMPORT_PATH = OUTPUT_DIR / "TBMT_for_import.xlsx"
MATCH_CANDIDATES_PATH = LOG_DIR / "Match_Candidates.xlsx"
DIRECT_SCAN_MEMORY_PATH = OUTPUT_DIR / "direct_scan_memory" / "direct_khlcnt_scan_memory.xlsx"
# Only manual/model related files are placed in a subfolder.
MODEL_WORK_DIR = OUTPUT_DIR / "manual_khlcnt_model"
MODEL_DIR = MODEL_WORK_DIR / "models"
CHECKPOINT_DIR = MODEL_WORK_DIR / "checkpoints"

for folder in [MODEL_WORK_DIR, MODEL_DIR, CHECKPOINT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MANUAL_FILL_PATH = MODEL_WORK_DIR / "Manual_KHLCNT_Code_To_Fill.xlsx"
MEMORY_PATH = MODEL_WORK_DIR / "manual_tbmt_khlcnt_code_memory.xlsx"
MAPPING_PATH = MODEL_WORK_DIR / "manual_tbmt_khlcnt_mapping.xlsx"
TRAINING_DATA_PATH = MODEL_WORK_DIR / "match_training_data.xlsx"
MODEL_PATH = MODEL_DIR / "tbmt_khlcnt_xgboost_model.joblib"
METRICS_PATH = MODEL_DIR / "tbmt_khlcnt_xgboost_metrics.xlsx"
EMBEDDING_CACHE_PATH = MODEL_DIR / "text_embedding_cache.joblib"
PREDICTIONS_PATH = MODEL_WORK_DIR / "Model_Predictions.xlsx"
STATE_PATH = CHECKPOINT_DIR / "last_success_state.json"
PIPELINE_CHECKPOINT_PATH = CHECKPOINT_DIR / "model_pipeline_checkpoint.xlsx"

TBMT_SHEET = "TBMT_Zoho_Import"
KHLCNT_SHEET = "KHLCNT_Zoho_Import"

ALLOW_OVERWRITE_EXISTING = False
MODEL_AUTO_APPLY = True
MODEL_PROB_THRESHOLD = 0.92
MODEL_MARGIN_THRESHOLD = 0.08
MIN_POSITIVE_LABELS_TO_TRAIN = 5
MIN_NEGATIVE_LABELS_TO_TRAIN = 5
MAX_NEGATIVES_PER_POSITIVE = 5

print("MASTER_OUTPUT_PATH:", MASTER_OUTPUT_PATH)
print("TBMT_IMPORT_PATH:", TBMT_IMPORT_PATH)
print("MATCH_CANDIDATES_PATH:", MATCH_CANDIDATES_PATH)
print("MODEL_WORK_DIR:", MODEL_WORK_DIR)
print("MANUAL_FILL_PATH:", MANUAL_FILL_PATH)

# ============================================================
# 1. GENERIC SAFE IO HELPERS
# ============================================================

TEXT_SAFE_COLUMNS = [
    "Số TBMT", "Mã gốc TBMT", "Mã Phiên Bản", "Phiên Bản Trước dự kiến",
    "Số KHLCNT", "Số KHLCNT scan từ PDF", "Số KHLCNT người nhập", "Số KHLCNT chuẩn hóa", "Số KHLCNT candidate",
    "Số KHLCNT ứng viên", "Mã gốc KHLCNT", "Mã gốc KHLCNT candidate",
    "Phiên bản KHLCNT", "Phiên bản KHLCNT candidate", "KHLCNT Item Key", "KHLCNT Item Key match",
]


def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def save_state(step, extra=None):
    state = {"step": step, "timestamp": now_str()}
    if extra:
        state.update(extra)
    tmp = STATE_PATH.with_suffix(".tmp.json")
    tmp.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(STATE_PATH)


def is_blank(v):
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except Exception:
        pass
    return str(v).strip() == "" or str(v).strip().lower() in ["nan", "none", "null"]


def clean_value(v):
    if is_blank(v):
        return ""
    return str(v).strip()


def fix_version_value(v):
    if is_blank(v):
        return ""
    s = str(v).strip()
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    if re.fullmatch(r"\d+", s):
        return s.zfill(2)
    return s


def fix_text_value(v):
    if is_blank(v):
        return ""
    s = str(v).strip()
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    return s


def fix_code_text_columns(df):
    if df is None or df.empty:
        return df
    df = df.copy()
    for col in ["Mã Phiên Bản", "Phiên bản KHLCNT", "Phiên bản KHLCNT candidate"]:
        if col in df.columns:
            df[col] = df[col].apply(fix_version_value).astype(str)
    for col in TEXT_SAFE_COLUMNS:
        if col in df.columns:
            df[col] = df[col].apply(fix_text_value).astype(str)
    return df


def safe_replace(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        raise FileNotFoundError(f"Không thấy file tạm: {src}")
    src.replace(dst)


def write_excel_text_safe(path, sheets_dict):
    path = Path(path)
    tmp = path.with_suffix(".tmp.xlsx")
    with pd.ExcelWriter(tmp, engine="openpyxl") as writer:
        for sheet_name, df in sheets_dict.items():
            if df is None:
                df = pd.DataFrame()
            df = fix_code_text_columns(df.copy())
            safe_sheet = sheet_name[:31]
            df.to_excel(writer, index=False, sheet_name=safe_sheet)
            ws = writer.sheets[safe_sheet]
            ws.freeze_panes = "A2"
            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.alignment = Alignment(horizontal="center")
                cell.fill = PatternFill(start_color="E8F0FE", end_color="E8F0FE", fill_type="solid")
            for idx, col_name in enumerate(df.columns, start=1):
                col_letter = get_column_letter(idx)
                if col_name in TEXT_SAFE_COLUMNS:
                    for cell in ws[col_letter]:
                        cell.number_format = "@"
                try:
                    values = df[col_name].head(200).fillna("").astype(str).tolist()
                    width = max([len(str(col_name))] + [len(x) for x in values]) + 2
                    ws.column_dimensions[col_letter].width = min(max(width, 12), 55)
                except Exception:
                    pass
    safe_replace(tmp, path)


def read_excel_sheet(path, sheet_name=None, dtype=str):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        if sheet_name:
            return pd.read_excel(path, sheet_name=sheet_name, dtype=dtype)
        return pd.read_excel(path, dtype=dtype)
    except Exception:
        try:
            return pd.read_excel(path, sheet_name=0, dtype=dtype)
        except Exception:
            return pd.DataFrame()


def read_all_sheets(path):
    path = Path(path)
    if not path.exists():
        return {}
    try:
        return pd.read_excel(path, sheet_name=None, dtype=str)
    except Exception as e:
        raise RuntimeError(f"Không đọc được workbook: {path}. Error: {e}")


def replace_sheet_in_workbook(path, sheet_name, new_df):
    """Preserve other sheets, replace only one sheet."""
    path = Path(path)
    sheets = read_all_sheets(path)
    sheets[sheet_name] = new_df
    write_excel_text_safe(path, sheets)

# ============================================================
# 2. COLUMN + CODE HELPERS
# ============================================================


def find_col_soft(df, names, required=False):
    lower = {str(c).strip().lower(): c for c in df.columns}
    for n in names:
        key = n.strip().lower()
        if key in lower:
            return lower[key]
    if required:
        raise KeyError(f"Không tìm thấy cột nào trong danh sách: {names}")
    return None


def normalize_tbmt_code(v):
    s = clean_value(v).upper().replace(" ", "")
    m = re.search(r"(IB\d+)\s*-?\s*(\d{2})", s)
    if m:
        return f"{m.group(1)}-{m.group(2)}"
    return s


def normalize_khlcnt_code(v):
    s = clean_value(v).upper().replace(" ", "")
    m = re.search(r"(PL\d+)\s*-?\s*(\d{2})", s)
    if m:
        return f"{m.group(1)}-{m.group(2)}"
    return s


def is_valid_khlcnt_format(v):
    s = normalize_khlcnt_code(v)
    return bool(re.fullmatch(r"PL\d+-\d{2}", s))


def kh_root(code):
    s = normalize_khlcnt_code(code)
    return s.split("-")[0] if "-" in s else s


def get_version_num(code_or_version):
    s = clean_value(code_or_version)
    m = re.search(r"-(\d{2})$", s)
    if m:
        return int(m.group(1))
    if re.fullmatch(r"\d+", s):
        return int(s)
    if re.fullmatch(r"\d+\.0+", s):
        return int(float(s))
    return None


def clean_number(v):
    if is_blank(v):
        return None
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        try:
            if pd.isna(v):
                return None
        except Exception:
            pass
        return float(v)
    s = str(v).strip()
    s = re.sub(r"[^\d,.\-]", "", s)
    if s in ["", "-", ".", ","]:
        return None
    try:
        if "." in s and "," in s:
            if s.rfind(",") > s.rfind("."):
                s = s.replace(".", "").replace(",", ".")
            else:
                s = s.replace(",", "")
        elif "," in s:
            parts = s.split(",")
            if len(parts) == 2 and len(parts[1]) in [1, 2]:
                s = parts[0].replace(".", "") + "." + parts[1]
            else:
                s = s.replace(",", "")
        elif "." in s:
            parts = s.split(".")
            if len(parts) > 2:
                s = "".join(parts)
            elif len(parts) == 2 and len(parts[1]) == 3:
                s = s.replace(".", "")
        return float(s)
    except Exception:
        return None


def norm_text(v):
    s = clean_value(v).lower()
    s = unidecode(s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def token_set(v):
    s = norm_text(v)
    stop = {
        "goi", "thau", "so", "mua", "sam", "cung", "cap", "nam", "du", "an",
        "hang", "hoa", "dich", "vu", "theo", "cho", "tai", "va", "cac", "cua",
        "lan", "dot", "thu", "trong", "ngoai", "nuoc"
    }
    toks = [t for t in s.split() if len(t) >= 2 and t not in stop and not t.isdigit()]
    return set(toks)


def keyword_overlap(a, b):
    ta, tb = token_set(a), token_set(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / max(1, min(len(ta), len(tb)))


def jaccard(a, b):
    ta, tb = token_set(a), token_set(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / max(1, len(ta | tb))


def char_ngrams(s, n=3):
    s = norm_text(s).replace(" ", "_")
    if len(s) < n:
        return {s} if s else set()
    return {s[i:i+n] for i in range(len(s)-n+1)}


def char_ngram_similarity(a, b, n=3):
    ga, gb = char_ngrams(a, n), char_ngrams(b, n)
    if not ga or not gb:
        return 0.0
    return len(ga & gb) / max(1, len(ga | gb))

# ============================================================
# 3. LOAD MASTER / IMPORT / EXISTING MANUAL FILES
# ============================================================

if not MASTER_OUTPUT_PATH.exists():
    raise FileNotFoundError(f"Không thấy file tổng. Hãy chạy code cũ Cell 1-3 trước: {MASTER_OUTPUT_PATH}")
if not TBMT_IMPORT_PATH.exists():
    raise FileNotFoundError(f"Không thấy TBMT_for_import. Hãy chạy code cũ Cell 1-3 trước: {TBMT_IMPORT_PATH}")

master_sheets = read_all_sheets(MASTER_OUTPUT_PATH)
if TBMT_SHEET not in master_sheets:
    raise KeyError(f"File tổng không có sheet {TBMT_SHEET}")
if KHLCNT_SHEET not in master_sheets:
    raise KeyError(f"File tổng không có sheet {KHLCNT_SHEET}")

df_tbmt_master = fix_code_text_columns(master_sheets[TBMT_SHEET])
df_kh_master = fix_code_text_columns(master_sheets[KHLCNT_SHEET])
df_tbmt_import = fix_code_text_columns(read_excel_sheet(TBMT_IMPORT_PATH, TBMT_SHEET))

C_TBMT_SO = find_col_soft(df_tbmt_master, ["Số TBMT", "So TBMT", "Name"], required=True)
C_TBMT_KHLCNT = find_col_soft(df_tbmt_master, ["Số KHLCNT", "So KHLCNT", "Số KHLCNT.Name", "So KHLCNT.Name"], required=False)
C_TBMT_GOI = find_col_soft(df_tbmt_master, ["Tên gói thầu", "Ten goi thau"], required=False)
C_TBMT_DU_AN = find_col_soft(df_tbmt_master, ["Dự án", "Du an"], required=False)
C_TBMT_CDT = find_col_soft(df_tbmt_master, ["Chủ đầu tư", "Chu dau tu"], required=False)
C_TBMT_DVMT = find_col_soft(df_tbmt_master, ["Đơn vị mời thầu", "Don vi moi thau"], required=False)
C_TBMT_GIA = find_col_soft(df_tbmt_master, ["Giá gói thầu", "Gia goi thau", "Tổng giá trị", "Tong gia tri"], required=False)
C_TBMT_VERSION = find_col_soft(df_tbmt_master, ["Mã Phiên Bản", "Ma Phien Ban", "Phiên bản TBMT", "Phien ban TBMT"], required=False)

if C_TBMT_KHLCNT is None:
    C_TBMT_KHLCNT = "Số KHLCNT"
    df_tbmt_master[C_TBMT_KHLCNT] = ""

C_KH_SO = find_col_soft(df_kh_master, ["Số KHLCNT", "So KHLCNT", "Name"], required=True)
C_KH_GOI = find_col_soft(df_kh_master, ["Tên gói thầu trong KHMT", "Ten goi thau trong KHMT", "Tên gói thầu", "Ten goi thau"], required=False)
C_KH_DU_AN = find_col_soft(df_kh_master, ["Tên dự án", "Ten du an", "Dự án", "Du an"], required=False)
C_KH_CDT = find_col_soft(df_kh_master, ["Chủ đầu tư-KHLCNT", "Chu dau tu-KHLCNT", "Chủ đầu tư", "Chu dau tu"], required=False)
C_KH_GIA = find_col_soft(df_kh_master, ["Giá gói thầu trong KHMT", "Gia goi thau trong KHMT", "Giá gói thầu", "Gia goi thau", "Tổng mức đầu tư", "Tong muc dau tu"], required=False)
C_KH_VERSION = find_col_soft(df_kh_master, ["Phiên bản KHLCNT", "Phien ban KHLCNT"], required=False)
C_KH_LATEST = find_col_soft(df_kh_master, ["KHLCNT mới nhất", "KHLCNT moi nhat"], required=False)

for col in ["Trạng thái match KHLCNT", "Điểm match KHLCNT", "Nguồn match KHLCNT", "Ghi chú match KHLCNT", "Số KHLCNT ứng viên"]:
    if col not in df_tbmt_master.columns:
        df_tbmt_master[col] = ""
    if col not in df_tbmt_import.columns:
        df_tbmt_import[col] = ""

if C_TBMT_KHLCNT not in df_tbmt_import.columns:
    df_tbmt_import[C_TBMT_KHLCNT] = ""

# Normalize code columns.
df_tbmt_master[C_TBMT_SO] = df_tbmt_master[C_TBMT_SO].apply(normalize_tbmt_code)
df_tbmt_import[C_TBMT_SO] = df_tbmt_import[C_TBMT_SO].apply(normalize_tbmt_code)
df_kh_master[C_KH_SO] = df_kh_master[C_KH_SO].apply(normalize_khlcnt_code)

kh_codes_set = set(df_kh_master[C_KH_SO].dropna().astype(str).str.strip())
kh_by_code = {str(row[C_KH_SO]).strip(): row for _, row in df_kh_master.iterrows() if str(row[C_KH_SO]).strip()}
tbmt_by_code = {str(row[C_TBMT_SO]).strip(): row for _, row in df_tbmt_master.iterrows() if str(row[C_TBMT_SO]).strip()}

save_state("loaded_master_files", {"tbmt_rows": len(df_tbmt_master), "khlcnt_rows": len(df_kh_master)})

# ============================================================
# 4. MANUAL MEMORY + MAPPING
# ============================================================

MEMORY_COLUMNS = [
    "Số TBMT", "Tên gói thầu", "Dự án", "Chủ đầu tư", "Giá gói thầu",
    "Số KHLCNT người nhập", "Số KHLCNT chuẩn hóa", "Trạng thái memory",
    "Ghi chú", "First input at", "Last checked at", "Source"
]

MAPPING_COLUMNS = [
    "Số TBMT", "Số KHLCNT", "Mã gốc KHLCNT", "Phiên bản KHLCNT",
    "Confirmed at", "Applied to master at", "Applied to import at", "Source", "Ghi chú"
]


def load_table(path, columns):
    df = read_excel_sheet(path)
    if df.empty:
        df = pd.DataFrame(columns=columns)
    for c in columns:
        if c not in df.columns:
            df[c] = ""
    return fix_code_text_columns(df[columns + [c for c in df.columns if c not in columns]])


def row_tbmt_info(so_tbmt):
    row = tbmt_by_code.get(normalize_tbmt_code(so_tbmt))
    if row is None:
        return {"Tên gói thầu": "", "Dự án": "", "Chủ đầu tư": "", "Giá gói thầu": ""}
    return {
        "Tên gói thầu": clean_value(row.get(C_TBMT_GOI, "")),
        "Dự án": clean_value(row.get(C_TBMT_DU_AN, "")),
        "Chủ đầu tư": clean_value(row.get(C_TBMT_CDT, "")) or clean_value(row.get(C_TBMT_DVMT, "")),
        "Giá gói thầu": clean_value(row.get(C_TBMT_GIA, "")),
    }
def import_direct_scan_memory_to_model_memory(df_memory):
    """
    Nhận direct memory từ Cell 2:
    - TBMT scan được Số KHLCNT từ PDF
    - Đưa vào memory của module manual/model
    - Để loại khỏi file nhập tay
    - Khi KHLCNT tổng có mã thì tự apply master/import
    """
    if not DIRECT_SCAN_MEMORY_PATH.exists():
        print("Không có direct scan memory từ Cell 2.")
        return df_memory

    df_direct = read_excel_sheet(DIRECT_SCAN_MEMORY_PATH)
    if df_direct.empty:
        print("Direct scan memory rỗng.")
        return df_memory

    rows = []

    for _, r in df_direct.iterrows():
        so_tbmt = normalize_tbmt_code(r.get("Số TBMT", ""))
        kh_norm = normalize_khlcnt_code(
            r.get("Số KHLCNT chuẩn hóa", "") or r.get("Số KHLCNT scan từ PDF", "")
        )

        if not so_tbmt or not kh_norm:
            continue

        if not is_valid_khlcnt_format(kh_norm):
            status = "Mã KHLCNT scan từ PDF sai format"
        elif kh_norm in kh_codes_set:
            status = "Đã có trong KHLCNT tổng - chờ apply/đã apply"
        else:
            status = "Chưa có trong KHLCNT tổng"

        rows.append({
            "Số TBMT": so_tbmt,
            "Tên gói thầu": clean_value(r.get("Tên gói thầu", "")),
            "Dự án": clean_value(r.get("Dự án", "")),
            "Chủ đầu tư": clean_value(r.get("Chủ đầu tư", "")),
            "Giá gói thầu": clean_value(r.get("Giá gói thầu", "")),
            "Số KHLCNT người nhập": kh_norm,
            "Số KHLCNT chuẩn hóa": kh_norm,
            "Trạng thái memory": status,
            "Ghi chú": "Import từ direct scan memory của Cell 2",
            "First input at": clean_value(r.get("First seen at", "")) or now_str(),
            "Last checked at": now_str(),
            "Source": "Direct scan TBMT PDF from Cell 2",
        })

    if rows:
        df_memory = pd.concat([df_memory, pd.DataFrame(rows)], ignore_index=True, sort=False)
        df_memory["Số TBMT"] = df_memory["Số TBMT"].apply(normalize_tbmt_code)
        df_memory["Số KHLCNT chuẩn hóa"] = df_memory["Số KHLCNT chuẩn hóa"].apply(normalize_khlcnt_code)
        df_memory = df_memory.drop_duplicates(subset=["Số TBMT"], keep="last").reset_index(drop=True)

    print("Imported direct scan memory to model memory:", len(rows))
    return df_memory


df_memory = load_table(MEMORY_PATH, MEMORY_COLUMNS)
df_mapping = load_table(MAPPING_PATH, MAPPING_COLUMNS)
# Nhận direct scan memory từ Cell 2 vào memory của Cell 4
df_memory = import_direct_scan_memory_to_model_memory(df_memory)
# Read current manual fill file BEFORE regenerating it, so user entries are not lost.
df_manual_existing = read_excel_sheet(MANUAL_FILL_PATH)
if not df_manual_existing.empty:
    c_manual_tbmt = find_col_soft(df_manual_existing, ["Số TBMT", "So TBMT"], required=False)
    c_manual_kh = find_col_soft(df_manual_existing, ["Số KHLCNT người nhập", "So KHLCNT nguoi nhap", "Số KHLCNT nguoi nhap"], required=False)

    if c_manual_tbmt and c_manual_kh:
        new_memory_rows = []
        for _, r in df_manual_existing.iterrows():
            so_tbmt = normalize_tbmt_code(r.get(c_manual_tbmt, ""))
            kh_input = clean_value(r.get(c_manual_kh, ""))
            if not so_tbmt or not kh_input:
                continue
            kh_norm = normalize_khlcnt_code(kh_input)
            info = row_tbmt_info(so_tbmt)
            if not is_valid_khlcnt_format(kh_norm):
                status = "Mã KHLCNT sai format"
                note = "Người nhập đã điền nhưng mã không đúng dạng PL...-00"
            elif kh_norm in kh_codes_set:
                status = "Đã có trong KHLCNT tổng - chờ apply/đã apply"
                note = "Mã người nhập tồn tại trong KHLCNT tổng"
            else:
                status = "Chưa có trong KHLCNT tổng"
                note = "Đã lưu memory, lần sau nạp KHLCNT mới sẽ kiểm tra lại"
            new_memory_rows.append({
                "Số TBMT": so_tbmt,
                "Tên gói thầu": info["Tên gói thầu"],
                "Dự án": info["Dự án"],
                "Chủ đầu tư": info["Chủ đầu tư"],
                "Giá gói thầu": info["Giá gói thầu"],
                "Số KHLCNT người nhập": kh_input,
                "Số KHLCNT chuẩn hóa": kh_norm,
                "Trạng thái memory": status,
                "Ghi chú": note,
                "First input at": now_str(),
                "Last checked at": now_str(),
                "Source": "Manual_KHLCNT_Code_To_Fill.xlsx",
            })
        if new_memory_rows:
            df_memory = pd.concat([df_memory, pd.DataFrame(new_memory_rows)], ignore_index=True, sort=False)
            # Dedupe by TBMT. Keep the latest manual entry for each TBMT.
            df_memory["Số TBMT"] = df_memory["Số TBMT"].apply(normalize_tbmt_code)
            df_memory["Số KHLCNT chuẩn hóa"] = df_memory["Số KHLCNT chuẩn hóa"].apply(normalize_khlcnt_code)
            df_memory = df_memory.drop_duplicates(subset=["Số TBMT"], keep="last").reset_index(drop=True)

# Re-check all memory rows against the latest KHLCNT master every run.
for i, r in df_memory.iterrows():
    kh_norm = normalize_khlcnt_code(r.get("Số KHLCNT chuẩn hóa", "") or r.get("Số KHLCNT người nhập", ""))
    df_memory.at[i, "Số KHLCNT chuẩn hóa"] = kh_norm
    df_memory.at[i, "Last checked at"] = now_str()
    if not is_valid_khlcnt_format(kh_norm):
        df_memory.at[i, "Trạng thái memory"] = "Mã KHLCNT sai format"
    elif kh_norm in kh_codes_set:
        df_memory.at[i, "Trạng thái memory"] = "Đã có trong KHLCNT tổng - chờ apply/đã apply"
    else:
        df_memory.at[i, "Trạng thái memory"] = "Chưa có trong KHLCNT tổng"

# Convert valid memory rows to confirmed mapping if KHLCNT exists in master.
new_mapping_rows = []
existing_mapping_keys = set(zip(df_mapping["Số TBMT"].astype(str), df_mapping["Số KHLCNT"].astype(str))) if not df_mapping.empty else set()
for _, r in df_memory.iterrows():
    so_tbmt = normalize_tbmt_code(r.get("Số TBMT", ""))
    kh_norm = normalize_khlcnt_code(r.get("Số KHLCNT chuẩn hóa", ""))

    if not so_tbmt or not kh_norm or kh_norm not in kh_codes_set:
        continue

    key = (so_tbmt, kh_norm)
    if key in existing_mapping_keys:
        continue

    memory_source = clean_value(r.get("Source", ""))
    if "direct scan" in memory_source.lower():
        mapping_source = "Direct scan TBMT PDF as label=1"
        mapping_note = "Mã KHLCNT scan trực tiếp từ TBMT PDF; dùng làm label=1 để train model"
    else:
        mapping_source = "Manual input as label=1"
        mapping_note = "Người nhập tay mã KHLCNT full; dùng làm label=1 để train model"

    new_mapping_rows.append({
        "Số TBMT": so_tbmt,
        "Số KHLCNT": kh_norm,
        "Mã gốc KHLCNT": kh_root(kh_norm),
        "Phiên bản KHLCNT": fix_version_value(get_version_num(kh_norm)),
        "Confirmed at": now_str(),
        "Applied to master at": "",
        "Applied to import at": "",
        "Source": mapping_source,
        "Ghi chú": mapping_note,
    })

if new_mapping_rows:
    df_mapping = pd.concat([df_mapping, pd.DataFrame(new_mapping_rows)], ignore_index=True, sort=False)

# Dedupe mapping by Số TBMT: keep latest confirmed KHLCNT for the TBMT.
df_mapping["Số TBMT"] = df_mapping["Số TBMT"].apply(normalize_tbmt_code)
df_mapping["Số KHLCNT"] = df_mapping["Số KHLCNT"].apply(normalize_khlcnt_code)
df_mapping = df_mapping.drop_duplicates(subset=["Số TBMT"], keep="last").reset_index(drop=True)

write_excel_text_safe(MEMORY_PATH, {"Memory": df_memory})
write_excel_text_safe(MAPPING_PATH, {"Mapping": df_mapping})
save_state("updated_memory_and_mapping", {"memory_rows": len(df_memory), "mapping_rows": len(df_mapping)})

# ============================================================
# 5. APPLY CONFIRMED MAPPING TO MASTER + IMPORT
# ============================================================


def apply_mapping_to_df(df, mapping_df, source_label):
    df = df.copy()
    if C_TBMT_KHLCNT not in df.columns:
        df[C_TBMT_KHLCNT] = ""
    for col in ["Trạng thái match KHLCNT", "Điểm match KHLCNT", "Nguồn match KHLCNT", "Ghi chú match KHLCNT", "Số KHLCNT ứng viên"]:
        if col not in df.columns:
            df[col] = ""
    if C_TBMT_SO not in df.columns:
        return df, set()

    df[C_TBMT_SO] = df[C_TBMT_SO].apply(normalize_tbmt_code)
    idx_by_tbmt = {str(v).strip(): idx for idx, v in df[C_TBMT_SO].items() if str(v).strip()}
    applied_tbmt = set()

    for _, m in mapping_df.iterrows():
        so_tbmt = normalize_tbmt_code(m.get("Số TBMT", ""))
        so_kh = normalize_khlcnt_code(m.get("Số KHLCNT", ""))
        if not so_tbmt or not so_kh or so_kh not in kh_codes_set:
            continue
        if so_tbmt not in idx_by_tbmt:
            continue
        idx = idx_by_tbmt[so_tbmt]
        old = clean_value(df.at[idx, C_TBMT_KHLCNT])
        if old and old != so_kh and not ALLOW_OVERWRITE_EXISTING:
            df.at[idx, "Ghi chú match KHLCNT"] = f"Đã có Số KHLCNT khác ({old}); không overwrite bằng {so_kh}"
            continue
        if old == so_kh:
            applied_tbmt.add(so_tbmt)
            continue
        df.at[idx, C_TBMT_KHLCNT] = so_kh
        df.at[idx, "Trạng thái match KHLCNT"] = "Manual confirmed"
        df.at[idx, "Điểm match KHLCNT"] = 100
        df.at[idx, "Nguồn match KHLCNT"] = source_label
        df.at[idx, "Số KHLCNT ứng viên"] = so_kh
        df.at[idx, "Ghi chú match KHLCNT"] = "Map từ mã KHLCNT người nhập; mã đã tồn tại trong KHLCNT tổng"
        applied_tbmt.add(so_tbmt)
    return df, applied_tbmt


df_tbmt_master, applied_master = apply_mapping_to_df(df_tbmt_master, df_mapping, "Manual code memory")
df_tbmt_import, applied_import = apply_mapping_to_df(df_tbmt_import, df_mapping, "Manual code memory")

for i, m in df_mapping.iterrows():
    so_tbmt = normalize_tbmt_code(m.get("Số TBMT", ""))
    if so_tbmt in applied_master and is_blank(df_mapping.at[i, "Applied to master at"]):
        df_mapping.at[i, "Applied to master at"] = now_str()
    if so_tbmt in applied_import and is_blank(df_mapping.at[i, "Applied to import at"]):
        df_mapping.at[i, "Applied to import at"] = now_str()

# Write back master/import after manual mapping apply.
master_sheets[TBMT_SHEET] = df_tbmt_master
write_excel_text_safe(MASTER_OUTPUT_PATH, master_sheets)
write_excel_text_safe(TBMT_IMPORT_PATH, {TBMT_SHEET: df_tbmt_import})
write_excel_text_safe(MAPPING_PATH, {"Mapping": df_mapping})
save_state("applied_manual_mapping_to_master_import", {"applied_master": len(applied_master), "applied_import": len(applied_import)})

# ============================================================
# 6. FEATURE ENGINEERING FOR MODEL
# ============================================================

embedding_cache = {}
embedding_model = None

if EMBEDDING_CACHE_PATH.exists():
    try:
        embedding_cache = joblib.load(EMBEDDING_CACHE_PATH)
    except Exception:
        embedding_cache = {}

if USE_SENTENCE_EMBEDDING:
    try:
        embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    except Exception as e:
        embedding_model = None
        USE_SENTENCE_EMBEDDING = False
        print("Cannot load embedding model. Fallback features only.")
        print("Reason:", repr(e))


def get_embedding(text):
    text = norm_text(text)
    if not text or embedding_model is None:
        return None
    if text in embedding_cache:
        return embedding_cache[text]
    vec = embedding_model.encode([text], normalize_embeddings=True)[0]
    embedding_cache[text] = vec
    return vec


def embedding_similarity(a, b):
    if not USE_SENTENCE_EMBEDDING or embedding_model is None:
        return 0.0
    va = get_embedding(a)
    vb = get_embedding(b)
    if va is None or vb is None:
        return 0.0
    try:
        return float(np.dot(va, vb))
    except Exception:
        return 0.0


def price_score(tb_price, kh_price):
    a = clean_number(tb_price)
    b = clean_number(kh_price)
    if a is None or b is None or a <= 0 or b <= 0:
        return -1.0
    diff = abs(a - b) / max(a, b)
    if diff <= 0.001:
        return 100.0
    return max(0.0, 100.0 * (1.0 - diff))


def version_score(tb_code_or_version, kh_code_or_version):
    tv = get_version_num(tb_code_or_version)
    kv = get_version_num(kh_code_or_version)
    if tv is None or kv is None:
        return 50.0
    diff = abs(tv - kv)
    if diff == 0:
        return 100.0
    if diff == 1:
        return 85.0
    if diff == 2:
        return 60.0
    return 30.0


def bool_to_float(v):
    s = clean_value(v).lower()
    return 1.0 if s in ["true", "1", "yes", "x", "✓", "co", "có"] else 0.0


def build_features(tb_row, kh_row):
    tb_goi = clean_value(tb_row.get(C_TBMT_GOI, ""))
    kh_goi = clean_value(kh_row.get(C_KH_GOI, "")) if C_KH_GOI else ""
    tb_du_an = clean_value(tb_row.get(C_TBMT_DU_AN, "")) if C_TBMT_DU_AN else ""
    kh_du_an = clean_value(kh_row.get(C_KH_DU_AN, "")) if C_KH_DU_AN else ""
    tb_cdt = clean_value(tb_row.get(C_TBMT_CDT, "")) or clean_value(tb_row.get(C_TBMT_DVMT, ""))
    kh_cdt = clean_value(kh_row.get(C_KH_CDT, "")) if C_KH_CDT else ""
    tb_gia = tb_row.get(C_TBMT_GIA, "") if C_TBMT_GIA else ""
    kh_gia = kh_row.get(C_KH_GIA, "") if C_KH_GIA else ""
    tb_ver = tb_row.get(C_TBMT_VERSION, "") if C_TBMT_VERSION else tb_row.get(C_TBMT_SO, "")
    kh_ver = kh_row.get(C_KH_VERSION, "") if C_KH_VERSION else kh_row.get(C_KH_SO, "")

    goi_fuzzy = fuzz.token_set_ratio(norm_text(tb_goi), norm_text(kh_goi)) if tb_goi and kh_goi else 0.0
    du_an_fuzzy = fuzz.token_set_ratio(norm_text(tb_du_an), norm_text(kh_du_an)) if tb_du_an and kh_du_an else 0.0
    cdt_fuzzy = fuzz.token_set_ratio(norm_text(tb_cdt), norm_text(kh_cdt)) if tb_cdt and kh_cdt else 0.0
    ps = price_score(tb_gia, kh_gia)
    vs = version_score(tb_ver, kh_ver)
    emb_goi = embedding_similarity(tb_goi, kh_goi)
    emb_du_an = embedding_similarity(tb_du_an, kh_du_an)
    char_sim = char_ngram_similarity(tb_goi, kh_goi)
    key_overlap = keyword_overlap(tb_goi, kh_goi)
    jac = jaccard(tb_goi, kh_goi)
    len_tb = max(1, len(norm_text(tb_goi)))
    len_kh = max(1, len(norm_text(kh_goi)))
    length_ratio = min(len_tb, len_kh) / max(len_tb, len_kh)
    is_latest = bool_to_float(kh_row.get(C_KH_LATEST, "")) if C_KH_LATEST else 0.0

    rule_score = (
        0.45 * goi_fuzzy
        + 0.15 * max(ps, 0)
        + 0.15 * cdt_fuzzy
        + 0.10 * vs
        + 0.10 * (emb_goi * 100.0)
        + 0.05 * du_an_fuzzy
    )

    return {
        "goi_fuzzy_score": float(goi_fuzzy),
        "du_an_fuzzy_score": float(du_an_fuzzy),
        "cdt_fuzzy_score": float(cdt_fuzzy),
        "gia_score": float(ps),
        "version_score": float(vs),
        "embedding_goi_similarity": float(emb_goi),
        "embedding_du_an_similarity": float(emb_du_an),
        "char_ngram_goi_similarity": float(char_sim),
        "keyword_overlap_ratio": float(key_overlap),
        "keyword_jaccard": float(jac),
        "length_ratio": float(length_ratio),
        "is_latest_khlcnt": float(is_latest),
        "rule_score": float(rule_score),
    }

FEATURE_COLUMNS = [
    "goi_fuzzy_score", "du_an_fuzzy_score", "cdt_fuzzy_score", "gia_score",
    "version_score", "embedding_goi_similarity", "embedding_du_an_similarity",
    "char_ngram_goi_similarity", "keyword_overlap_ratio", "keyword_jaccard",
    "length_ratio", "is_latest_khlcnt", "rule_score",
]

# ============================================================
# 7. BUILD / UPDATE TRAINING DATA FROM MANUAL + DIRECT SCAN + MAPPING
# ============================================================

TRAINING_COLUMNS_BASE = ["Số TBMT", "Số KHLCNT candidate", "label", "source", "created_at"]

df_training_old = read_excel_sheet(TRAINING_DATA_PATH)
if df_training_old.empty:
    df_training_old = pd.DataFrame(columns=TRAINING_COLUMNS_BASE + FEATURE_COLUMNS)

training_rows = []

# Load match candidates to create negative examples and future predictions.
df_candidates = read_excel_sheet(MATCH_CANDIDATES_PATH)
if df_candidates.empty:
    print("Không thấy Match_Candidates.xlsx hoặc file rỗng. Model vẫn học positive manual labels nếu có, nhưng prediction sẽ hạn chế.")

cand_tbmt_col = find_col_soft(df_candidates, ["Số TBMT", "So TBMT", "TBMT"], required=False) if not df_candidates.empty else None
cand_kh_col = find_col_soft(df_candidates, ["Số KHLCNT", "Số KHLCNT ứng viên", "So KHLCNT", "Candidate", "Candidate 1"], required=False) if not df_candidates.empty else None


# ------------------------------------------------------------
# 7.1. Gom tất cả mapping đã xác nhận đúng 100%
# Nguồn 1: df_mapping
# Nguồn 2: df_memory gồm manual input + direct scan
# ------------------------------------------------------------

confirmed_pairs = []

# Nguồn 1: mapping đã confirm
if not df_mapping.empty:
    for _, m in df_mapping.iterrows():
        so_tbmt = normalize_tbmt_code(m.get("Số TBMT", ""))
        so_kh = normalize_khlcnt_code(m.get("Số KHLCNT", ""))

        if so_tbmt and so_kh:
            confirmed_pairs.append({
                "Số TBMT": so_tbmt,
                "Số KHLCNT": so_kh,
                "source": clean_value(m.get("Source", "")) or "confirmed_mapping"
            })

# Nguồn 2: memory/manual/direct scan
if not df_memory.empty:
    for _, r in df_memory.iterrows():
        so_tbmt = normalize_tbmt_code(r.get("Số TBMT", ""))
        so_kh = normalize_khlcnt_code(
            r.get("Số KHLCNT chuẩn hóa", "") or r.get("Số KHLCNT người nhập", "")
        )

        if not so_tbmt or not so_kh:
            continue

        if not is_valid_khlcnt_format(so_kh):
            continue

        src = clean_value(r.get("Source", ""))

        if "direct scan" in src.lower():
            pair_source = "direct_scan_label_1"
        else:
            pair_source = "manual_input_label_1"

        confirmed_pairs.append({
            "Số TBMT": so_tbmt,
            "Số KHLCNT": so_kh,
            "source": pair_source
        })
# Nguồn 3: những dòng đã có Số KHLCNT trong file tổng
# Chỉ lấy làm label=1 nếu đã có mã KHLCNT và nguồn match đủ tin cậy.

auto_master_pairs = 0

if not df_tbmt_master.empty and C_TBMT_KHLCNT in df_tbmt_master.columns:
    for _, r in df_tbmt_master.iterrows():
        so_tbmt = normalize_tbmt_code(r.get(C_TBMT_SO, ""))
        so_kh = normalize_khlcnt_code(r.get(C_TBMT_KHLCNT, ""))

        if not so_tbmt or not so_kh:
            continue

        if not is_valid_khlcnt_format(so_kh):
            continue

        match_status = clean_value(r.get("Trạng thái match KHLCNT", ""))
        match_source = clean_value(r.get("Nguồn match KHLCNT", ""))
        match_score = clean_number(r.get("Điểm match KHLCNT", ""))

        status_text = (match_status + " " + match_source).lower()

        # Chỉ lấy những case đáng tin:
        # - Manual confirmed
        # - Direct scan
        # - Rule/Auto match có điểm 100
        is_trusted_auto_match = (
            "manual" in status_text
            or "direct" in status_text
            or "scan" in status_text
            or (
                match_score is not None
                and match_score >= 100
            )
        )

        if not is_trusted_auto_match:
            continue

        confirmed_pairs.append({
            "Số TBMT": so_tbmt,
            "Số KHLCNT": so_kh,
            "source": "master_auto_matched_label_1"
        })

        auto_master_pairs += 1

print("Auto/master matched pairs added:", auto_master_pairs)
df_confirmed_pairs = pd.DataFrame(confirmed_pairs)

if df_confirmed_pairs.empty:
    print("Không có confirmed manual/direct scan pairs để tạo training data.")
else:
    df_confirmed_pairs["Số TBMT"] = df_confirmed_pairs["Số TBMT"].apply(normalize_tbmt_code)
    df_confirmed_pairs["Số KHLCNT"] = df_confirmed_pairs["Số KHLCNT"].apply(normalize_khlcnt_code)
    df_confirmed_pairs = df_confirmed_pairs.drop_duplicates(
        subset=["Số TBMT", "Số KHLCNT"],
        keep="last"
    ).reset_index(drop=True)

print("Confirmed manual/direct scan pairs:", len(df_confirmed_pairs))


# ------------------------------------------------------------
# 7.2. Tạo positive + negative training rows
# Positive: TBMT + KHLCNT người nhập/direct scan = 1
# Negative: TBMT đó + KHLCNT khác = 0
# ------------------------------------------------------------

skipped_not_in_tbmt_master = 0
skipped_not_in_kh_master = 0

for _, m in df_confirmed_pairs.iterrows():
    so_tbmt = normalize_tbmt_code(m.get("Số TBMT", ""))
    so_kh_true = normalize_khlcnt_code(m.get("Số KHLCNT", ""))

    # TBMT phải có trong master để lấy feature
    if so_tbmt not in tbmt_by_code:
        skipped_not_in_tbmt_master += 1
        continue

    # KHLCNT phải có trong master để lấy feature KHLCNT
    # Nếu chưa có thì lưu memory, chưa train được vì chưa có dòng KHLCNT để so sánh đặc trưng.
    if so_kh_true not in kh_by_code:
        skipped_not_in_kh_master += 1
        continue

    tb_row = tbmt_by_code[so_tbmt]
    kh_true_row = kh_by_code[so_kh_true]

    # Positive label = 1
    feats = build_features(tb_row, kh_true_row)
    training_rows.append({
        "Số TBMT": so_tbmt,
        "Số KHLCNT candidate": so_kh_true,
        "label": 1,
        "source": clean_value(m.get("source", "")) or "manual_or_direct_label_1",
        "created_at": now_str(),
        **feats,
    })

    # Negative samples: KHLCNT ứng viên khác với KHLCNT đúng
    neg_codes = []

    # Ưu tiên lấy negative từ Match_Candidates cùng TBMT
    if not df_candidates.empty and cand_tbmt_col and cand_kh_col:
        tmp = df_candidates[
            df_candidates[cand_tbmt_col].astype(str).apply(normalize_tbmt_code) == so_tbmt
        ].copy()

        for _, cr in tmp.iterrows():
            ck = normalize_khlcnt_code(cr.get(cand_kh_col, ""))
            if ck and ck != so_kh_true and ck in kh_by_code and ck not in neg_codes:
                neg_codes.append(ck)

            if len(neg_codes) >= MAX_NEGATIVES_PER_POSITIVE:
                break

    # Nếu không có candidate log, lấy vài KHLCNT khác làm negative yếu
    if not neg_codes:
        for ck in list(kh_by_code.keys()):
            ck = normalize_khlcnt_code(ck)
            if ck and ck != so_kh_true and ck in kh_by_code:
                neg_codes.append(ck)

            if len(neg_codes) >= min(3, MAX_NEGATIVES_PER_POSITIVE):
                break

    for ck in neg_codes[:MAX_NEGATIVES_PER_POSITIVE]:
        feats_neg = build_features(tb_row, kh_by_code[ck])
        training_rows.append({
            "Số TBMT": so_tbmt,
            "Số KHLCNT candidate": ck,
            "label": 0,
            "source": "auto_negative_from_confirmed_manual_or_scan",
            "created_at": now_str(),
            **feats_neg,
        })


print("Skipped confirmed pairs not in TBMT master:", skipped_not_in_tbmt_master)
print("Skipped confirmed pairs not in KHLCNT master:", skipped_not_in_kh_master)


# ------------------------------------------------------------
# 7.3. Gộp với training cũ và lưu lại
# ------------------------------------------------------------

if training_rows:
    df_training_new = pd.DataFrame(training_rows)
    df_training = pd.concat([df_training_old, df_training_new], ignore_index=True, sort=False)
else:
    df_training = df_training_old.copy()

for col in TRAINING_COLUMNS_BASE + FEATURE_COLUMNS:
    if col not in df_training.columns:
        df_training[col] = ""

df_training["Số TBMT"] = df_training["Số TBMT"].apply(normalize_tbmt_code)
df_training["Số KHLCNT candidate"] = df_training["Số KHLCNT candidate"].apply(normalize_khlcnt_code)
df_training["label"] = pd.to_numeric(df_training["label"], errors="coerce").fillna(0).astype(int)

# Một cặp TBMT-KHLCNT chỉ nên giữ 1 label mới nhất.
# Nếu trước đó từng là negative mà sau này người nhập confirm đúng, label=1 sẽ ghi đè.
df_training = df_training.drop_duplicates(
    subset=["Số TBMT", "Số KHLCNT candidate"],
    keep="last"
).reset_index(drop=True)

write_excel_text_safe(TRAINING_DATA_PATH, {"Training_Data": df_training})

try:
    joblib.dump(embedding_cache, EMBEDDING_CACHE_PATH.with_suffix(".tmp.joblib"))
    EMBEDDING_CACHE_PATH.with_suffix(".tmp.joblib").replace(EMBEDDING_CACHE_PATH)
except Exception:
    pass

pos_now = int((df_training["label"] == 1).sum()) if not df_training.empty else 0
neg_now = int((df_training["label"] == 0).sum()) if not df_training.empty else 0

print("Training data saved:", TRAINING_DATA_PATH)
print("Training rows:", len(df_training))
print("Positive:", pos_now)
print("Negative:", neg_now)

save_state("updated_training_data", {
    "training_rows": len(df_training),
    "positive": pos_now,
    "negative": neg_now,
    "confirmed_pairs": len(df_confirmed_pairs),
    "skipped_not_in_tbmt_master": skipped_not_in_tbmt_master,
    "skipped_not_in_kh_master": skipped_not_in_kh_master,
})
# ============================================================
# 8. TRAIN XGBOOST MODEL IF ENOUGH DATA
# ============================================================

model = None
metrics_rows = []

pos_count = int((df_training["label"] == 1).sum()) if not df_training.empty else 0
neg_count = int((df_training["label"] == 0).sum()) if not df_training.empty else 0

if pos_count >= MIN_POSITIVE_LABELS_TO_TRAIN and neg_count >= MIN_NEGATIVE_LABELS_TO_TRAIN:
    train_df = df_training.copy()
    for col in FEATURE_COLUMNS:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce").fillna(0.0)
    X = train_df[FEATURE_COLUMNS].astype(float)
    y = train_df["label"].astype(int)

    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
    except Exception:
        X_train, X_test, y_train, y_test = X, X, y, y

    scale_pos_weight = max(1.0, (len(y_train) - y_train.sum()) / max(1, y_train.sum()))
    model = XGBClassifier(
        n_estimators=180,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        scale_pos_weight=scale_pos_weight,
    )
    model.fit(X_train, y_train)

    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    try:
        auc = roc_auc_score(y_test, prob) if len(set(y_test)) > 1 else ""
    except Exception:
        auc = ""
    metrics_rows.append({
        "trained_at": now_str(),
        "rows": len(train_df),
        "positive": pos_count,
        "negative": neg_count,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "auc": auc,
    })
    tmp_model = MODEL_PATH.with_suffix(".tmp.joblib")
    joblib.dump({"model": model, "features": FEATURE_COLUMNS, "trained_at": now_str()}, tmp_model)
    tmp_model.replace(MODEL_PATH)
    write_excel_text_safe(METRICS_PATH, {"Metrics": pd.DataFrame(metrics_rows)})
    print("XGBoost trained:", MODEL_PATH)
else:
    if MODEL_PATH.exists():
        try:
            saved = joblib.load(MODEL_PATH)
            model = saved["model"] if isinstance(saved, dict) and "model" in saved else saved
            print("Loaded existing model:", MODEL_PATH)
        except Exception:
            model = None
    print(f"XGBoost chưa đủ dữ liệu train mới. Positive={pos_count}, Negative={neg_count}")

save_state("trained_or_loaded_model", {"has_model": model is not None, "positive": pos_count, "negative": neg_count})

# ============================================================
# 9. MODEL PREDICTION + AUTO APPLY HIGH CONFIDENCE
# ============================================================

prediction_rows = []
model_mapping_rows = []

# Reload after manual apply, so model does not overwrite already-filled TBMT.
master_sheets = read_all_sheets(MASTER_OUTPUT_PATH)
df_tbmt_master = fix_code_text_columns(master_sheets[TBMT_SHEET])
df_tbmt_import = fix_code_text_columns(read_excel_sheet(TBMT_IMPORT_PATH, TBMT_SHEET))
df_tbmt_master[C_TBMT_SO] = df_tbmt_master[C_TBMT_SO].apply(normalize_tbmt_code)
df_tbmt_import[C_TBMT_SO] = df_tbmt_import[C_TBMT_SO].apply(normalize_tbmt_code)

tbmt_by_code = {str(row[C_TBMT_SO]).strip(): row for _, row in df_tbmt_master.iterrows() if str(row[C_TBMT_SO]).strip()}
memory_has_khlcnt_tbmt = set()

if not df_memory.empty and "Số TBMT" in df_memory.columns:
    tmp_mem = df_memory.copy()
    tmp_mem["Số TBMT"] = tmp_mem["Số TBMT"].fillna("").astype(str).apply(normalize_tbmt_code)

    if "Số KHLCNT chuẩn hóa" in tmp_mem.columns:
        tmp_mem["_kh"] = tmp_mem["Số KHLCNT chuẩn hóa"].fillna("").astype(str).str.strip()
    else:
        tmp_mem["_kh"] = ""

    memory_has_khlcnt_tbmt = set(
        tmp_mem.loc[tmp_mem["_kh"] != "", "Số TBMT"].tolist()
    )

print("TBMT có mã KHLCNT trong memory, bỏ qua model prediction:", len(memory_has_khlcnt_tbmt))
if model is not None and not df_candidates.empty and cand_tbmt_col and cand_kh_col:
    # Build candidate feature matrix from Match_Candidates.xlsx.
    candidate_feature_rows = []
    for _, cr in df_candidates.iterrows():
        so_tbmt = normalize_tbmt_code(cr.get(cand_tbmt_col, ""))
        so_kh = normalize_khlcnt_code(cr.get(cand_kh_col, ""))
        if not so_tbmt or not so_kh:
            continue
        if so_tbmt not in tbmt_by_code or so_kh not in kh_by_code:
            continue
        tb_existing = clean_value(tbmt_by_code[so_tbmt].get(C_TBMT_KHLCNT, ""))

        if tb_existing or so_tbmt in memory_has_khlcnt_tbmt:
            continue
        feats = build_features(tbmt_by_code[so_tbmt], kh_by_code[so_kh])
        candidate_feature_rows.append({
            "Số TBMT": so_tbmt,
            "Số KHLCNT candidate": so_kh,
            **feats,
        })

    if candidate_feature_rows:
        df_pred = pd.DataFrame(candidate_feature_rows)
        for col in FEATURE_COLUMNS:
            df_pred[col] = pd.to_numeric(df_pred[col], errors="coerce").fillna(0.0)
        probs = model.predict_proba(df_pred[FEATURE_COLUMNS].astype(float))[:, 1]
        df_pred["model_probability"] = probs
        df_pred = df_pred.sort_values(["Số TBMT", "model_probability"], ascending=[True, False])

        for so_tbmt, grp in df_pred.groupby("Số TBMT"):
            grp = grp.sort_values("model_probability", ascending=False).reset_index(drop=True)
            top1 = grp.iloc[0]
            top2_prob = float(grp.iloc[1]["model_probability"]) if len(grp) > 1 else 0.0
            prob1 = float(top1["model_probability"])
            margin = prob1 - top2_prob
            so_kh = normalize_khlcnt_code(top1["Số KHLCNT candidate"])
            decision = "Review"
            if MODEL_AUTO_APPLY and prob1 >= MODEL_PROB_THRESHOLD and margin >= MODEL_MARGIN_THRESHOLD and so_kh in kh_codes_set:
                decision = "Model auto"
                model_mapping_rows.append({
                    "Số TBMT": so_tbmt,
                    "Số KHLCNT": so_kh,
                    "Mã gốc KHLCNT": kh_root(so_kh),
                    "Phiên bản KHLCNT": fix_version_value(get_version_num(so_kh)),
                    "Confirmed at": now_str(),
                    "Applied to master at": "",
                    "Applied to import at": "",
                    "Source": "XGBoost auto prediction",
                    "Ghi chú": f"Model probability={prob1:.4f}, margin={margin:.4f}",
                })
            prediction_rows.append({
                "Số TBMT": so_tbmt,
                "Số KHLCNT candidate": so_kh,
                "model_probability": prob1,
                "top2_probability": top2_prob,
                "margin": margin,
                "decision": decision,
                "generated_at": now_str(),
            })

if prediction_rows:
    write_excel_text_safe(PREDICTIONS_PATH, {"Predictions": pd.DataFrame(prediction_rows)})
else:
    write_excel_text_safe(PREDICTIONS_PATH, {"Predictions": pd.DataFrame(columns=["Số TBMT", "Số KHLCNT candidate", "model_probability", "decision"])})

# Apply model mappings if any.
if model_mapping_rows:
    df_model_mapping = pd.DataFrame(model_mapping_rows)
    df_tbmt_master, applied_master_model = apply_mapping_to_df(df_tbmt_master, df_model_mapping, "XGBoost model")
    df_tbmt_import, applied_import_model = apply_mapping_to_df(df_tbmt_import, df_model_mapping, "XGBoost model")
    master_sheets[TBMT_SHEET] = df_tbmt_master
    write_excel_text_safe(MASTER_OUTPUT_PATH, master_sheets)
    write_excel_text_safe(TBMT_IMPORT_PATH, {TBMT_SHEET: df_tbmt_import})
    # Save model-applied mapping into mapping memory too.
    df_mapping = pd.concat([df_mapping, df_model_mapping], ignore_index=True, sort=False)
    df_mapping = df_mapping.drop_duplicates(subset=["Số TBMT"], keep="last").reset_index(drop=True)
    write_excel_text_safe(MAPPING_PATH, {"Mapping": df_mapping})
    print("Model auto-applied rows:", len(model_mapping_rows))

save_state("completed_model_prediction", {"prediction_rows": len(prediction_rows), "model_auto_applied": len(model_mapping_rows)})

# ============================================================
# 10. REGENERATE MANUAL INPUT FILE - ONLY UNMAPPED / PENDING
# ============================================================

# Reload final state after all applies.
master_sheets = read_all_sheets(MASTER_OUTPUT_PATH)
df_tbmt_master = fix_code_text_columns(master_sheets[TBMT_SHEET])
df_tbmt_master[C_TBMT_SO] = df_tbmt_master[C_TBMT_SO].apply(normalize_tbmt_code)

mapped_tbmt = set()
if C_TBMT_KHLCNT in df_tbmt_master.columns:
    mapped_tbmt.update(
        df_tbmt_master.loc[
            df_tbmt_master[C_TBMT_KHLCNT].astype(str).str.strip().replace({"nan": "", "None": "", "NaN": ""}).ne(""),
            C_TBMT_SO
        ].astype(str).tolist()
    )
# Also count confirmed manual/model mapping as mapped if KHLCNT exists.
for _, m in df_mapping.iterrows():
    so_tbmt = normalize_tbmt_code(m.get("Số TBMT", ""))
    so_kh = normalize_khlcnt_code(m.get("Số KHLCNT", ""))
    if so_tbmt and so_kh in kh_codes_set:
        mapped_tbmt.add(so_tbmt)

# ============================================================
# REGENERATE MANUAL INPUT FILE
# Chỉ hiện TBMT chưa map và chưa từng được người nhập mã KHLCNT
# ============================================================

# TBMT đã có người nhập mã KHLCNT thì KHÔNG hiện lại trong file nhập tay.
# Lấy từ 2 nguồn:
# 1) df_memory đã lưu
# 2) Manual_KHLCNT_Code_To_Fill.xlsx hiện tại người dùng vừa nhập
memory_entered_tbmt = set()

# Nguồn 1: từ memory đã lưu
for _, r in df_memory.iterrows():
    so_tbmt = normalize_tbmt_code(r.get("Số TBMT", ""))
    kh_input = clean_value(r.get("Số KHLCNT người nhập", ""))
    kh_norm = normalize_khlcnt_code(r.get("Số KHLCNT chuẩn hóa", ""))

    if so_tbmt and (kh_input or kh_norm):
        memory_entered_tbmt.add(so_tbmt)

# Nguồn 2: đọc lại trực tiếp file manual hiện tại
# để chắc chắn không mất những dòng người nhập vừa điền
df_manual_check = read_excel_sheet(MANUAL_FILL_PATH)

if not df_manual_check.empty:
    c_check_tbmt = find_col_soft(
        df_manual_check,
        ["Số TBMT", "So TBMT"],
        required=False
    )
    c_check_kh = find_col_soft(
        df_manual_check,
        ["Số KHLCNT người nhập", "So KHLCNT nguoi nhap", "Số KHLCNT nguoi nhap"],
        required=False
    )

    print("Manual file rows before regenerate:", len(df_manual_check))
    print("Detected manual TBMT col:", c_check_tbmt)
    print("Detected manual KHLCNT input col:", c_check_kh)

    if c_check_tbmt and c_check_kh:
        filled_count = (
            df_manual_check[c_check_kh]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .sum()
        )
        print("Manual filled KHLCNT rows before regenerate:", filled_count)

        for _, r in df_manual_check.iterrows():
            so_tbmt = normalize_tbmt_code(r.get(c_check_tbmt, ""))
            kh_input = clean_value(r.get(c_check_kh, ""))

            if so_tbmt and kh_input:
                memory_entered_tbmt.add(so_tbmt)
    else:
        print("Không tìm thấy đúng cột Số TBMT hoặc Số KHLCNT người nhập trong manual file.")

print("TBMT đã có mã KHLCNT người nhập trong memory/manual:", len(memory_entered_tbmt))

manual_rows = []

for _, row in df_tbmt_master.iterrows():
    so_tbmt = normalize_tbmt_code(row.get(C_TBMT_SO, ""))

    if not so_tbmt:
        continue

    # Nếu TBMT đã map rồi thì không hiện.
    if so_tbmt in mapped_tbmt:
        continue

    # Nếu TBMT đã có người nhập mã KHLCNT rồi thì không hiện lại.
    # Dù KHLCNT chưa có trong KHLCNT tổng, nó vẫn nằm trong memory/manual
    # và các lần sau Cell 4 sẽ tự kiểm tra lại.
    if so_tbmt in memory_entered_tbmt:
        continue

    manual_rows.append({
        "Số TBMT": so_tbmt,
         "Số KHLCNT người nhập": "",
        "Tên gói thầu": clean_value(row.get(C_TBMT_GOI, "")) if C_TBMT_GOI else "",
        "Dự án": clean_value(row.get(C_TBMT_DU_AN, "")) if C_TBMT_DU_AN else "",
        "Chủ đầu tư": clean_value(row.get(C_TBMT_CDT, "")) if C_TBMT_CDT else clean_value(row.get(C_TBMT_DVMT, "")),
        "Giá gói thầu": clean_value(row.get(C_TBMT_GIA, "")) if C_TBMT_GIA else "",

        "Ghi chú": "Người nhập copy mã KHLCNT full từ web vào cột 'Số KHLCNT người nhập'",
    })
df_manual_out = pd.DataFrame(manual_rows)

MANUAL_INPUT_COLUMNS = [
    "Số TBMT",
    "Số KHLCNT người nhập",
    "Tên gói thầu",
    "Dự án",
    "Chủ đầu tư",
    "Giá gói thầu",
    "Ghi chú",
]

for col in MANUAL_INPUT_COLUMNS:
    if col not in df_manual_out.columns:
        df_manual_out[col] = ""

df_manual_out = df_manual_out[MANUAL_INPUT_COLUMNS]
write_excel_text_safe(MANUAL_FILL_PATH, {"Manual_Input": df_manual_out})

# Save compact checkpoint snapshot.
write_excel_text_safe(PIPELINE_CHECKPOINT_PATH, {
    "Manual_Pending": df_manual_out,
    "Memory": df_memory,
    "Mapping": df_mapping,
    "Training_Summary": pd.DataFrame([{
        "training_rows": len(df_training),
        "positive": pos_count,
        "negative": neg_count,
        "has_model": model is not None,
        "generated_at": now_str(),
    }]),
})

save_state("completed", {
    "manual_pending_rows": len(df_manual_out),
    "memory_rows": len(df_memory),
    "mapping_rows": len(df_mapping),
    "training_rows": len(df_training),
    "positive": pos_count,
    "negative": neg_count,
})

print("\nDONE CELL 4")
print("Manual file for input:", MANUAL_FILL_PATH)
print("Memory:", MEMORY_PATH)
print("Mapping:", MAPPING_PATH)
print("Training data:", TRAINING_DATA_PATH)
print("Predictions:", PREDICTIONS_PATH)
print("Master updated:", MASTER_OUTPUT_PATH)
print("TBMT import updated:", TBMT_IMPORT_PATH)
print("Pending manual rows:", len(df_manual_out))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MASTER_OUTPUT_PATH: /content/drive/MyDrive/Zoho_Procurement/output_excel/Zoho_PDF_Import_Output_LATEST.xlsx
TBMT_IMPORT_PATH: /content/drive/MyDrive/Zoho_Procurement/output_excel/TBMT_for_import.xlsx
MATCH_CANDIDATES_PATH: /content/drive/MyDrive/Zoho_Procurement/logs/Match_Candidates.xlsx
MODEL_WORK_DIR: /content/drive/MyDrive/Zoho_Procurement/output_excel/manual_khlcnt_model
MANUAL_FILL_PATH: /content/drive/MyDrive/Zoho_Procurement/output_excel/manual_khlcnt_model/Manual_KHLCNT_Code_To_Fill.xlsx
Imported direct scan memory to model memory: 0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Auto/master matched pairs added: 1886
Confirmed manual/direct scan pairs: 1965
Skipped confirmed pairs not in TBMT master: 0
Skipped confirmed pairs not in KHLCNT master: 76
Training data saved: /content/drive/MyDrive/Zoho_Procurement/output_excel/manual_khlcnt_model/match_training_data.xlsx
Training rows: 5577
Positive: 1888
Negative: 3689
XGBoost trained: /content/drive/MyDrive/Zoho_Procurement/output_excel/manual_khlcnt_model/models/tbmt_khlcnt_xgboost_model.joblib
TBMT có mã KHLCNT trong memory, bỏ qua model prediction: 168
Model auto-applied rows: 539
Manual file rows before regenerate: 910
Detected manual TBMT col: Số TBMT
Detected manual KHLCNT input col: Số KHLCNT người nhập
Manual filled KHLCNT rows before regenerate: 0
TBMT đã có mã KHLCNT người nhập trong memory/manual: 168

DONE CELL 4
Manual file for input: /content/drive/MyDrive/Zoho_Procurement/output_excel/manual_khlcnt_model/Manual_KHLCNT_Code_To_Fill.xlsx
Memory: /content/drive/MyDrive/Zoho_Procurement/output_excel/ma

In [ ]:
# ============================================================
# MATCH ONLY CELL - RERUN TBMT ↔ KHLCNT MATCH
# Không parse PDF lại, chỉ đọc output_excel đã có rồi match lại
# ============================================================

import re
import sys
import subprocess
from pathlib import Path
from datetime import datetime

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pip_name])

ensure_package("pandas")
ensure_package("openpyxl")
ensure_package("rapidfuzz")
ensure_package("unidecode")
ensure_package("tqdm")

import pandas as pd
import numpy as np
from rapidfuzz import fuzz
from unidecode import unidecode
from tqdm import tqdm
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

# ============================================================
# OPTIONAL GPU PREFILTER
# GPU chỉ dùng để lọc top-K KHLCNT candidate trước,
# sau đó vẫn dùng ratio/fuzzy score để quyết định match.
# ============================================================

USE_GPU_EMBED_PREFILTER = True
EMBED_TOP_K = 150
EMBED_BATCH_SIZE = 128

try:
    ensure_package("torch")
    ensure_package("sentence_transformers", "sentence-transformers")
    import torch
    from sentence_transformers import SentenceTransformer

    GPU_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("GPU_DEVICE:", GPU_DEVICE)
    if GPU_DEVICE == "cuda":
        print("GPU name:", torch.cuda.get_device_name(0))
except Exception as e:
    USE_GPU_EMBED_PREFILTER = False
    torch = None
    SentenceTransformer = None
    GPU_DEVICE = "cpu"
    print("Không bật được GPU embedding prefilter, sẽ match CPU.")
    print("Reason:", repr(e))


# ============================================================
# PATHS
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

BASE_DIR = Path("/content/drive/MyDrive/Zoho_Procurement")
OUTPUT_DIR = BASE_DIR / "output_excel"
LOG_DIR = BASE_DIR / "logs"

MASTER_OUTPUT_PATH = OUTPUT_DIR / "Zoho_PDF_Import_Output_LATEST.xlsx"
TBMT_IMPORT_PATH = OUTPUT_DIR / "TBMT_for_import.xlsx"
KHLCNT_ITEMS_MATCH_PATH = OUTPUT_DIR / "KHLCNT_items_for_match.xlsx"
MATCH_CANDIDATES_PATH = LOG_DIR / "Match_Candidates.xlsx"
MATCH_CANDIDATES_HISTORY_PATH = LOG_DIR / "Match_Candidates_History.xlsx"

TBMT_SHEET = "TBMT_Zoho_Import"
KHLCNT_SHEET = "KHLCNT_Zoho_Import"
KHLCNT_ITEMS_SHEET = "KHLCNT_Items_Match"

# Threshold
AUTO_MATCH_SCORE = 88
REVIEW_MATCH_SCORE = 72
MIN_MARGIN_FOR_AUTO = 4

ALLOW_OVERWRITE_EXISTING = False

print("MASTER_OUTPUT_PATH:", MASTER_OUTPUT_PATH)
print("TBMT_IMPORT_PATH:", TBMT_IMPORT_PATH)
print("KHLCNT_ITEMS_MATCH_PATH:", KHLCNT_ITEMS_MATCH_PATH)
print("MATCH_CANDIDATES_PATH:", MATCH_CANDIDATES_PATH)


# ============================================================
# HELPERS
# ============================================================

TEXT_SAFE_COLUMNS = [
    "Số TBMT", "Mã gốc TBMT", "Mã Phiên Bản", "Phiên Bản Trước dự kiến",
    "Số KHLCNT", "Số KHLCNT scan từ PDF", "Số KHLCNT ứng viên",
    "Mã gốc KHLCNT", "Phiên bản KHLCNT",
    "Mã TBMT match mới nhất", "Mã TBMT mới nhất", "TBMT mới nhất",
    "KHLCNT Item Key", "KHLCNT Item Key match",
    "Phiên bản KHLCNT match",
]

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def is_blank(v):
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except Exception:
        pass
    return str(v).strip() == "" or str(v).strip().lower() in ["nan", "none", "null"]

def clean_value(v):
    if is_blank(v):
        return ""
    return str(v).replace("\xa0", " ").strip()

def clean_text(v):
    s = clean_value(v)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def normalize_key(v):
    s = clean_text(v).lower()
    s = unidecode(s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_tbmt_code(v):
    s = clean_value(v).upper().replace(" ", "")
    m = re.search(r"(IB\d+)\s*-?\s*(\d{2})", s)
    if m:
        return f"{m.group(1)}-{m.group(2)}"
    return s

def normalize_khlcnt_code(v):
    s = clean_value(v).upper().replace(" ", "")
    m = re.search(r"(PL\d+)\s*-?\s*(\d{2})", s)
    if m:
        return f"{m.group(1)}-{m.group(2)}"
    return s

def fix_version_value(v):
    if is_blank(v):
        return ""
    s = str(v).strip()
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    if re.fullmatch(r"\d+", s):
        return s.zfill(2)
    return s

def fix_text_value(v):
    if is_blank(v):
        return ""
    s = str(v).strip()
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    return s

def fix_code_text_columns(df):
    if df is None or df.empty:
        return df
    df = df.copy()
    for col in ["Mã Phiên Bản", "Phiên bản KHLCNT", "Phiên bản KHLCNT match"]:
        if col in df.columns:
            df[col] = df[col].apply(fix_version_value).astype(str)
    for col in TEXT_SAFE_COLUMNS:
        if col in df.columns:
            df[col] = df[col].apply(fix_text_value).astype(str)
    return df

def clean_number(v):
    if is_blank(v):
        return None
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        try:
            if pd.isna(v):
                return None
        except Exception:
            pass
        return float(v)

    s = str(v).strip()
    s = re.sub(r"[^\d,.\-]", "", s)
    if s in ["", "-", ".", ","]:
        return None

    try:
        if "." in s and "," in s:
            if s.rfind(",") > s.rfind("."):
                s = s.replace(".", "").replace(",", ".")
            else:
                s = s.replace(",", "")
        elif "," in s:
            parts = s.split(",")
            if len(parts) == 2 and len(parts[1]) in [1, 2]:
                s = parts[0].replace(".", "") + "." + parts[1]
            else:
                s = s.replace(",", "")
        elif "." in s:
            parts = s.split(".")
            if len(parts) > 2:
                s = "".join(parts)
            elif len(parts) == 2 and len(parts[1]) == 3:
                s = s.replace(".", "")
        return float(s)
    except Exception:
        return None

def price_score(a, b):
    x = clean_number(a)
    y = clean_number(b)
    if x is None or y is None or x <= 0 or y <= 0:
        return -1.0
    diff = abs(x - y) / max(x, y)
    return max(0.0, 100.0 * (1.0 - diff))

def get_version_num(v):
    s = clean_value(v)
    m = re.search(r"-(\d{2})$", s)
    if m:
        return int(m.group(1))
    if re.fullmatch(r"\d+", s):
        return int(s)
    if re.fullmatch(r"\d+\.0+", s):
        return int(float(s))
    return None

def version_score(tb_ver, kh_ver):
    tv = get_version_num(tb_ver)
    kv = get_version_num(kh_ver)
    if tv is None or kv is None:
        return 50.0
    diff = abs(tv - kv)
    if diff == 0:
        return 100.0
    if diff == 1:
        return 85.0
    if diff == 2:
        return 60.0
    return 30.0

def find_col_soft(df, names, required=False):
    lower = {str(c).strip().lower(): c for c in df.columns}
    for n in names:
        key = n.strip().lower()
        if key in lower:
            return lower[key]
    if required:
        raise KeyError(f"Không tìm thấy cột nào trong danh sách: {names}")
    return None

def read_all_sheets(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_excel(path, sheet_name=None, dtype=str)

def read_excel_first_sheet(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_excel(path, sheet_name=0, dtype=str)
    except Exception:
        return pd.DataFrame()

def safe_replace(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        raise FileNotFoundError(f"Không thấy file tạm: {src}")
    src.replace(dst)

def write_excel_text_safe(path, sheets_dict):
    path = Path(path)
    tmp = path.with_suffix(".tmp.xlsx")
    with pd.ExcelWriter(tmp, engine="openpyxl") as writer:
        for sheet_name, df in sheets_dict.items():
            if df is None:
                df = pd.DataFrame()
            df = fix_code_text_columns(df.copy())
            safe_sheet = sheet_name[:31]
            df.to_excel(writer, index=False, sheet_name=safe_sheet)

            ws = writer.sheets[safe_sheet]
            ws.freeze_panes = "A2"

            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.alignment = Alignment(horizontal="center")
                cell.fill = PatternFill(start_color="E8F0FE", end_color="E8F0FE", fill_type="solid")

            for idx, col_name in enumerate(df.columns, start=1):
                col_letter = get_column_letter(idx)

                if col_name in TEXT_SAFE_COLUMNS:
                    for cell in ws[col_letter]:
                        cell.number_format = "@"

                try:
                    values = df[col_name].head(200).fillna("").astype(str).tolist()
                    width = max([len(str(col_name))] + [len(x) for x in values]) + 2
                    ws.column_dimensions[col_letter].width = min(max(width, 12), 55)
                except Exception:
                    pass

    safe_replace(tmp, path)


# ============================================================
# LOAD DATA
# ============================================================

if not MASTER_OUTPUT_PATH.exists():
    raise FileNotFoundError(f"Không thấy file tổng: {MASTER_OUTPUT_PATH}. Cần chạy parse Cell 2 xong trước.")

master_sheets = read_all_sheets(MASTER_OUTPUT_PATH)

if TBMT_SHEET not in master_sheets:
    raise KeyError(f"File tổng không có sheet {TBMT_SHEET}")

df_tbmt = fix_code_text_columns(master_sheets[TBMT_SHEET])

# Ưu tiên sheet KHLCNT item-level trong file tổng.
if KHLCNT_ITEMS_SHEET in master_sheets:
    df_kh = fix_code_text_columns(master_sheets[KHLCNT_ITEMS_SHEET])
    kh_source_name = KHLCNT_ITEMS_SHEET
elif KHLCNT_ITEMS_MATCH_PATH.exists():
    df_kh = fix_code_text_columns(read_excel_first_sheet(KHLCNT_ITEMS_MATCH_PATH))
    kh_source_name = str(KHLCNT_ITEMS_MATCH_PATH)
elif KHLCNT_SHEET in master_sheets:
    df_kh = fix_code_text_columns(master_sheets[KHLCNT_SHEET])
    kh_source_name = KHLCNT_SHEET
else:
    raise KeyError("Không tìm thấy KHLCNT_Items_Match hoặc KHLCNT_Zoho_Import để match.")

print("Using KHLCNT for matching:", kh_source_name)
print("TBMT rows:", len(df_tbmt))
print("KHLCNT candidate rows:", len(df_kh))


# ============================================================
# DETECT COLUMNS
# ============================================================

C_TBMT_SO = find_col_soft(df_tbmt, ["Số TBMT", "So TBMT", "Name"], required=True)
C_TBMT_KHLCNT = find_col_soft(df_tbmt, ["Số KHLCNT", "So KHLCNT"], required=False)
if C_TBMT_KHLCNT is None:
    C_TBMT_KHLCNT = "Số KHLCNT"
    df_tbmt[C_TBMT_KHLCNT] = ""

C_TBMT_SCAN = find_col_soft(df_tbmt, ["Số KHLCNT scan từ PDF", "So KHLCNT scan tu PDF"], required=False)
C_TBMT_GOI = find_col_soft(df_tbmt, ["Tên gói thầu", "Ten goi thau"], required=False)
C_TBMT_DU_AN = find_col_soft(df_tbmt, ["Dự án", "Du an"], required=False)
C_TBMT_CDT = find_col_soft(df_tbmt, ["Chủ đầu tư", "Chu dau tu"], required=False)
C_TBMT_DVMT = find_col_soft(df_tbmt, ["Đơn vị mời thầu", "Don vi moi thau"], required=False)
C_TBMT_GIA = find_col_soft(df_tbmt, ["Giá gói thầu", "Gia goi thau", "Tổng giá trị", "Tong gia tri"], required=False)
C_TBMT_VERSION = find_col_soft(df_tbmt, ["Mã Phiên Bản", "Ma Phien Ban", "Phiên bản TBMT", "Phien ban TBMT"], required=False)

C_KH_SO = find_col_soft(df_kh, ["Số KHLCNT", "So KHLCNT", "Name"], required=True)
C_KH_GOI = find_col_soft(df_kh, ["Tên gói thầu trong KHMT", "Ten goi thau trong KHMT", "Tên gói thầu", "Ten goi thau"], required=False)
C_KH_DU_AN = find_col_soft(df_kh, ["Tên dự án", "Ten du an", "Dự án", "Du an"], required=False)
C_KH_CDT = find_col_soft(df_kh, ["Chủ đầu tư-KHLCNT", "Chu dau tu-KHLCNT", "Chủ đầu tư", "Chu dau tu"], required=False)
C_KH_GIA = find_col_soft(df_kh, ["Giá gói thầu trong KHMT", "Gia goi thau trong KHMT", "Giá gói thầu", "Gia goi thau", "Tổng mức đầu tư", "Tong muc dau tu"], required=False)
C_KH_VERSION = find_col_soft(df_kh, ["Phiên bản KHLCNT", "Phien ban KHLCNT"], required=False)
C_KH_LATEST = find_col_soft(df_kh, ["KHLCNT mới nhất", "KHLCNT moi nhat"], required=False)
C_KH_ITEM_KEY = find_col_soft(df_kh, ["KHLCNT Item Key", "Item Key"], required=False)

print("\nDetected columns:")
print("TBMT Số:", C_TBMT_SO)
print("TBMT KHLCNT:", C_TBMT_KHLCNT)
print("TBMT scan:", C_TBMT_SCAN)
print("TBMT Tên gói:", C_TBMT_GOI)
print("TBMT Dự án:", C_TBMT_DU_AN)
print("TBMT Chủ đầu tư:", C_TBMT_CDT)
print("TBMT ĐVMT:", C_TBMT_DVMT)
print("TBMT Giá:", C_TBMT_GIA)
print("KHLCNT Số:", C_KH_SO)
print("KHLCNT Tên gói:", C_KH_GOI)
print("KHLCNT Dự án:", C_KH_DU_AN)
print("KHLCNT Chủ đầu tư:", C_KH_CDT)
print("KHLCNT Giá:", C_KH_GIA)
print("KHLCNT item key:", C_KH_ITEM_KEY)


# ============================================================
# CLEANUP + OUTPUT COLUMNS
# ============================================================

df_tbmt[C_TBMT_SO] = df_tbmt[C_TBMT_SO].apply(normalize_tbmt_code)
df_kh[C_KH_SO] = df_kh[C_KH_SO].apply(normalize_khlcnt_code)

for col in [
    "Trạng thái match KHLCNT", "Điểm match KHLCNT", "Nguồn match KHLCNT",
    "Ghi chú match KHLCNT", "Số KHLCNT ứng viên", "Tên gói KHLCNT match",
    "Giá KHLCNT match", "Phiên bản KHLCNT match", "KHLCNT Item Key match",
]:
    if col not in df_tbmt.columns:
        df_tbmt[col] = ""

# Ép các cột match về object/string để tránh FutureWarning dtype.
for col in [
    C_TBMT_KHLCNT, "Trạng thái match KHLCNT", "Điểm match KHLCNT",
    "Nguồn match KHLCNT", "Ghi chú match KHLCNT", "Số KHLCNT ứng viên",
    "Tên gói KHLCNT match", "Giá KHLCNT match", "Phiên bản KHLCNT match",
    "KHLCNT Item Key match",
]:
    if col in df_tbmt.columns:
        df_tbmt[col] = df_tbmt[col].astype("object")

kh_codes_set = set(df_kh[C_KH_SO].dropna().astype(str).str.strip())
print("\nKHLCNT code count:", len(kh_codes_set))


# ============================================================
# SCORE FUNCTIONS
# ============================================================

def row_get(row, col):
    if not col:
        return ""
    try:
        return clean_value(row.get(col, ""))
    except Exception:
        return ""

def build_match_text(row, goi_col, du_an_col, cdt_col, alt_cdt_col=None):
    parts = []
    for col in [goi_col, du_an_col, cdt_col, alt_cdt_col]:
        if col:
            val = row_get(row, col)
            if val:
                parts.append(val)
    return normalize_key(" ".join(parts))

def calc_score(tb_row, kh_row):
    tb_goi = normalize_key(row_get(tb_row, C_TBMT_GOI))
    kh_goi = normalize_key(row_get(kh_row, C_KH_GOI))

    tb_du_an = normalize_key(row_get(tb_row, C_TBMT_DU_AN))
    kh_du_an = normalize_key(row_get(kh_row, C_KH_DU_AN))

    tb_cdt = normalize_key(row_get(tb_row, C_TBMT_CDT))
    tb_dvmt = normalize_key(row_get(tb_row, C_TBMT_DVMT))
    kh_cdt = normalize_key(row_get(kh_row, C_KH_CDT))

    goi_score = fuzz.token_set_ratio(tb_goi, kh_goi) if tb_goi and kh_goi else 0.0
    du_an_score = fuzz.token_set_ratio(tb_du_an, kh_du_an) if tb_du_an and kh_du_an else 0.0

    cdt_score_1 = fuzz.token_set_ratio(tb_cdt, kh_cdt) if tb_cdt and kh_cdt else 0.0
    cdt_score_2 = fuzz.token_set_ratio(tb_dvmt, kh_cdt) if tb_dvmt and kh_cdt else 0.0
    cdt_score = max(cdt_score_1, cdt_score_2)

    ps = price_score(row_get(tb_row, C_TBMT_GIA), row_get(kh_row, C_KH_GIA))
    vs = version_score(row_get(tb_row, C_TBMT_VERSION), row_get(kh_row, C_KH_VERSION))

    if ps >= 0:
        total = 0.45 * goi_score + 0.25 * du_an_score + 0.20 * cdt_score + 0.10 * ps
    else:
        total = 0.50 * goi_score + 0.30 * du_an_score + 0.20 * cdt_score

    # version bonus nhỏ
    total = 0.97 * total + 0.03 * vs

    # KHLCNT latest bonus rất nhẹ
    latest_txt = row_get(kh_row, C_KH_LATEST).lower()
    if latest_txt in ["true", "1", "yes", "x", "✓", "có", "co"]:
        total += 1.0

    total = min(100.0, total)

    return {
        "total": float(total),
        "goi_score": float(goi_score),
        "du_an_score": float(du_an_score),
        "cdt_score": float(cdt_score),
        "price_score": float(ps),
        "version_score": float(vs),
    }

def is_safe_auto(best_score, second_score):
    if best_score["total"] < AUTO_MATCH_SCORE:
        return False

    margin = best_score["total"] - second_score if second_score is not None else 100.0
    if margin < MIN_MARGIN_FOR_AUTO:
        return False

    # Tên gói quá thấp thì không auto.
    if best_score["goi_score"] < 65:
        return False

    # Phải có ít nhất 1 trong 2: dự án hoặc chủ đầu tư ổn.
    if max(best_score["du_an_score"], best_score["cdt_score"]) < 60:
        return False

    # Nếu có giá ở cả 2 bên thì giá không được lệch quá mạnh.
    if best_score["price_score"] >= 0 and best_score["price_score"] < 80:
        return False

    return True


# ============================================================
# GPU PREFILTER BUILD
# ============================================================

kh_top_indices_by_tb_pos = None
rows_to_match = []

# Xử lý direct scan / existing trước, rồi gom rows cần fuzzy match.
direct_confirmed = 0
direct_pending = 0
kept_existing = 0

for idx, tb_row in df_tbmt.iterrows():
    existing_kh = normalize_khlcnt_code(row_get(tb_row, C_TBMT_KHLCNT))
    scan_kh = normalize_khlcnt_code(row_get(tb_row, C_TBMT_SCAN)) if C_TBMT_SCAN else ""

    if existing_kh and not ALLOW_OVERWRITE_EXISTING:
        kept_existing += 1
        continue

    if scan_kh:
        if scan_kh in kh_codes_set:
            df_tbmt.at[idx, C_TBMT_KHLCNT] = scan_kh
            df_tbmt.at[idx, "Trạng thái match KHLCNT"] = "Direct scan - confirmed in KHLCNT master"
            df_tbmt.at[idx, "Điểm match KHLCNT"] = 100
            df_tbmt.at[idx, "Nguồn match KHLCNT"] = "Scan trực tiếp từ TBMT PDF"
            df_tbmt.at[idx, "Số KHLCNT ứng viên"] = scan_kh
            df_tbmt.at[idx, "Ghi chú match KHLCNT"] = "Scan được mã KHLCNT từ PDF và mã đã có trong KHLCNT master"
            direct_confirmed += 1
        else:
            df_tbmt.at[idx, "Trạng thái match KHLCNT"] = "Direct scan pending - KHLCNT chưa có trong master"
            df_tbmt.at[idx, "Điểm match KHLCNT"] = ""
            df_tbmt.at[idx, "Nguồn match KHLCNT"] = "Scan trực tiếp từ TBMT PDF"
            df_tbmt.at[idx, "Số KHLCNT ứng viên"] = scan_kh
            df_tbmt.at[idx, "Ghi chú match KHLCNT"] = "Có mã KHLCNT scan từ PDF nhưng chưa có trong KHLCNT master; chờ dữ liệu KHLCNT mới"
            direct_pending += 1
        continue

    rows_to_match.append(idx)

print("\nBefore fuzzy match:")
print("Existing KHLCNT kept:", kept_existing)
print("Direct scan confirmed:", direct_confirmed)
print("Direct scan pending:", direct_pending)
print("Rows need fuzzy match:", len(rows_to_match))


# GPU prefilter: encode toàn bộ KHLCNT và TBMT cần match.
if USE_GPU_EMBED_PREFILTER and GPU_DEVICE == "cuda" and rows_to_match:
    try:
        embed_model = SentenceTransformer(
            "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            device=GPU_DEVICE
        )

        kh_texts = [
            build_match_text(kh_row, C_KH_GOI, C_KH_DU_AN, C_KH_CDT)
            for _, kh_row in df_kh.iterrows()
        ]

        tb_texts = [
            build_match_text(df_tbmt.loc[idx], C_TBMT_GOI, C_TBMT_DU_AN, C_TBMT_CDT, C_TBMT_DVMT)
            for idx in rows_to_match
        ]

        print("\nEncoding KHLCNT candidates on GPU...")
        kh_emb = embed_model.encode(
            kh_texts,
            batch_size=EMBED_BATCH_SIZE,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=True
        ).to(GPU_DEVICE)

        print("Encoding TBMT rows on GPU...")
        tb_emb = embed_model.encode(
            tb_texts,
            batch_size=EMBED_BATCH_SIZE,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=True
        ).to(GPU_DEVICE)

        print("Computing top-K candidates on GPU...")
        k = min(EMBED_TOP_K, len(df_kh))
        sims = tb_emb @ kh_emb.T
        top_indices = torch.topk(sims, k=k, dim=1).indices.detach().cpu().numpy()

        kh_top_indices_by_tb_pos = top_indices

        print("GPU prefilter ready.")
        print("Top-K:", k)

    except Exception as e:
        kh_top_indices_by_tb_pos = None
        USE_GPU_EMBED_PREFILTER = False
        print("GPU prefilter lỗi, fallback match full CPU.")
        print("Reason:", repr(e))


# ============================================================
# FUZZY MATCH LOOP
# ============================================================

match_candidate_rows = []
auto_count = 0
review_count = 0
no_match_count = 0

for pos, tb_idx in enumerate(tqdm(rows_to_match, desc="Matching TBMT ↔ KHLCNT")):
    tb_row = df_tbmt.loc[tb_idx]

    if kh_top_indices_by_tb_pos is not None:
        candidate_indices = kh_top_indices_by_tb_pos[pos].tolist()
        kh_iter = df_kh.iloc[candidate_indices]
    else:
        kh_iter = df_kh

    scored = []

    for kh_idx, kh_row in kh_iter.iterrows():
        so_kh = normalize_khlcnt_code(row_get(kh_row, C_KH_SO))
        if not so_kh:
            continue

        sc = calc_score(tb_row, kh_row)

        scored.append({
            "kh_idx": kh_idx,
            "Số KHLCNT": so_kh,
            "total": sc["total"],
            "goi_score": sc["goi_score"],
            "du_an_score": sc["du_an_score"],
            "cdt_score": sc["cdt_score"],
            "price_score": sc["price_score"],
            "version_score": sc["version_score"],
            "Tên gói KHLCNT": row_get(kh_row, C_KH_GOI),
            "Giá KHLCNT": row_get(kh_row, C_KH_GIA),
            "Phiên bản KHLCNT": row_get(kh_row, C_KH_VERSION),
            "KHLCNT Item Key": row_get(kh_row, C_KH_ITEM_KEY),
        })

    if not scored:
        no_match_count += 1
        df_tbmt.at[tb_idx, "Trạng thái match KHLCNT"] = "Không có candidate"
        continue

    scored = sorted(scored, key=lambda x: x["total"], reverse=True)
    top3 = scored[:3]
    best = top3[0]
    second_total = top3[1]["total"] if len(top3) > 1 else None

    so_tbmt = normalize_tbmt_code(row_get(tb_row, C_TBMT_SO))

    # Save top candidates
    for rank, cand in enumerate(top3, start=1):
        match_candidate_rows.append({
            "Số TBMT": so_tbmt,
            "Rank": rank,
            "Số KHLCNT ứng viên": cand["Số KHLCNT"],
            "Tổng điểm": cand["total"],
            "Điểm tên gói": cand["goi_score"],
            "Điểm dự án": cand["du_an_score"],
            "Điểm chủ đầu tư": cand["cdt_score"],
            "Điểm giá": cand["price_score"],
            "Điểm version": cand["version_score"],
            "Tên gói KHLCNT": cand["Tên gói KHLCNT"],
            "Giá KHLCNT": cand["Giá KHLCNT"],
            "Phiên bản KHLCNT": cand["Phiên bản KHLCNT"],
            "KHLCNT Item Key": cand["KHLCNT Item Key"],
            "Generated at": now_str(),
        })

    # Decide
    best_score_obj = {
        "total": best["total"],
        "goi_score": best["goi_score"],
        "du_an_score": best["du_an_score"],
        "cdt_score": best["cdt_score"],
        "price_score": best["price_score"],
    }

    if is_safe_auto(best_score_obj, second_total):
        df_tbmt.at[tb_idx, C_TBMT_KHLCNT] = best["Số KHLCNT"]
        df_tbmt.at[tb_idx, "Trạng thái match KHLCNT"] = "Auto matched - safe"
        df_tbmt.at[tb_idx, "Điểm match KHLCNT"] = round(best["total"], 2)
        df_tbmt.at[tb_idx, "Nguồn match KHLCNT"] = "GPU embedding prefilter + RapidFuzz ratio" if kh_top_indices_by_tb_pos is not None else "RapidFuzz ratio"
        df_tbmt.at[tb_idx, "Số KHLCNT ứng viên"] = best["Số KHLCNT"]
        df_tbmt.at[tb_idx, "Tên gói KHLCNT match"] = best["Tên gói KHLCNT"]
        df_tbmt.at[tb_idx, "Giá KHLCNT match"] = best["Giá KHLCNT"]
        df_tbmt.at[tb_idx, "Phiên bản KHLCNT match"] = fix_version_value(best["Phiên bản KHLCNT"])
        df_tbmt.at[tb_idx, "KHLCNT Item Key match"] = best["KHLCNT Item Key"]
        df_tbmt.at[tb_idx, "Ghi chú match KHLCNT"] = f"Auto match. Top1={best['total']:.2f}; Top2={second_total if second_total is not None else ''}"
        auto_count += 1

    elif best["total"] >= REVIEW_MATCH_SCORE:
        df_tbmt.at[tb_idx, "Trạng thái match KHLCNT"] = "Cần kiểm tra"
        df_tbmt.at[tb_idx, "Điểm match KHLCNT"] = round(best["total"], 2)
        df_tbmt.at[tb_idx, "Nguồn match KHLCNT"] = "Review candidate from ratio"
        df_tbmt.at[tb_idx, "Số KHLCNT ứng viên"] = best["Số KHLCNT"]
        df_tbmt.at[tb_idx, "Tên gói KHLCNT match"] = best["Tên gói KHLCNT"]
        df_tbmt.at[tb_idx, "Giá KHLCNT match"] = best["Giá KHLCNT"]
        df_tbmt.at[tb_idx, "Phiên bản KHLCNT match"] = fix_version_value(best["Phiên bản KHLCNT"])
        df_tbmt.at[tb_idx, "KHLCNT Item Key match"] = best["KHLCNT Item Key"]
        df_tbmt.at[tb_idx, "Ghi chú match KHLCNT"] = f"Ứng viên cần kiểm tra. Top1={best['total']:.2f}; Top2={second_total if second_total is not None else ''}"
        review_count += 1

    else:
        df_tbmt.at[tb_idx, "Trạng thái match KHLCNT"] = "Không match"
        df_tbmt.at[tb_idx, "Điểm match KHLCNT"] = round(best["total"], 2)
        df_tbmt.at[tb_idx, "Nguồn match KHLCNT"] = "Ratio below threshold"
        df_tbmt.at[tb_idx, "Số KHLCNT ứng viên"] = best["Số KHLCNT"]
        df_tbmt.at[tb_idx, "Ghi chú match KHLCNT"] = f"Điểm thấp hơn REVIEW_MATCH_SCORE={REVIEW_MATCH_SCORE}. Top1={best['total']:.2f}"
        no_match_count += 1


# ============================================================
# SAVE OUTPUT
# ============================================================

df_tbmt = fix_code_text_columns(df_tbmt)

# Update master workbook, preserve other sheets
master_sheets[TBMT_SHEET] = df_tbmt
write_excel_text_safe(MASTER_OUTPUT_PATH, master_sheets)

# Update TBMT import
write_excel_text_safe(TBMT_IMPORT_PATH, {TBMT_SHEET: df_tbmt})

# Save candidates
df_candidates = pd.DataFrame(match_candidate_rows)
write_excel_text_safe(MATCH_CANDIDATES_PATH, {"Match_Candidates": df_candidates})

# Append history if exists
if MATCH_CANDIDATES_HISTORY_PATH.exists():
    try:
        old_hist = pd.read_excel(MATCH_CANDIDATES_HISTORY_PATH, dtype=str)
        hist = pd.concat([old_hist, df_candidates], ignore_index=True, sort=False)
    except Exception:
        hist = df_candidates.copy()
else:
    hist = df_candidates.copy()

write_excel_text_safe(MATCH_CANDIDATES_HISTORY_PATH, {"Match_Candidates_History": hist})


# ============================================================
# SUMMARY
# ============================================================

mapped_now = df_tbmt[C_TBMT_KHLCNT].fillna("").astype(str).str.strip().ne("").sum()

print("\nDONE MATCH ONLY CELL")
print("Master updated:", MASTER_OUTPUT_PATH)
print("TBMT import updated:", TBMT_IMPORT_PATH)
print("Match candidates:", MATCH_CANDIDATES_PATH)
print("Match candidates history:", MATCH_CANDIDATES_HISTORY_PATH)

print("\nSummary:")
print("TBMT rows:", len(df_tbmt))
print("KHLCNT candidate rows:", len(df_kh))
print("Existing kept:", kept_existing)
print("Direct scan confirmed:", direct_confirmed)
print("Direct scan pending:", direct_pending)
print("Rows fuzzy matched:", len(rows_to_match))
print("Auto matched:", auto_count)
print("Need review:", review_count)
print("No match:", no_match_count)
print("Total TBMT có Số KHLCNT sau match:", mapped_now)

GPU_DEVICE: cuda
GPU name: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MASTER_OUTPUT_PATH: /content/drive/MyDrive/Zoho_Procurement/output_excel/Zoho_PDF_Import_Output_LATEST.xlsx
TBMT_IMPORT_PATH: /content/drive/MyDrive/Zoho_Procurement/output_excel/TBMT_for_import.xlsx
KHLCNT_ITEMS_MATCH_PATH: /content/drive/MyDrive/Zoho_Procurement/output_excel/KHLCNT_items_for_match.xlsx
MATCH_CANDIDATES_PATH: /content/drive/MyDrive/Zoho_Procurement/logs/Match_Candidates.xlsx
Using KHLCNT for matching: KHLCNT_Items_Match
TBMT rows: 6123
KHLCNT candidate rows: 10873

Detected columns:
TBMT Số: Số TBMT
TBMT KHLCNT: Số KHLCNT
TBMT scan: Số KHLCNT scan từ PDF
TBMT Tên gói: Tên gói thầu
TBMT Dự án: Dự án
TBMT Chủ đầu tư: Chủ đầu tư
TBMT ĐVMT: Đơn vị mời thầu
TBMT Giá: Giá gói thầu
KHLCNT Số: Số KHLCNT
KHLCNT Tên gói: Tên gói thầu trong KHMT
KHLCNT Dự án: Tên dự án
KHLCNT Chủ đầu tư: Chủ đầu tư-KHLCNT
KHLCNT Giá:

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

Encoding TBMT rows on GPU...


Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Computing top-K candidates on GPU...
GPU prefilter ready.
Top-K: 150


Matching TBMT ↔ KHLCNT: 100%|██████████| 6123/6123 [06:58<00:00, 14.63it/s]
/tmp/ipykernel_1848/1251581538.py:316: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  values = df[col_name].head(200).fillna("").astype(str).tolist()
/tmp/ipykernel_1848/1251581538.py:316: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  values = df[col_name].head(200).fillna("").astype(str).tolist()



DONE MATCH ONLY CELL
Master updated: /content/drive/MyDrive/Zoho_Procurement/output_excel/Zoho_PDF_Import_Output_LATEST.xlsx
TBMT import updated: /content/drive/MyDrive/Zoho_Procurement/output_excel/TBMT_for_import.xlsx
Match candidates: /content/drive/MyDrive/Zoho_Procurement/logs/Match_Candidates.xlsx
Match candidates history: /content/drive/MyDrive/Zoho_Procurement/logs/Match_Candidates_History.xlsx

Summary:
TBMT rows: 6123
KHLCNT candidate rows: 10873
Existing kept: 0
Direct scan confirmed: 0
Direct scan pending: 0
Rows fuzzy matched: 6123
Auto matched: 1948
Need review: 1473
No match: 2702
Total TBMT có Số KHLCNT sau match: 1948


In [ ]:
# ============================================================
# COUNT TBMT ĐÃ ĐƯỢC MATCH KHLCNT
# ============================================================

import pandas as pd
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/Zoho_Procurement")
OUTPUT_DIR = BASE_DIR / "output_excel"

MASTER_OUTPUT_PATH = OUTPUT_DIR / "Zoho_PDF_Import_Output_LATEST.xlsx"
TBMT_SHEET = "TBMT_Zoho_Import"

df_tbmt = pd.read_excel(MASTER_OUTPUT_PATH, sheet_name=TBMT_SHEET, dtype=str)

def clean_col(s):
    return s.fillna("").astype(str).str.strip()

# Detect cột Số KHLCNT
possible_kh_cols = ["Số KHLCNT", "So KHLCNT", "Số KHLCNT.Name", "So KHLCNT.Name"]
C_KHLCNT = None

for col in possible_kh_cols:
    if col in df_tbmt.columns:
        C_KHLCNT = col
        break

if C_KHLCNT is None:
    raise KeyError("Không tìm thấy cột Số KHLCNT trong TBMT_Zoho_Import")

# Count tổng
total_tbmt = len(df_tbmt)
matched_tbmt = clean_col(df_tbmt[C_KHLCNT]).ne("").sum()
unmatched_tbmt = total_tbmt - matched_tbmt

print("===== TBMT MATCH SUMMARY =====")
print("Tổng số TBMT:", total_tbmt)
print("TBMT đã có Số KHLCNT / đã match:", matched_tbmt)
print("TBMT chưa có Số KHLCNT / chưa match:", unmatched_tbmt)

if total_tbmt > 0:
    print("Tỷ lệ đã match:", round(matched_tbmt / total_tbmt * 100, 2), "%")
    print("Tỷ lệ chưa match:", round(unmatched_tbmt / total_tbmt * 100, 2), "%")

# Count theo trạng thái match nếu có
status_col = "Trạng thái match KHLCNT"

if status_col in df_tbmt.columns:
    print("\n===== COUNT THEO TRẠNG THÁI MATCH =====")
    status_counts = (
        df_tbmt[status_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .replace("", "Trống")
        .value_counts()
    )

    print(status_counts)

# Count số TBMT có ứng viên nhưng chưa chắc đã match
candidate_col = "Số KHLCNT ứng viên"

if candidate_col in df_tbmt.columns:
    candidate_count = clean_col(df_tbmt[candidate_col]).ne("").sum()
    print("\nTBMT có Số KHLCNT ứng viên:", candidate_count)

# Count direct scan nếu có
scan_col = "Số KHLCNT scan từ PDF"

if scan_col in df_tbmt.columns:
    scan_count = clean_col(df_tbmt[scan_col]).ne("").sum()
    print("TBMT scan được Số KHLCNT từ PDF:", scan_count)

===== TBMT MATCH SUMMARY =====
Tổng số TBMT: 6123
TBMT đã có Số KHLCNT / đã match: 2514
TBMT chưa có Số KHLCNT / chưa match: 3609
Tỷ lệ đã match: 41.06 %
Tỷ lệ chưa match: 58.94 %

===== COUNT THEO TRẠNG THÁI MATCH =====
Trạng thái match KHLCNT
Không match            2698
Auto matched - safe    1948
Cần kiểm tra            911
Manual confirmed        566
Name: count, dtype: int64

TBMT có Số KHLCNT ứng viên: 6123
TBMT scan được Số KHLCNT từ PDF: 0
